# When Is the Sub-Barcode Bound Attained? — experiments and exact checks

This notebook produces every computational result, table and figure of the paper. All code is in the cells below. The only downloads are the SDRBench fields of Section 20 and, if it is missing, the `hdf5plugin` package from PyPI.

**How to run.** Run all cells from the top, or call `make quick` or `make full` from the root of the repository. Each finished configuration is checkpointed, so an interrupted run resumes where it stopped. Start from an empty results folder when the configuration changes.
- `SOCG_PROFILE=FULL` (the default) gives the sizes reported in the paper. It streams about 1.5 GB from SDRBench, and the reported run took 72 minutes on 4 CPU cores.
- `SOCG_PROFILE=QUICK` runs every section on reduced sizes and skips SDRBench. Use it to check the pipeline.
- `SOCG_RESULTS` sets the output folder (default `results`). The repository's `results/` folder holds the reported FULL run, so write new runs elsewhere, for example `SOCG_RESULTS=runs/full`.
- `ANON_FORBIDDEN` can hold a comma-separated list of extra strings, such as names, for the anonymity scan of Section 24 to redact. Set it in the shell, never in a cell, because a cell would store the strings in the notebook.

**Outputs.** `paper_numbers.md` lists every number the paper quotes, recomputed from the run. `figures/` holds Figures 1–4, `tables/` holds Tables 2 and 3, and `logs/` holds the run log, the validation, the reproducibility report and the anonymity scan. A zip of the whole results folder is written next to it.

**Anonymity.** The notebook records nothing about the machine beyond software versions, processor type, core count and memory size, and paths in manifests are relative to the results folder. Before zipping, Section 24 scans every text output for e-mail addresses and home-directory paths and redacts them.

**Where each result of the paper is checked.** The LaTeX labels are given next to the numbers.

| Result in the paper | Label | Section |
|---|---|---|
| Theorem 1 (sub-barcode bound), Lemma 2 (optimal certificates) | `thm:cgs`, `lem:optimal` | 6 |
| Theorem 5 (degree 0: $f_U$ attains) | `thm:deg0` | 4, 5, 6 |
| Theorem 8 (codimension one: $f_D$ attains on full subcomplexes of $S^d$) | `thm:sphere` | 9 ($d=2$), 18 ($d=3$) |
| Theorem 10 (surfaces: finite bars and counts) | `thm:surface-finite` | 13 |
| Theorem 11 (non-planar surfaces), Figure 2 | `thm:nonplanar`, `fig:certD` | 10 (certificates D, E and five closed surfaces) |
| Lemma 12, Theorem 13 (deciding attainment on surfaces) | `lem:maximal`, `thm:decide` | 16 |
| Corollary 14 (near-linear time on surfaces) | `cor:fast` | 15 |
| Proposition 15 (graphs) | `prop:graphs` | 14 |
| Lemma 16 (KC vertices) | `lem:kc` | 8 |
| Examples 17–20 | `ex:A`, `ex:B`, `ex:C`, `ex:counts` | 10 (certificates A, B, C, F) |
| Theorem 21 (middle degrees fail on spheres and balls) | `thm:middle` | 17; 18 for degree 1 on 3-D grids |
| Table 2 and the degree-1 paragraph of Section 7 | `tab:census` | 7, 11, 12 |
| Degree-0 paragraph of Section 7 | — | 5, 6 |
| Figure 3 and the scaling paragraph | `fig:experiments` | 15, 19 |
| Compression paragraph, Table 3, Appendix D | `tab:compression` | 20 |
| Figures 1–4, Tables 2 and 3 | — | 22 |
| Every number quoted in the paper | — | 23 |

A few identifiers and unit-test names, such as `lemma1_check`, `tau_intervals` and tests T02, T09, T14 and T17, come from the development of the degree-0 sweep and do not refer to numbered results of the paper. `docs/unit-tests.md` in the repository describes every test.

## 1 · Environment

In [1]:
import os, sys, platform, json, time, math, hashlib, subprocess, importlib, warnings, itertools, gc, io, tarfile, shutil, resource, zipfile, bisect, collections, re, types
from dataclasses import dataclass, field, asdict
from fractions import Fraction
from pathlib import Path

# Thread pinning for fair, reproducible timings (set before numpy import).
for _v in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS", "NUMBA_NUM_THREADS"]:
    os.environ.setdefault(_v, "1")

def _ensure(module, pip_name=None):
    """Import `module`; install it with pip only if it is missing (retrying for externally managed environments)."""
    try:
        return importlib.import_module(module)
    except ImportError:
        for extra in ([], ["--break-system-packages"]):
            r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *extra, pip_name or module],
                               capture_output=True, text=True)
            if r.returncode == 0:
                break
        importlib.invalidate_caches()
        return importlib.import_module(module)

np = _ensure("numpy")
scipy = _ensure("scipy")
pd = _ensure("pandas")
_ensure("numba")
psutil = _ensure("psutil")
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker
try:                                    # show figures inline when running inside Jupyter; files are always saved
    from IPython.display import display as _display
    _IN_NB = get_ipython() is not None  # noqa: F821
except Exception:
    _IN_NB = False
from numba import njit
import numba
from scipy import stats as sstats
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_bipartite_matching
from scipy.spatial import Delaunay
from scipy import ndimage

def detect_platform():
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle"):
        return "kaggle"
    if "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ:
        return "colab"
    return "local"

def gpu_info():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=10)
        return out.stdout.strip() or "none"
    except Exception:
        return "none"

ENV = {
    "platform": detect_platform(),
    "python": sys.version.split()[0],
    "os": f"{platform.system()} {platform.release()}",
    "cpu": platform.processor() or platform.machine(),
    "logical_cpus": os.cpu_count(),
    "ram_gb": round(psutil.virtual_memory().total / 1e9, 2),
    "gpu": gpu_info(),
    "numpy": np.__version__, "scipy": scipy.__version__, "pandas": pd.__version__,
    "numba": numba.__version__, "matplotlib": matplotlib.__version__,
    "threads": {k: os.environ.get(k) for k in ["OMP_NUM_THREADS", "NUMBA_NUM_THREADS"]},
}
for k, v in ENV.items():
    print(f"{k:>14}: {v}")

      platform: kaggle
        python: 3.13.16
            os: Linux 6.18.55+
           cpu: x86_64
  logical_cpus: 4
        ram_gb: 33.66
           gpu: Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
         numpy: 2.1.3
         scipy: 1.16.3
        pandas: 2.3.3
         numba: 0.61.2
    matplotlib: 3.10.0
       threads: {'OMP_NUM_THREADS': '1', 'NUMBA_NUM_THREADS': '1'}


## 2 · Configuration

Two profiles: `FULL` gives every size reported in the paper; `QUICK` runs each section on a reduced size to check the pipeline. All randomness flows from one master seed: each experiment draws from its own generator, seeded by a hash of the master seed and the experiment's name, so rerunning a section gives the same results. Output paths are relative.

In [2]:
PROFILE = os.environ.get("SOCG_PROFILE", "FULL").upper()
assert PROFILE in ("FULL", "QUICK"), PROFILE

_FULL = dict(
    # degree 0: Theorem 5, Lemma 2 and the tau = 0 identities (Section 6)
    gkls_instances=500, opt_instances=100_000, opt_brute_instances=1000,
    # degree-1 exact machinery, Lemma 16, Theorem 8, census (Sections 7-12)
    d1x_selfcheck=400, d1x_indep=60, d1x_lemma_complexes=1500, d1x_planar=300, d1x_census=3000, d1x_small_surfaces=400,
    d1x_pinched=200, d1x_order_cap=200_000,
    # Theorem 10, Proposition 15, Corollary 14 (Sections 13-15)
    d1x_thmc=400, d1x_graphs=600, d1f_corr=3.0, d1f_halfwidth=0.05, d1f_exact_sides=[4, 5, 6, 7, 8, 9, 10], d1f_exact_reps=3,
    d1f_witness_sides=[20, 40, 60, 80, 100], d1f_sides=[100, 200, 400, 800, 1600], d1f_reps=3,
    # Theorem 13 and Lemma 12 (Section 16): (surface, boxes, seed). The reference runs are the ones the paper cites; the extended runs
    # use ten times as many boxes and new seeds.
    t13_decide=[("mobius", 400, 11), ("rp2", 300, 12), ("ptorus", 200, 21), ("torus", 200, 22)],
    t13_max=[("mobius", 150, 32), ("ptorus", 60, 31)], t13_max_closed=[("rp2", 150, 41), ("torus", 60, 42)],
    t13_extended=[("decide", "mobius", 4000, 111), ("decide", "rp2", 3000, 112), ("decide", "ptorus", 2000, 121), ("decide", "torus", 2000, 122),
                  ("max", "mobius", 1500, 132), ("max", "ptorus", 600, 131), ("max_closed", "rp2", 1500, 141), ("max_closed", "torus", 600, 142)],
    # 3-D Freudenthal grids (Section 18)
    g3_dual_shapes=[(3, 3, 3), (3, 3, 4), (4, 4, 4), (4, 4, 5), (5, 5, 5)], g3_dual_boxes=100,
    g3_census_jobs=[((2, 3, 3), 1), ((2, 3, 3), 2), ((2, 3, 3), 4), ((2, 3, 3), 6), ((3, 3, 3), 1), ((3, 3, 3), 2), ((3, 3, 3), 4),
                    ((3, 3, 3), 6), ((3, 3, 4), 2), ((3, 3, 4), 4)], g3_census_boxes=200,
    # Example 20 (Section 10)
    ex20_grid_step=0.25,
    # scaling (Section 19)
    p2_sizes=[10_000, 30_000, 100_000, 300_000, 1_000_000, 3_000_000, 10_000_000, 20_000_000, 50_000_000],
    p2_reps=10, p2_density_n=200_000, p2_density_degrees=[2, 4, 8, 16, 32, 64], p2_tau_n=1_000_000, p2v2_reps=5, p2_mem_reps=3,
    # compression (Section 20)
    run_compression=True, comp_synth_2d=4, comp_synth_3d=2, comp_shape2d=(256, 256), comp_shape3d=(48, 96, 96),
    comp_rel_bounds=[1e-2, 1e-3, 1e-4],
)
_QUICK = dict(
    gkls_instances=40, opt_instances=4000, opt_brute_instances=60,
    d1x_selfcheck=40, d1x_indep=8, d1x_lemma_complexes=150, d1x_planar=20, d1x_census=150, d1x_small_surfaces=25,
    d1x_pinched=15, d1x_order_cap=50_000,
    d1x_thmc=30, d1x_graphs=60, d1f_corr=3.0, d1f_halfwidth=0.05, d1f_exact_sides=[4, 5, 6], d1f_exact_reps=2,
    d1f_witness_sides=[20, 30], d1f_sides=[64, 128, 256], d1f_reps=1,
    t13_decide=[("mobius", 40, 11), ("rp2", 30, 12), ("ptorus", 20, 21), ("torus", 20, 22)],
    t13_max=[("mobius", 15, 32), ("ptorus", 6, 31)], t13_max_closed=[("rp2", 15, 41), ("torus", 6, 42)], t13_extended=[],
    g3_dual_shapes=[(3, 3, 3), (3, 3, 4)], g3_dual_boxes=6,
    g3_census_jobs=[((2, 3, 3), 2), ((3, 3, 3), 4)], g3_census_boxes=8,
    ex20_grid_step=0.5,
    p2_sizes=[10_000, 30_000, 100_000, 300_000, 1_000_000], p2_reps=3, p2_density_n=50_000, p2_density_degrees=[2, 4, 8, 16],
    p2_tau_n=200_000, p2v2_reps=2, p2_mem_reps=2,
    run_compression=False, comp_synth_2d=1, comp_synth_3d=1, comp_shape2d=(96, 96), comp_shape3d=(24, 32, 32), comp_rel_bounds=[1e-2, 1e-3],
)
CONFIG = dict(
    profile=PROFILE, seed=20260925, dtype="float64",
    p2_slope_range=(0.95, 1.15), p2_slope_falsify=1.2,
    compression=dict(
        sdrbench=[dict(name="CESM-ATM", shape=(1800, 3600), n_fields=4,
                       url="https://g-d0cd3f.fd635.8443.data.globus.org/raw-data/CESM-ATM/SDRBENCH-CESM-ATM-1800x3600.tar.gz"),
                  dict(name="Hurricane-ISABEL-Pf", shape=(100, 500, 500), n_fields=1,
                       url="https://g-d0cd3f.fd635.8443.data.globus.org/raw-data/Hurricane-ISABEL/SDRBENCH-Hurricane-ISABEL-Pf.tar.gz")],
        methods=["SZ3", "ZFP", "SZ", "SZ3-pwrel"], tau_fracs=[0.005, 0.02], conn2d=6, conn3d=6, max_download_gb=3.0),
    **(_FULL if PROFILE == "FULL" else _QUICK),
)
RESULTS = Path(os.environ.get("SOCG_RESULTS", "results"))
for sub in ["raw", "processed", "tables", "figures", "configs", "logs", "checkpoints", "data", "code"]:
    (RESULTS / sub).mkdir(parents=True, exist_ok=True)


class _Tee:
    """Mirror everything written to stdout/stderr (prints, warnings, tracebacks) into results/logs/run_log.txt."""
    _socg_tee = True
    def __init__(self, stream, fh):
        self._s = stream; self._f = fh
    def write(self, x):
        try:
            self._f.write(x)
        except Exception:
            pass
        return self._s.write(x)
    def flush(self):
        try:
            self._f.flush()
        except Exception:
            pass
        self._s.flush()
    def __getattr__(self, name):
        return getattr(self._s, name)


if not getattr(sys.stdout, "_socg_tee", False):            # re-running this cell must not wrap twice
    _RUN_LOG = open(RESULTS / "logs" / "run_log.txt", "a", encoding="utf-8", buffering=1)
    _RUN_LOG.write(f"\n===== session start {time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())} UTC | profile {PROFILE} =====\n")
    sys.stdout = _Tee(sys.stdout, _RUN_LOG); sys.stderr = _Tee(sys.stderr, _RUN_LOG)
(RESULTS / "configs" / "configuration.json").write_text(json.dumps(CONFIG, indent=2, default=str))
CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:12]
print("profile:", PROFILE, "| config hash:", CONFIG_HASH, "| results folder:", RESULTS if not RESULTS.is_absolute() else RESULTS.name)

profile: FULL | config hash: 8f4b785c7a4a | results folder: results


## 3 · Utilities and data sources

Seeds, JSON helpers and resumable checkpoints (one JSON line per finished configuration; an interrupted run resumes where it stopped). The only external data are the SDRBench fields of Section 20, streamed from the SDRBench HTTPS host and recorded with their SHA-256 checksums.

In [3]:
def seed_for(*keys):
    """Deterministic 32-bit seed from the master seed and any keys (stable across sessions, unlike hash())."""
    h = hashlib.sha256((str(CONFIG["seed"]) + "|" + "|".join(map(str, keys))).encode()).hexdigest()
    return int(h[:8], 16)

def rng_for(*keys):
    return np.random.default_rng(seed_for(*keys))

np.random.seed(CONFIG["seed"])
import random as _pyrandom
_pyrandom.seed(CONFIG["seed"])

STATUS = {}
def set_status(name, status, reason="", **extra):
    STATUS[name] = dict(status=status, reason=reason, profile=PROFILE, **extra)
    (RESULTS / "logs" / "status.json").write_text(json.dumps(STATUS, indent=2, default=str))
    print(f"[STATUS] {name}: {status}" + (f" — {reason}" if reason else ""))

def jsonable(x):
    if isinstance(x, (np.integer,)): return int(x)
    if isinstance(x, (np.floating,)):
        v = float(x); return None if (math.isnan(v) or math.isinf(v)) else v
    if isinstance(x, float): return None if (math.isnan(x) or math.isinf(x)) else x
    if isinstance(x, np.ndarray): return [jsonable(v) for v in x.tolist()]
    if isinstance(x, dict): return {str(k): jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)): return [jsonable(v) for v in x]
    return x

def save_json(obj, path):
    Path(path).write_text(json.dumps(jsonable(obj), indent=2))

class Checkpoint:
    """Append-only JSONL checkpoint: one line per finished configuration key. Resume skips finished keys."""
    def __init__(self, name):
        self.path = RESULTS / "checkpoints" / f"{name}_{PROFILE}.jsonl"
        self.rows, self.keys = [], set()
        if self.path.exists():
            for line in self.path.read_text().splitlines():
                if line.strip():
                    rec = json.loads(line); self.keys.add(rec["_key"]); self.rows.extend(rec["rows"])
    def done(self, key):
        return str(key) in self.keys
    def add(self, key, rows):
        rows = [jsonable(r) for r in rows]
        with open(self.path, "a") as fh:
            fh.write(json.dumps({"_key": str(key), "rows": rows}) + "\n")
        self.keys.add(str(key)); self.rows.extend(rows)
    def frame(self):
        return pd.DataFrame(self.rows)

class Timer:
    def __enter__(self):
        self.t0 = time.perf_counter(); return self
    def __exit__(self, *a):
        self.dt = time.perf_counter() - self.t0

NOTEBOOK_T0 = time.perf_counter()


def loglog_slope(x, y):
    x = np.log(np.asarray(x, float)); y = np.log(np.asarray(y, float))
    A = np.stack([x, np.ones_like(x)], 1); coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    return float(coef[0])

In [4]:
DATASETS = {
    "synthetic": dict(source="generated in this notebook (Gaussian random fields, graphs, triangulated spheres and surfaces)", url=None,
                      license="n/a", version=CONFIG_HASH, note="deterministic given CONFIG['seed']"),
    "sdrbench": dict(source="SDRBench raw float32 fields (Zhao et al., IEEE BigData 2020)", url="https://sdrbench.github.io/datasets.html",
                     license="see SDRBench", version="archives on the SDRBench HTTPS host named in CONFIG['compression']"),
    "compressors": dict(source="SZ3, SZ and ZFP HDF5 filters from hdf5plugin (pip)", url="https://pypi.org/project/hdf5plugin/",
                        license="per filter (BSD-type)", version="hdf5plugin version recorded in ENV"),
}
MANIFEST_PATH = RESULTS / "data" / "manifest.json"
MANIFEST = json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else {}


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def _relative(path):
    """Paths are recorded relative to the results folder, so no manifest or log reveals the machine's directory layout."""
    try:
        return str(Path(path).resolve().relative_to(Path(RESULTS).resolve()))
    except Exception:
        return Path(path).name


def record_manifest(name, path=None, **extra):
    entry = dict(DATASETS.get(name.split(":")[0], {}), **extra, time=time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()))
    if path is not None and Path(path).exists():
        entry.update(path=_relative(path), sha256=sha256_file(path), bytes=Path(path).stat().st_size)
    MANIFEST[name] = entry
    save_json(MANIFEST, MANIFEST_PATH)


def stream_tar_members(url, dest_dir, n_members, accept_bytes=None, suffixes=(".f32", ".dat", ".bin"), max_bytes=3e9, timeout=60):
    """Stream a .tar.gz over HTTPS and extract the first `n_members` regular files that match (by exact byte size when `accept_bytes` is
    given, otherwise by suffix). Reading stops as soon as enough members are found."""
    import urllib.request
    dest_dir = Path(dest_dir); dest_dir.mkdir(parents=True, exist_ok=True)
    got = []
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        with tarfile.open(fileobj=resp, mode="r|gz") as tf:
            read = 0
            for member in tf:
                read += member.size
                if read > max_bytes:
                    break
                ok = member.isfile() and ((member.size == accept_bytes) if accept_bytes else member.name.endswith(tuple(suffixes)))
                if ok:
                    target = dest_dir / Path(member.name).name
                    with tf.extractfile(member) as src, open(target, "wb") as dst:
                        shutil.copyfileobj(src, dst)
                    got.append(target)
                    if len(got) >= n_members:
                        break
    return got

## 4 · Generators and the degree-0 algorithms

- **Generators** for graphs, grids, triangulated spheres and fields. Each is deterministic given an explicit random generator.
- **Lower-star persistence in degree 0**: union–find with the elder rule. Ties are broken by $(f(v), \mathrm{id}(v))$.
- **The degree-0 bound.** One sweep over the vertices in order of $\ell$ builds the merge forest of the sublevel sets of $\ell$. Each tree node records when its component first meets $\{u \le t\}$ ($\mu$) and when it merges ($\lambda$), which yields $\mathrm{Bar}_0(u\ge\ell)$. A top-down pass then gives the τ-counts. The trap test $\mu+\tau<\ell(w)$ is decided exactly for floating-point inputs.
- **Baselines** (independent implementations): the mandatory minima of Günther, Salmon and Tierny (τ = 0), the uniform-box count, and the rank certificate.
- **Implicit 2-D and 3-D grid kernels** for the scaling and compression experiments.
- **The image barcode and the reconstruction $f_U$.** $f_U(v)=\min_{w,\gamma}\max(u(w),\max_\gamma \ell)$ is the reconstruction by erosion of $u$ over $\ell$. It is computed from the merge forest, and Theorem 5 says that $\mathrm{Bar}_0(f_U)=\mathrm{Bar}_0(u\ge\ell)$.

In [5]:
@dataclass
class Graph:
    n: int
    indptr: np.ndarray
    indices: np.ndarray
    m: int
    name: str = ""
    triangles: object = None
    grid_shape: object = None

def csr_from_edges(n, edges, assume_unique=False):
    """Mathematical object: undirected simple graph (1-skeleton) → CSR arrays (both edge directions, neighbours sorted)."""
    e = np.asarray(edges, dtype=np.int64).reshape(-1, 2)
    if e.shape[0] == 0:
        return np.zeros(n + 1, np.int64), np.zeros(0, np.int64), 0
    a = np.minimum(e[:, 0], e[:, 1]); b = np.maximum(e[:, 0], e[:, 1])
    keep = a != b
    a, b = a[keep], b[keep]
    if not assume_unique:
        key = np.unique(a * np.int64(n) + b)
        a, b = key // n, key % n
    src = np.concatenate([a, b]); dst = np.concatenate([b, a])
    o = np.lexsort((dst, src))
    src, dst = src[o], dst[o]
    counts = np.bincount(src, minlength=n)
    indptr = np.zeros(n + 1, np.int64); np.cumsum(counts, out=indptr[1:])
    return indptr, dst.astype(np.int64), int(a.shape[0])

def graph_from_edges(n, edges, name="", triangles=None, grid_shape=None, assume_unique=False):
    ip, ix, m = csr_from_edges(n, edges, assume_unique=assume_unique)
    return Graph(n, ip, ix, m, name, triangles, grid_shape)

def grid_edges(H, W, conn):
    idx = np.arange(H * W, dtype=np.int64).reshape(H, W)
    E = [np.stack([idx[:, :-1].ravel(), idx[:, 1:].ravel()], 1), np.stack([idx[:-1, :].ravel(), idx[1:, :].ravel()], 1)]
    if conn in (6, 8):
        E.append(np.stack([idx[:-1, :-1].ravel(), idx[1:, 1:].ravel()], 1))
    if conn == 8:
        E.append(np.stack([idx[:-1, 1:].ravel(), idx[1:, :-1].ravel()], 1))
    return np.concatenate(E) if E else np.zeros((0, 2), np.int64)

def graph_grid(H, W, conn=6):
    tri = None
    if conn == 6:   # Freudenthal triangulation of the square grid: a genuine 2-dimensional simplicial complex
        idx = np.arange(H * W, dtype=np.int64).reshape(H, W)
        a, b, c, d = idx[:-1, :-1].ravel(), idx[:-1, 1:].ravel(), idx[1:, :-1].ravel(), idx[1:, 1:].ravel()
        tri = np.concatenate([np.stack([a, b, d], 1), np.stack([a, c, d], 1)])
    return graph_from_edges(H * W, grid_edges(H, W, conn), f"grid{H}x{W}c{conn}", tri, (H, W), assume_unique=True)

def grid_boundary(H, W):
    idx = np.arange(H * W).reshape(H, W)
    return np.unique(np.concatenate([idx[0], idx[-1], idx[:, 0], idx[:, -1]]))

def add_outlet(graph, boundary):
    """Cone the domain boundary to one extra 'outlet' vertex (index n): drainage off the edge of a terrain."""
    n = graph.n
    src = np.repeat(np.arange(n), np.diff(graph.indptr)); dst = graph.indices
    e = np.stack([src[src < dst], dst[src < dst]], 1)
    e2 = np.stack([np.full(len(boundary), n), np.asarray(boundary)], 1)
    return graph_from_edges(n + 1, np.concatenate([e, e2]), graph.name + "+outlet", grid_shape=graph.grid_shape)

def graph_path(n):
    return graph_from_edges(n, np.stack([np.arange(n - 1), np.arange(1, n)], 1) if n > 1 else np.zeros((0, 2)), f"path{n}")

def graph_cycle(n):
    if n < 3:
        return graph_path(n)
    return graph_from_edges(n, np.stack([np.arange(n), (np.arange(n) + 1) % n], 1), f"cycle{n}")

def graph_random_tree(n, rng):
    if n < 2:
        return graph_path(n)
    parents = np.array([rng.integers(0, i) for i in range(1, n)])
    return graph_from_edges(n, np.stack([np.arange(1, n), parents], 1), f"tree{n}")

def graph_er(n, p, rng):
    iu = np.triu_indices(n, 1)
    mask = rng.random(len(iu[0])) < p
    return graph_from_edges(n, np.stack([iu[0][mask], iu[1][mask]], 1), f"er{n}")

def graph_er_sparse(n, avg_deg, rng):
    """Large sparse random graph with about n*avg_deg/2 edges (duplicates removed)."""
    m = int(n * avg_deg / 2)
    e = rng.integers(0, n, size=(m, 2), dtype=np.int64)
    return graph_from_edges(n, e, f"ers{n}d{avg_deg}")

def graph_rgg(n, radius, rng):
    P = rng.random((n, 2))
    from scipy.spatial import cKDTree
    pairs = cKDTree(P).query_pairs(radius, output_type="ndarray")
    return graph_from_edges(n, pairs.reshape(-1, 2), f"rgg{n}")

def triangulated_sphere(k, rng):
    """Delaunay triangulation of k random points in the unit square, coned off at its convex hull: a triangulated S^2."""
    P = rng.random((k, 2))
    tri = Delaunay(P)
    T = [tuple(sorted(t)) for t in tri.simplices.tolist()]
    cone = k
    T += [tuple(sorted((a, b, cone))) for a, b in tri.convex_hull.tolist()]
    E = sorted({e for a, b, c in T for e in ((a, b), (a, c), (b, c))})
    nv = k + 1
    assert nv - len(E) + len(T) == 2, "Euler characteristic of the sphere must be 2"
    return graph_from_edges(nv, np.array(E), f"sphere{nv}", triangles=np.array(T))

def padded_sphere(k, del_frac, rng):
    """Padded sphere (Lemma 4): planar G = Delaunay graph minus random edges; each deleted edge is stellar-subdivided by a padding
    vertex, the hull is coned off. Result triangulates S^2 and G is the induced subgraph on the k original vertices."""
    P = rng.random((k, 2))
    tri = Delaunay(P)
    T = [tuple(sorted(t)) for t in tri.simplices.tolist()]
    cone = k; nv = k + 1
    T += [tuple(sorted((a, b, cone))) for a, b in tri.convex_hull.tolist()]
    DT = sorted({e for a, b, c in tri.simplices.tolist() for e in (tuple(sorted((a, b))), tuple(sorted((a, c))), tuple(sorted((b, c))))})
    deleted = [e for e in DT if rng.random() < del_frac]
    for (a, b) in deleted:
        p = nv; nv += 1
        newT = []
        for t in T:
            if a in t and b in t:
                x = [v for v in t if v != a and v != b][0]
                newT += [tuple(sorted((a, p, x))), tuple(sorted((p, b, x)))]
            else:
                newT.append(t)
        T = newT
    E = sorted({e for a, b, c in T for e in ((a, b), (a, c), (b, c))})
    assert nv - len(E) + len(T) == 2
    G = [e for e in DT if e not in set(deleted)]
    induced = {(a, b) for a, b in E if a < k and b < k}
    assert induced == set(G), "padding must leave G as the induced subgraph on original vertices"
    return k, graph_from_edges(nv, np.array(E), f"padded{nv}", triangles=np.array(T)), G

def grf_batch(rng, N, H, W, corr_len):
    """N independent stationary Gaussian random fields on an HxW grid (Gaussian spectral filter), each standardised."""
    noise = rng.standard_normal((N, H, W))
    ky = np.fft.fftfreq(H)[:, None]; kx = np.fft.rfftfreq(W)[None, :]
    filt = np.exp(-2.0 * (np.pi * corr_len) ** 2 * (kx ** 2 + ky ** 2))
    F = np.fft.irfft2(np.fft.rfft2(noise, axes=(1, 2)) * filt, s=(H, W), axes=(1, 2))
    F -= F.mean(axis=(1, 2), keepdims=True)
    F /= F.std(axis=(1, 2), keepdims=True)
    return F

def heteroscedastic_sigma(H, W, rho, sigma_min):
    """sigma(x) = sigma_min * rho**(col/(W-1)): a left-to-right log-linear ramp with ratio max/min = rho."""
    col = np.arange(W) / max(W - 1, 1)
    return np.tile(sigma_min * rho ** col, (H, 1))

# --- structured families with known answers, used by the unit tests --------------------------------------------------------
def staircase_path(K, depth=10, rim=1.5):
    """Staircase: global minimum z then K basins a_k at depth 10k, each with rim a_k + rim.
    Exact values (ell = u). With tau = 1 each basin has persistence 1.5 and disjoint rank windows."""
    vals = [-5.0]
    for k in range(K):
        vals += [depth * k + rim, depth * k]
    return graph_path(len(vals)), np.array(vals)

def comb_path(K, tooth=(0.0, 1.0), ridge=(0.8, 1.8)):
    """K equal-depth teeth separated by ridges whose intervals overlap the teeth: T_tau = 1 for tau >= 0.5 although
    typical realisations show many basins."""
    lo, hi = [], []
    for k in range(K):
        lo.append(tooth[0]); hi.append(tooth[1])
        if k < K - 1:
            lo.append(ridge[0]); hi.append(ridge[1])
    return graph_path(len(lo)), np.array(lo), np.array(hi)

def nested_path(D, step=3.0, pocket=1.5):
    """Deep sweep forest: pockets p_d = step*d - pocket between ridges r_{d+1} > r_d = step*d, walls to the right.
    With exact values and tau < pocket every pocket and the centre are tau-persistent, so T_tau = D + 1."""
    left = []
    for d in range(D, 0, -1):
        left += [step * d - pocket, step * d]
    right = [step * d + 1.0 for d in range(1, D + 1)]
    vals = left + [0.0] + right
    return graph_path(len(vals)), np.array(vals)

In [6]:
@njit
def _find(p, x):
    r = x
    while p[r] != r:
        r = p[r]
    while p[x] != r:
        nx = p[x]; p[x] = r; x = nx
    return r

@njit
def _find_count(p, x, ops):
    r = x
    while p[r] != r:
        r = p[r]; ops[0] += 1
    while p[x] != r:
        nx = p[x]; p[x] = r; x = nx; ops[0] += 1
    return r

@njit
def _lt_sum(a, b, c):
    """Exact decision of a + b < c for finite float64 a, b, c. Two error-free sums build the
    non-overlapping expansion of a + b - c (Shewchuk's grow-expansion); its sign is the sign of its largest
    non-zero component, so rounding can never flip the trap test mu + tau < ell(w)."""
    s = a + b
    bv = s - a
    e = (a - (s - bv)) + (b - bv)          # a + b = s + e exactly
    q = -c
    x = q + e
    bv = x - q
    h0 = (q - (x - bv)) + (e - bv)
    q = x
    x = q + s
    bv = x - q
    h1 = (q - (x - bv)) + (s - bv)
    q = x                                  # a + b - c = h0 + h1 + q exactly (increasing magnitude)
    if q != 0.0:
        return q < 0.0
    if h1 != 0.0:
        return h1 < 0.0
    return h0 < 0.0

@njit
def persistence0(indptr, indices, f, order):
    n = f.shape[0]
    par = np.full(n, -1, np.int64); size = np.zeros(n, np.int64)
    bv = np.empty(n, np.int64); pos = np.empty(n, np.int64)
    for i in range(n):
        pos[order[i]] = i
    death = np.full(n, np.nan); isb = np.zeros(n, np.bool_)
    stamp = np.full(n, -1, np.int64)
    maxdeg = 0
    for v in range(n):
        d = indptr[v + 1] - indptr[v]
        if d > maxdeg:
            maxdeg = d
    roots = np.empty(maxdeg + 1, np.int64)
    for i in range(n):
        v = order[i]
        cnt = 0
        for q in range(indptr[v], indptr[v + 1]):
            x = indices[q]
            if par[x] != -1:
                r = _find(par, x)
                if stamp[r] != i:
                    stamp[r] = i; roots[cnt] = r; cnt += 1
        par[v] = v; size[v] = 1
        if cnt == 0:
            bv[v] = v; isb[v] = True
            continue
        e = roots[0]
        for j in range(1, cnt):
            if pos[bv[roots[j]]] < pos[bv[e]]:
                e = roots[j]
        eld = bv[e]
        for j in range(cnt):
            r = roots[j]
            if r != e:
                death[bv[r]] = f[v]
        root = v
        for j in range(cnt):
            r = roots[j]
            if size[r] > size[root]:
                par[root] = r; size[r] += size[root]; root = r
            else:
                par[r] = root; size[root] += size[r]
        bv[root] = eld
    for v in range(n):
        if isb[v] and np.isnan(death[v]):
            death[v] = np.inf
    return death, isb

def order_of(x):
    return np.argsort(x, kind="stable")

def diagram0(graph, f):
    """Return (birth_vertices, births, deaths) of Dgm_0(f)."""
    f = np.asarray(f, dtype=np.float64)
    death, isb = persistence0(graph.indptr, graph.indices, f, order_of(f))
    bvtx = np.nonzero(isb)[0]
    return bvtx, f[bvtx], death[bvtx]

def n_tau(graph, f, tau):
    _, b, d = diagram0(graph, f)
    return int(np.sum((d - b) > tau))

@njit
def _count_tau(indptr, indices, f, tau):
    death, isb = persistence0(indptr, indices, f, np.argsort(f, kind="mergesort"))
    c = 0
    for v in range(f.shape[0]):
        if isb[v] and death[v] - f[v] > tau:
            c += 1
    return c

In [7]:
@njit
def _certify_sweep(indptr, indices, ell, u, tau, order, count_ops, exact):
    n = ell.shape[0]
    uf = np.full(n, -1, np.int64); ufs = np.zeros(n, np.int64)
    cnode = np.empty(n, np.int64)
    npar = np.full(n, -1, np.int64)
    nmu = np.empty(n); nlam = np.full(n, np.inf); nminl = np.empty(n)
    ntrap = np.zeros(n, np.bool_); nmin = np.zeros(n, np.bool_)
    nmct = np.full(n, -np.inf); nvert = np.empty(n, np.int64)
    Dv = np.empty(n, np.int64); stamp = np.full(n, -1, np.int64)
    ops = np.zeros(1, np.int64)
    maxdeg = 0
    for v in range(n):
        d = indptr[v + 1] - indptr[v]
        if d > maxdeg:
            maxdeg = d
    roots = np.empty(maxdeg + 1, np.int64)
    T = 0; nn = 0
    for i in range(n):
        w = order[i]
        cnt = 0
        for q in range(indptr[w], indptr[w + 1]):
            x = indices[q]
            if uf[x] != -1:
                r = _find_count(uf, x, ops) if count_ops else _find(uf, x)
                if stamp[r] != i:
                    stamp[r] = i; roots[cnt] = r; cnt += 1
        D = nn; nn += 1
        nvert[D] = w
        mu = u[w]; me = ell[w]; tr = False; mct = -np.inf
        for j in range(cnt):
            c = cnode[roots[j]]
            nlam[c] = ell[w]                                   # lambda[r] <- ell(w)
            if (not ntrap[c]) and (_lt_sum(nmu[c], tau, ell[w]) if exact else (nmu[c] + tau < ell[w])):   # exact trap test
                ntrap[c] = True; nmin[c] = True; T += 1
            npar[c] = D
            if nmu[c] < mu:
                mu = nmu[c]
            if nminl[c] < me:
                me = nminl[c]
            if ntrap[c]:
                tr = True
            tc = nlam[c] - nmu[c]
            if tc > mct:
                mct = tc
        nmu[D] = mu; nminl[D] = me; ntrap[D] = tr; nmct[D] = mct
        Dv[w] = D
        uf[w] = w; ufs[w] = 1
        root = w
        for j in range(cnt):
            r = roots[j]
            ops[0] += 1
            if ufs[r] > ufs[root]:
                uf[root] = r; ufs[r] += ufs[root]; root = r
            else:
                uf[r] = root; ufs[root] += ufs[r]
        cnode[root] = D
    for k in range(nn):                                        # roots of F: lambda = infinity
        if npar[k] == -1:
            nlam[k] = np.inf
            if not ntrap[k]:
                ntrap[k] = True; nmin[k] = True; T += 1
    return T, nn, npar, nmu, nlam, nminl, ntrap, nmin, nmct, nvert, Dv, ops[0]

@njit
def _certify_topdown(nn, npar, nlam, ntrap, Dv, ell, tau):
    val = np.empty(nn)
    for k in range(nn - 1, -1, -1):
        if ntrap[k]:
            val[k] = -np.inf
        else:
            p = npar[k]
            val[k] = nlam[k] if ntrap[p] else val[p]
    n = ell.shape[0]
    fs = np.empty(n)
    for v in range(n):
        a = val[Dv[v]] - tau
        fs[v] = ell[v] if ell[v] >= a else a
    return fs, val

@njit
def _membership(nn, npar, ntrap, nmin, Dv):
    mt = np.full(nn, -1, np.int64)
    for k in range(nn - 1, -1, -1):
        if nmin[k]:
            mt[k] = k
        elif ntrap[k]:
            mt[k] = -1
        else:
            p = npar[k]
            if nmin[p]:
                mt[k] = p
            elif ntrap[p]:
                mt[k] = -1
            else:
                mt[k] = mt[p]
    memb = np.empty(Dv.shape[0], np.int64)
    for v in range(Dv.shape[0]):
        memb[v] = mt[Dv[v]]
    return memb

@dataclass
class Certificate:
    T: int
    tau: float
    parent: np.ndarray
    mu: np.ndarray
    lam: np.ndarray
    minell: np.ndarray
    trap: np.ndarray
    minimal: np.ndarray
    maxchild_tau: np.ndarray
    vertex: np.ndarray
    Dv: np.ndarray
    fstar: np.ndarray
    ops: int = 0
    _memb: object = None

    @property
    def minimal_ids(self):
        return np.nonzero(self.minimal)[0]

    def membership(self):
        if self._memb is None:
            self._memb = _membership(len(self.parent), self.parent, self.trap, self.minimal, self.Dv)
        return self._memb

    def quadrants(self):
        """Q(M) = [min_M ell, mu(M)] x [lambda(M), inf] for every minimal trap M."""
        ids = self.minimal_ids
        return np.stack([self.minell[ids], self.mu[ids], self.lam[ids]], 1) if len(ids) else np.zeros((0, 3))

    def tau_intervals(self):
        """tau-curve: node x contributes [max_child tau_c, tau_x)."""
        return self.maxchild_tau.copy(), (self.lam - self.mu)

def certify_graph(graph, ell, u, tau, count_ops=False, exact=True):
    ell = np.ascontiguousarray(ell, dtype=np.float64); u = np.ascontiguousarray(u, dtype=np.float64)
    if np.any(ell > u) or not (np.all(np.isfinite(ell)) and np.all(np.isfinite(u))):
        raise ValueError("A1 violated: need finite ell <= u (empty or ill-defined box)")
    order = order_of(ell)
    T, nn, npar, nmu, nlam, nminl, ntrap, nmin, nmct, nvert, Dv, ops = _certify_sweep(
        graph.indptr, graph.indices, ell, u, float(tau), order, count_ops, exact)
    fs, _ = _certify_topdown(nn, npar, nlam, ntrap, Dv, ell, float(tau))
    return Certificate(int(T), float(tau), npar[:nn], nmu[:nn], nlam[:nn], nminl[:nn], ntrap[:nn], nmin[:nn],
                       nmct[:nn], nvert[:nn], Dv, fs, int(ops))

def tau_curve(cert, taus):
    lo, hi = cert.tau_intervals()
    lo_s = np.sort(lo); hi_s = np.sort(hi)
    taus = np.asarray(taus, dtype=np.float64)
    return np.searchsorted(lo_s, taus, side="right") - np.searchsorted(hi_s, taus, side="right")

# ---- implicit-neighbour grid kernel for the scaling experiment (no CSR arrays; identical semantics) -------------------------
@njit
def _grid_offsets(conn):
    if conn == 4:
        return np.array([-1, 1, 0, 0], np.int64), np.array([0, 0, -1, 1], np.int64)
    if conn == 6:
        return np.array([-1, 1, 0, 0, 1, -1], np.int64), np.array([0, 0, -1, 1, 1, -1], np.int64)
    return np.array([-1, 1, 0, 0, 1, -1, 1, -1], np.int64), np.array([0, 0, -1, 1, 1, -1, -1, 1], np.int64)

@njit
def _certify_grid_sweep(H, W, conn, ell, u, tau, order, exact):
    n = H * W
    dr, dc = _grid_offsets(conn)
    uf = np.full(n, -1, np.int64); ufs = np.zeros(n, np.int64)
    cnode = np.empty(n, np.int64)
    npar = np.full(n, -1, np.int64)
    nmu = np.empty(n); nlam = np.full(n, np.inf); nminl = np.empty(n)
    ntrap = np.zeros(n, np.bool_); nmin = np.zeros(n, np.bool_)
    nmct = np.full(n, -np.inf); nvert = np.empty(n, np.int64)
    Dv = np.empty(n, np.int64)
    ops = np.zeros(1, np.int64)
    roots = np.empty(dr.shape[0], np.int64)
    T = 0; nn = 0
    for i in range(n):
        w = order[i]
        r0 = w // W; c0 = w - r0 * W
        cnt = 0
        for t in range(dr.shape[0]):
            rr = r0 + dr[t]; cc = c0 + dc[t]
            if rr < 0 or rr >= H or cc < 0 or cc >= W:
                continue
            x = rr * W + cc
            if uf[x] != -1:
                r = _find_count(uf, x, ops)
                dup = False
                for j in range(cnt):
                    if roots[j] == r:
                        dup = True; break
                if not dup:
                    roots[cnt] = r; cnt += 1
        D = nn; nn += 1
        nvert[D] = w
        mu = u[w]; me = ell[w]; tr = False; mct = -np.inf
        for j in range(cnt):
            c = cnode[roots[j]]
            nlam[c] = ell[w]
            if (not ntrap[c]) and (_lt_sum(nmu[c], tau, ell[w]) if exact else (nmu[c] + tau < ell[w])):
                ntrap[c] = True; nmin[c] = True; T += 1
            npar[c] = D
            if nmu[c] < mu:
                mu = nmu[c]
            if nminl[c] < me:
                me = nminl[c]
            if ntrap[c]:
                tr = True
            tc = nlam[c] - nmu[c]
            if tc > mct:
                mct = tc
        nmu[D] = mu; nminl[D] = me; ntrap[D] = tr; nmct[D] = mct
        Dv[w] = D
        uf[w] = w; ufs[w] = 1
        root = w
        for j in range(cnt):
            r = roots[j]
            ops[0] += 1
            if ufs[r] > ufs[root]:
                uf[root] = r; ufs[r] += ufs[root]; root = r
            else:
                uf[r] = root; ufs[root] += ufs[r]
        cnode[root] = D
    for k in range(nn):
        if npar[k] == -1:
            nlam[k] = np.inf
            if not ntrap[k]:
                ntrap[k] = True; nmin[k] = True; T += 1
    return T, nn, npar, nmu, nlam, nminl, ntrap, nmin, nmct, nvert, Dv, ops[0]

def certify_grid(ell2d, u2d, tau, conn=4, return_timing=False, exact=True):
    H, W = ell2d.shape
    ell = np.ascontiguousarray(ell2d.ravel(), dtype=np.float64); u = np.ascontiguousarray(u2d.ravel(), dtype=np.float64)
    t0 = time.perf_counter(); order = order_of(ell); t1 = time.perf_counter()
    out = _certify_grid_sweep(H, W, conn, ell, u, float(tau), order, exact); t2 = time.perf_counter()
    T, nn, npar, nmu, nlam, nminl, ntrap, nmin, nmct, nvert, Dv, ops = out
    fs, _ = _certify_topdown(nn, npar, nlam, ntrap, Dv, ell, float(tau)); t3 = time.perf_counter()
    cert = Certificate(int(T), float(tau), npar[:nn], nmu[:nn], nlam[:nn], nminl[:nn], ntrap[:nn], nmin[:nn],
                       nmct[:nn], nvert[:nn], Dv, fs, int(ops))
    if return_timing:
        return cert, dict(t_sort=t1 - t0, t_sweep=t2 - t1, t_topdown=t3 - t2, t_total=t3 - t0)
    return cert

@njit
def _persistence_grid(H, W, conn, f, order):
    """persistence0 for the implicit HxW grid (same semantics, no CSR arrays)."""
    n = H * W
    dr, dc = _grid_offsets(conn)
    par = np.full(n, -1, np.int64); size = np.zeros(n, np.int64)
    bv = np.empty(n, np.int64); pos = np.empty(n, np.int64)
    for i in range(n):
        pos[order[i]] = i
    death = np.full(n, np.nan); isb = np.zeros(n, np.bool_)
    roots = np.empty(dr.shape[0], np.int64)
    for i in range(n):
        v = order[i]
        r0 = v // W; c0 = v - r0 * W
        cnt = 0
        for t in range(dr.shape[0]):
            rr = r0 + dr[t]; cc = c0 + dc[t]
            if rr < 0 or rr >= H or cc < 0 or cc >= W:
                continue
            x = rr * W + cc
            if par[x] != -1:
                r = _find(par, x)
                dup = False
                for j in range(cnt):
                    if roots[j] == r:
                        dup = True; break
                if not dup:
                    roots[cnt] = r; cnt += 1
        par[v] = v; size[v] = 1
        if cnt == 0:
            bv[v] = v; isb[v] = True
            continue
        e = roots[0]
        for j in range(1, cnt):
            if pos[bv[roots[j]]] < pos[bv[e]]:
                e = roots[j]
        eld = bv[e]
        for j in range(cnt):
            if roots[j] != e:
                death[bv[roots[j]]] = f[v]
        root = v
        for j in range(cnt):
            r = roots[j]
            if size[r] > size[root]:
                par[root] = r; size[r] += size[root]; root = r
            else:
                par[r] = root; size[root] += size[r]
        bv[root] = eld
    for v in range(n):
        if isb[v] and np.isnan(death[v]):
            death[v] = np.inf
    return death, isb

def persistence_grid(f2d, conn):
    H, W = f2d.shape
    f = np.ascontiguousarray(f2d.ravel(), np.float64)
    death, isb = _persistence_grid(H, W, conn, f, order_of(f))
    return f, death, isb

def n_tau_grid(f2d, tau, conn):
    f, death, isb = persistence_grid(f2d, conn)
    return int(np.sum(isb & ((death - f) > tau)))

# ---- pure-Python reference implementation (exact arithmetic with fractions.Fraction) ----------------------------------------
def _adjacency(graph):
    return [graph.indices[graph.indptr[v]:graph.indptr[v + 1]].tolist() for v in range(graph.n)]

def certify_reference(adj, ell, u, tau):
    """Direct transcription of CERTIFY-H0 on Python numbers (Fractions for exact replay). Returns (T, fstar)."""
    n = len(ell)
    order = sorted(range(n), key=lambda v: (ell[v], v))
    uf = [-1] * n
    def find(x):
        while uf[x] != x:
            uf[x] = uf[uf[x]]; x = uf[x]
        return x
    cnode = {}; par, mu, lam, trap, Dv = [], [], [], [], [None] * n
    T = 0
    for w in order:
        R = []
        for x in adj[w]:
            if uf[x] != -1:
                r = find(x)
                if r not in R:
                    R.append(r)
        D = len(par); par.append(-1); lam.append(None); trap.append(False)
        m = u[w]; tr = False
        for r in R:
            c = cnode[r]; lam[c] = ell[w]
            if (not trap[c]) and (mu[c] + tau < ell[w]):
                trap[c] = True; T += 1
            par[c] = D; m = min(m, mu[c]); tr = tr or trap[c]
        mu.append(m); trap[D] = tr; Dv[w] = D
        uf[w] = w
        for r in R:
            uf[r] = w
        cnode[w] = D
    for k in range(len(par)):
        if par[k] == -1:
            lam[k] = None
            if not trap[k]:
                trap[k] = True; T += 1
    val = [None] * len(par)
    for k in range(len(par) - 1, -1, -1):
        if trap[k]:
            val[k] = None
        else:
            p = par[k]
            val[k] = lam[k] if trap[p] else val[p]
    fstar = [ell[v] if val[Dv[v]] is None else max(ell[v], val[Dv[v]] - tau) for v in range(n)]
    return T, fstar

def ncount_reference(adj, f, tau):
    """Exact N_tau(f) with Python numbers (elder rule, lexicographic ties)."""
    n = len(f)
    order = sorted(range(n), key=lambda v: (f[v], v)); pos = {v: i for i, v in enumerate(order)}
    uf = [-1] * n; bv = [None] * n; births = {}
    def find(x):
        while uf[x] != x:
            uf[x] = uf[uf[x]]; x = uf[x]
        return x
    for v in order:
        R = []
        for x in adj[v]:
            if uf[x] != -1:
                r = find(x)
                if r not in R:
                    R.append(r)
        uf[v] = v
        if not R:
            bv[v] = v; births[v] = None
            continue
        e = min(R, key=lambda r: pos[bv[r]])
        for r in R:
            if r != e:
                births[bv[r]] = f[v]
        eld = bv[e]
        for r in R:
            uf[r] = v
        bv[v] = eld
    c = 0
    for b, d in births.items():
        if d is None or d - f[b] > tau:
            c += 1
    return c

# ---- checks of the certificate event on a realisation f ---------------------------------------------------------------------
def lemma1_check(cert, f, death, isb):
    """Map every minimal trap M to the lexicographic argmin of f on M, given the persistence arrays (death, isb) of f.
    Returns the number of minimal traps whose image is a birth with b in [min_M ell, mu(M)] and d >= lambda(M)."""
    ids = cert.minimal_ids
    if len(ids) == 0:
        return 0
    memb = cert.membership()
    o = order_of(f)
    mo = memb[o]
    sel = mo >= 0
    firsts_trap, firsts_idx = np.unique(mo[sel], return_index=True)   # first occurrence in (f,id) order
    arg = o[np.nonzero(sel)[0][firsts_idx]]
    v = arg
    ok = isb[v] & (cert.minell[firsts_trap] <= f[v]) & (f[v] <= cert.mu[firsts_trap]) & (death[v] >= cert.lam[firsts_trap])
    return int(np.sum(ok))

def lemma1_injection(cert, graph, f):
    f = np.asarray(f, np.float64)
    death, isb = persistence0(graph.indptr, graph.indices, f, order_of(f))
    return lemma1_check(cert, f, death, isb)

def quadrant_matching(cert, graph, f, tau):
    """Size of a maximum matching between minimal traps and tau-persistent diagram points p in Q(M)."""
    _, b, d = diagram0(graph, f)
    keep = (d - b) > tau
    b, d = b[keep], d[keep]
    Q = cert.quadrants()
    if len(Q) == 0:
        return 0
    A = (Q[:, 0][:, None] <= b[None, :]) & (b[None, :] <= Q[:, 1][:, None]) & (d[None, :] >= Q[:, 2][:, None])
    if A.shape[1] == 0:
        return 0
    match = maximum_bipartite_matching(csr_matrix(A.astype(np.int8)), perm_type="column")
    return int(np.sum(match >= 0))

In [8]:
@njit
def _gst14(indptr, indices, ell, u):
    n = ell.shape[0]
    uo = np.argsort(u, kind="mergesort"); ur = np.empty(n, np.int64)
    for i in range(n):
        ur[uo[i]] = i
    is_min = np.ones(n, np.bool_)
    for v in range(n):
        for q in range(indptr[v], indptr[v + 1]):
            if ur[indices[q]] < ur[v]:
                is_min[v] = False; break
    lo = np.argsort(ell, kind="mergesort")
    par = np.full(n, -1, np.int64); cnt = np.zeros(n, np.int64)
    ptr = 0; valid = 0
    for i in range(n):
        m = uo[i]
        if not is_min[m]:
            continue
        while ptr < n and ell[lo[ptr]] <= u[m]:
            v = lo[ptr]; par[v] = v
            for q in range(indptr[v], indptr[v + 1]):
                x = indices[q]
                if par[x] != -1:
                    a = _find(par, v); b = _find(par, x)
                    if a != b:
                        par[a] = b; cnt[b] += cnt[a]
            ptr += 1
        r = _find(par, m)
        if cnt[r] == 0:
            valid += 1
        cnt[r] += 1
    return valid

def gst14_count(graph, ell, u):
    return int(_gst14(graph.indptr, graph.indices, np.asarray(ell, np.float64), np.asarray(u, np.float64)))

def uniform_band_certificate(graph, fhat, delta, tau):
    """B2 diagram rule on the point estimate."""
    _, b, d = diagram0(graph, fhat)
    return int(np.sum((d - b) > tau + 2.0 * delta))

@njit
def _mc_counts(indptr, indices, ell, u, tau, K, seed, mode):
    np.random.seed(seed)
    n = ell.shape[0]
    out = np.empty(K, np.int64); f = np.empty(n)
    for k in range(K):
        for v in range(n):
            if mode == 0:
                f[v] = ell[v] + np.random.random() * (u[v] - ell[v])
            else:
                f[v] = ell[v] if np.random.random() < 0.5 else u[v]
        out[k] = _count_tau(indptr, indices, f, tau)
    return out

def mc_counts(graph, ell, u, tau, K, seed, mode="uniform"):
    return _mc_counts(graph.indptr, graph.indices, np.asarray(ell, np.float64), np.asarray(u, np.float64),
                      float(tau), int(K), int(seed) % (2 ** 31), 0 if mode == "uniform" else 1)

@njit
def counts_batch(indptr, indices, F, tau):
    out = np.empty(F.shape[0], np.int64)
    for k in range(F.shape[0]):
        out[k] = _count_tau(indptr, indices, F[k].copy(), tau)
    return out

@njit
def _fen_add(tree, i, val):
    i += 1
    while i < tree.shape[0]:
        tree[i] += val; i += i & (-i)

@njit
def _fen_prefix(tree, i):          # sum over positions [0, i)
    s = 0
    while i > 0:
        s += tree[i]; i -= i & (-i)
    return s

@njit
def _rank_certificate(indptr, indices, ell, u, tau):
    n = ell.shape[0]
    uo = np.argsort(u, kind="mergesort"); posu = np.empty(n, np.int64)
    for i in range(n):
        posu[uo[i]] = i
    su = u[uo]
    tree = np.zeros(n + 1, np.int64)
    lo = np.argsort(ell, kind="mergesort")
    par = np.full(n, -1, np.int64); size = np.zeros(n, np.int64); minpos = np.empty(n, np.int64)
    stamp = np.full(n, -1, np.int64)
    maxdeg = 0
    for v in range(n):
        d = indptr[v + 1] - indptr[v]
        if d > maxdeg:
            maxdeg = d
    roots = np.empty(maxdeg + 1, np.int64)
    best = 0
    for i in range(n):
        w = lo[i]
        cnt = 0
        for q in range(indptr[w], indptr[w + 1]):
            x = indices[q]
            if par[x] != -1:
                r = _find(par, x)
                if stamp[r] != i:
                    stamp[r] = i; roots[cnt] = r; cnt += 1
        mp = posu[w]
        for j in range(cnt):
            r = roots[j]
            _fen_add(tree, minpos[r], -1)
            if minpos[r] < mp:
                mp = minpos[r]
        par[w] = w; size[w] = 1; root = w
        for j in range(cnt):
            r = roots[j]
            if size[r] > size[root]:
                par[root] = r; size[r] += size[root]; root = r
            else:
                par[r] = root; size[root] += size[r]
        minpos[root] = mp
        _fen_add(tree, mp, 1)
        if i == n - 1:
            thr = np.inf
        elif ell[lo[i + 1]] > ell[w]:
            thr = ell[lo[i + 1]] - tau
        else:
            continue
        k = np.searchsorted(su, thr) if thr < np.inf else n   # positions with u < thr
        c = _fen_prefix(tree, k)
        if c > best:
            best = c
    return best

def rank_certificate(graph, ell, u, tau):
    return int(_rank_certificate(graph.indptr, graph.indices, np.asarray(ell, np.float64),
                                 np.asarray(u, np.float64), float(tau)))

def postfilter_count(graph, ell, u, tau):
    """Ablation: minimal 0-traps (tau removed from the trap test), then keep those with mu + tau < lambda."""
    c0 = certify_graph(graph, ell, u, 0.0)
    ids = c0.minimal_ids
    return int(np.sum(c0.mu[ids] + tau < c0.lam[ids]))

@njit
def _heap_push(hk, hv, size, key, val):
    i = size
    hk[i] = key; hv[i] = val
    while i > 0:
        p = (i - 1) // 2
        if hk[p] <= hk[i]:
            break
        hk[p], hk[i] = hk[i], hk[p]; hv[p], hv[i] = hv[i], hv[p]; i = p
    return size + 1

@njit
def _heap_pop(hk, hv, size):
    k = hk[0]; v = hv[0]
    size -= 1
    hk[0] = hk[size]; hv[0] = hv[size]
    i = 0
    while True:
        l = 2 * i + 1; r = l + 1; s = i
        if l < size and hk[l] < hk[s]:
            s = l
        if r < size and hk[r] < hk[s]:
            s = r
        if s == i:
            break
        hk[s], hk[i] = hk[i], hk[s]; hv[s], hv[i] = hv[i], hv[s]; i = s
    return k, v, size

@njit
def _priority_flood(z, conn):
    H, W = z.shape
    dr, dc = _grid_offsets(conn)
    filled = z.copy(); vis = np.zeros((H, W), np.bool_)
    hk = np.empty(H * W); hv = np.empty(H * W, np.int64); size = 0
    for r in range(H):
        for c in range(W):
            if r == 0 or c == 0 or r == H - 1 or c == W - 1:
                vis[r, c] = True; size = _heap_push(hk, hv, size, z[r, c], r * W + c)
    while size > 0:
        key, idx, size = _heap_pop(hk, hv, size)
        r0 = idx // W; c0 = idx - r0 * W
        for t in range(dr.shape[0]):
            rr = r0 + dr[t]; cc = c0 + dc[t]
            if rr < 0 or rr >= H or cc < 0 or cc >= W or vis[rr, cc]:
                continue
            vis[rr, cc] = True
            filled[rr, cc] = max(z[rr, cc], filled[r0, c0])
            size = _heap_push(hk, hv, size, filled[rr, cc], rr * W + cc)
    return filled

def priority_flood_depressions(z2d, tau, conn=8):
    """B6: maximal depressions of z (boundary drains) with depth > tau."""
    filled = _priority_flood(np.ascontiguousarray(z2d, np.float64), conn)
    depth = filled - z2d
    mask = depth > 0
    struct = np.ones((3, 3)) if conn == 8 else None
    lab, k = ndimage.label(mask, structure=struct)
    if k == 0:
        return 0
    dmax = ndimage.maximum(depth, lab, index=np.arange(1, k + 1))
    return int(np.sum(np.asarray(dmax) > tau))

In [9]:
@njit
def _nd_offsets(ndim, conn):
    if ndim == 2:
        dr, dc = _grid_offsets(conn)
        return np.zeros(dr.shape[0], np.int64), dr, dc
    if conn == 6:
        return (np.array([-1, 1, 0, 0, 0, 0], np.int64), np.array([0, 0, -1, 1, 0, 0], np.int64),
                np.array([0, 0, 0, 0, -1, 1], np.int64))
    dz = np.empty(26, np.int64); dy = np.empty(26, np.int64); dx = np.empty(26, np.int64); k = 0
    for a in range(-1, 2):
        for b in range(-1, 2):
            for c in range(-1, 2):
                if a == 0 and b == 0 and c == 0:
                    continue
                dz[k] = a; dy[k] = b; dx[k] = c; k += 1
    return dz, dy, dx

@njit
def _certify_nd_sweep(Dd, H, W, dz, dy, dx, ell, u, tau, order, exact):
    n = Dd * H * W; HW = H * W
    uf = np.full(n, -1, np.int64); ufs = np.zeros(n, np.int64)
    cnode = np.empty(n, np.int64)
    npar = np.full(n, -1, np.int64)
    nmu = np.empty(n); nlam = np.full(n, np.inf); nminl = np.empty(n)
    ntrap = np.zeros(n, np.bool_); nmin = np.zeros(n, np.bool_)
    nmct = np.full(n, -np.inf); nvert = np.empty(n, np.int64)
    Dv = np.empty(n, np.int64)
    ops = np.zeros(1, np.int64)
    K = dz.shape[0]
    roots = np.empty(K, np.int64)
    T = 0; nn = 0
    for i in range(n):
        w = order[i]
        z0 = w // HW; rem = w - z0 * HW; r0 = rem // W; c0 = rem - r0 * W
        cnt = 0
        for t in range(K):
            zz = z0 + dz[t]; rr = r0 + dy[t]; cc = c0 + dx[t]
            if zz < 0 or zz >= Dd or rr < 0 or rr >= H or cc < 0 or cc >= W:
                continue
            x = zz * HW + rr * W + cc
            if uf[x] != -1:
                r = _find_count(uf, x, ops)
                dup = False
                for j in range(cnt):
                    if roots[j] == r:
                        dup = True; break
                if not dup:
                    roots[cnt] = r; cnt += 1
        D = nn; nn += 1
        nvert[D] = w
        mu = u[w]; me = ell[w]; tr = False; mct = -np.inf
        for j in range(cnt):
            c = cnode[roots[j]]
            nlam[c] = ell[w]
            if (not ntrap[c]) and (_lt_sum(nmu[c], tau, ell[w]) if exact else (nmu[c] + tau < ell[w])):
                ntrap[c] = True; nmin[c] = True; T += 1
            npar[c] = D
            if nmu[c] < mu:
                mu = nmu[c]
            if nminl[c] < me:
                me = nminl[c]
            if ntrap[c]:
                tr = True
            tc = nlam[c] - nmu[c]
            if tc > mct:
                mct = tc
        nmu[D] = mu; nminl[D] = me; ntrap[D] = tr; nmct[D] = mct
        Dv[w] = D
        uf[w] = w; ufs[w] = 1
        root = w
        for j in range(cnt):
            r = roots[j]
            ops[0] += 1
            if ufs[r] > ufs[root]:
                uf[root] = r; ufs[r] += ufs[root]; root = r
            else:
                uf[r] = root; ufs[root] += ufs[r]
        cnode[root] = D
    for k in range(nn):
        if npar[k] == -1:
            nlam[k] = np.inf
            if not ntrap[k]:
                ntrap[k] = True; nmin[k] = True; T += 1
    return T, nn, npar, nmu, nlam, nminl, ntrap, nmin, nmct, nvert, Dv, ops[0]

@njit
def _persistence_nd(Dd, H, W, dz, dy, dx, f, order):
    n = Dd * H * W; HW = H * W
    par = np.full(n, -1, np.int64); size = np.zeros(n, np.int64)
    bv = np.empty(n, np.int64); pos = np.empty(n, np.int64)
    for i in range(n):
        pos[order[i]] = i
    death = np.full(n, np.nan); isb = np.zeros(n, np.bool_)
    K = dz.shape[0]
    roots = np.empty(K, np.int64)
    for i in range(n):
        v = order[i]
        z0 = v // HW; rem = v - z0 * HW; r0 = rem // W; c0 = rem - r0 * W
        cnt = 0
        for t in range(K):
            zz = z0 + dz[t]; rr = r0 + dy[t]; cc = c0 + dx[t]
            if zz < 0 or zz >= Dd or rr < 0 or rr >= H or cc < 0 or cc >= W:
                continue
            x = zz * HW + rr * W + cc
            if par[x] != -1:
                r = _find(par, x)
                dup = False
                for j in range(cnt):
                    if roots[j] == r:
                        dup = True; break
                if not dup:
                    roots[cnt] = r; cnt += 1
        par[v] = v; size[v] = 1
        if cnt == 0:
            bv[v] = v; isb[v] = True
            continue
        e = roots[0]
        for j in range(1, cnt):
            if pos[bv[roots[j]]] < pos[bv[e]]:
                e = roots[j]
        eld = bv[e]
        for j in range(cnt):
            if roots[j] != e:
                death[bv[roots[j]]] = f[v]
        root = v
        for j in range(cnt):
            r = roots[j]
            if size[r] > size[root]:
                par[root] = r; size[r] += size[root]; root = r
            else:
                par[r] = root; size[root] += size[r]
        bv[root] = eld
    for v in range(n):
        if isb[v] and np.isnan(death[v]):
            death[v] = np.inf
    return death, isb

def _nd_shape(a):
    return (1,) + a.shape if a.ndim == 2 else a.shape

def certify_nd(ell_arr, u_arr, tau, conn, exact=True):
    """CERTIFY-H0 on an implicit 2-D or 3-D grid (same semantics as certify_graph on the corresponding CSR grid)."""
    Dd, H, W = _nd_shape(ell_arr)
    dz, dy, dx = _nd_offsets(ell_arr.ndim, conn)
    ell = np.ascontiguousarray(ell_arr.ravel(), np.float64); u = np.ascontiguousarray(u_arr.ravel(), np.float64)
    if np.any(ell > u):
        raise ValueError("A1 violated: need ell <= u")
    order = order_of(ell)
    T, nn, npar, nmu, nlam, nminl, ntrap, nmin, nmct, nvert, Dv, ops = _certify_nd_sweep(
        Dd, H, W, dz, dy, dx, ell, u, float(tau), order, exact)
    fs, _ = _certify_topdown(nn, npar, nlam, ntrap, Dv, ell, float(tau))
    return Certificate(int(T), float(tau), npar[:nn], nmu[:nn], nlam[:nn], nminl[:nn], ntrap[:nn], nmin[:nn],
                       nmct[:nn], nvert[:nn], Dv, fs, int(ops))

def persistence_nd(f_arr, conn):
    Dd, H, W = _nd_shape(f_arr); dz, dy, dx = _nd_offsets(f_arr.ndim, conn)
    f = np.ascontiguousarray(f_arr.ravel(), np.float64)
    death, isb = _persistence_nd(Dd, H, W, dz, dy, dx, f, order_of(f))
    return f, death, isb

def n_tau_nd(f_arr, tau, conn):
    f, death, isb = persistence_nd(f_arr, conn)
    return int(np.sum(isb & ((death - f) > tau)))

def graph_grid3(D, H, W, conn):
    """Explicit CSR 1-skeleton of a 3-D grid (for cross-checking the implicit N-D kernels)."""
    dz, dy, dx = _nd_offsets(3, conn)
    idx = np.arange(D * H * W, dtype=np.int64).reshape(D, H, W); E = []
    for a, b, c in zip(dz.tolist(), dy.tolist(), dx.tolist()):
        src = idx[max(0, -a):D - max(0, a), max(0, -b):H - max(0, b), max(0, -c):W - max(0, c)]
        E.append(np.stack([src.ravel(), (src + a * H * W + b * W + c).ravel()], 1))
    return graph_from_edges(D * H * W, np.concatenate(E), f"grid3_{D}x{H}x{W}c{conn}")

# ---- sweep-order relabelling (post hoc engineering) -----------------------------------------------------------
@njit
def _relabel_grid_csr(H, W, conn, order):
    """CSR of the implicit HxW grid with vertex i := order[i], so the sweep visits 0..n-1 sequentially."""
    n = H * W
    dr, dc = _grid_offsets(conn)
    newid = np.empty(n, np.int64)
    for i in range(n):
        newid[order[i]] = i
    ip = np.zeros(n + 1, np.int64)
    for i in range(n):
        v = order[i]; r0 = v // W; c0 = v - r0 * W; d = 0
        for t in range(dr.shape[0]):
            rr = r0 + dr[t]; cc = c0 + dc[t]
            if rr >= 0 and rr < H and cc >= 0 and cc < W:
                d += 1
        ip[i + 1] = ip[i] + d
    ix = np.empty(ip[n], np.int32)
    for i in range(n):
        v = order[i]; r0 = v // W; c0 = v - r0 * W; k = ip[i]
        for t in range(dr.shape[0]):
            rr = r0 + dr[t]; cc = c0 + dc[t]
            if rr >= 0 and rr < H and cc >= 0 and cc < W:
                ix[k] = newid[rr * W + cc]; k += 1
    return ip, ix

def certify_relabeled_grid(ell2d, u2d, tau, conn=4, exact=True, return_timing=False):
    H, W = ell2d.shape; n = H * W
    ell = np.ascontiguousarray(ell2d.ravel(), np.float64); u = np.ascontiguousarray(u2d.ravel(), np.float64)
    t0 = time.perf_counter(); order = order_of(ell); t1 = time.perf_counter()
    ip, ix = _relabel_grid_csr(H, W, conn, order); ls = ell[order]; us = u[order]
    ident = np.arange(n, dtype=np.int64); t2 = time.perf_counter()
    T, nn, npar, nmu, nlam, nminl, ntrap, nmin, nmct, nvert, Dv, ops = _certify_sweep(
        ip, ix, ls, us, float(tau), ident, False, exact); t3 = time.perf_counter()
    fs_seq, _ = _certify_topdown(nn, npar, nlam, ntrap, Dv, ls, float(tau))
    fs = np.empty(n); fs[order] = fs_seq
    Dv_orig = np.empty(n, np.int64); Dv_orig[order] = Dv; t4 = time.perf_counter()
    cert = Certificate(int(T), float(tau), npar[:nn], nmu[:nn], nlam[:nn], nminl[:nn], ntrap[:nn], nmin[:nn],
                       nmct[:nn], order[nvert[:nn]], Dv_orig, fs, int(ops))
    if return_timing:
        return cert, dict(t_sort=t1 - t0, t_relabel=t2 - t1, t_sweep=t3 - t2, t_topdown=t4 - t3, t_total=t4 - t0)
    return cert

# ---- plateau-aware minima: the tau = 0 objective of Gray-Kammer-Loffler-Silveira ---------------------------------
@njit
def _minimal_plateaus(indptr, indices, f):
    n = f.shape[0]
    par = np.arange(n)
    for v in range(n):
        for q in range(indptr[v], indptr[v + 1]):
            w = indices[q]
            if w > v and f[w] == f[v]:
                a = _find(par, v); b = _find(par, w)
                if a != b:
                    par[a] = b
    lower = np.zeros(n, np.bool_)
    for v in range(n):
        r = _find(par, v)
        for q in range(indptr[v], indptr[v + 1]):
            if f[indices[q]] < f[v]:
                lower[r] = True
    cnt = 0
    for v in range(n):
        if _find(par, v) == v and not lower[v]:
            cnt += 1
    return cnt

def minimal_plateaus(graph, f):
    return int(_minimal_plateaus(graph.indptr, graph.indices, np.asarray(f, np.float64)))

@njit
def _brute_min_plateaus(indptr, indices, cand, ncand):
    n = ncand.shape[0]; total = 1
    for v in range(n):
        total *= ncand[v]
    f = np.empty(n); best = 1 << 30
    for t in range(total):
        rem = t
        for v in range(n):
            f[v] = cand[v, rem % ncand[v]]; rem //= ncand[v]
        c = _minimal_plateaus(indptr, indices, f)
        if c < best:
            best = c
    return best

# ---- rank certificate: exact characterisation -----------------------------------------------------------------------------------
def rank_via_forest(cert, tau):
    """max_{t-s>tau} rk(s,t) computed from the sweep forest as max_s #minimal elements of
    U_s = {trap nodes x : mu(x) <= s < lambda(x) - tau}. U_s is upward closed (windows grow towards the root), so
    its largest antichain is its set of minimal elements."""
    trap = np.nonzero(cert.trap)[0]
    if len(trap) == 0:
        return 0
    par = cert.parent; his = cert.lam - tau
    ent = trap[np.lexsort((-trap, cert.mu[trap]))]          # by mu; parents (larger ids) first at ties
    lv = trap[np.lexsort((trap, his[trap]))]                  # by lambda - tau; children first at ties
    inset = np.zeros(len(par), bool); child = np.zeros(len(par), np.int64)
    cur = best = 0; li = ei = 0
    for s in np.unique(cert.mu[trap]):
        while li < len(lv) and his[lv[li]] <= s:
            x = lv[li]; li += 1
            if not inset[x]:
                continue
            inset[x] = False; cur -= 1
            p = par[x]
            if p >= 0 and inset[p]:
                child[p] -= 1
                if child[p] == 0:
                    cur += 1
        while ei < len(ent) and cert.mu[ent[ei]] <= s:
            x = ent[ei]; ei += 1
            if his[x] <= s:
                continue
            inset[x] = True; cur += 1
            p = par[x]
            if p >= 0 and inset[p]:
                if child[p] == 0:
                    cur -= 1
                child[p] += 1
        best = max(best, cur)
    return best

def window_clique(cert, tau):
    """max_s #{minimal traps M : s in [mu(M), lambda(M) - tau)}: a lower bound on the maximum rank (window lemma)."""
    ids = cert.minimal_ids
    ev = sorted([(float(x), 1) for x in cert.mu[ids]] + [(float(x), -1) for x in cert.lam[ids] - tau])
    cur = best = 0
    for _, d in ev:
        cur += d; best = max(best, cur)
    return best

In [10]:
@njit
def _elder_children(nn, npar, nmu):
    elder = np.full(nn, -1, np.int64)
    for k in range(nn):
        p = npar[k]
        if p >= 0:
            e = elder[p]
            if e == -1 or nmu[k] < nmu[e]:
                elder[p] = k
    return elder

@njit
def _image_bars0(nn, npar, nmu, nlam):
    elder = _elder_children(nn, npar, nmu)
    births = np.empty(nn); deaths = np.empty(nn); nodes = np.empty(nn, np.int64); m = 0
    for k in range(nn):
        p = npar[k]
        if p < 0:
            births[m] = nmu[k]; deaths[m] = np.inf; nodes[m] = k; m += 1
        elif elder[p] != k and nmu[k] < nlam[k]:
            births[m] = nmu[k]; deaths[m] = nlam[k]; nodes[m] = k; m += 1
    return births[:m], deaths[:m], nodes[:m]

@njit
def _universal_phi(nn, npar, nmu, nlam, cval):
    phi = np.empty(nn)
    for k in range(nn - 1, -1, -1):          # parents have larger ids than children
        if nmu[k] < nlam[k]:                 # visible (roots have lambda = inf)
            phi[k] = cval[k] if cval[k] > nmu[k] else nmu[k]
        else:
            phi[k] = phi[npar[k]]            # invisible nodes are never roots
    return phi

def image_barcode0(cert):
    """Degree-0 barcode of im(H0(Sub_u) -> H0(Sub_ell)): (births, deaths, forest nodes)."""
    return _image_bars0(len(cert.parent), cert.parent, cert.mu, cert.lam)

def image_bars(cert):
    b, d, _ = image_barcode0(cert)
    return sorted(zip(b.tolist(), d.tolist()))

def universal_realizer(cert, ell):
    """The tau-free f_U in [ell, u] whose degree-0 barcode equals the image barcode."""
    cval = np.asarray(ell, np.float64)[cert.vertex]
    return _universal_phi(len(cert.parent), cert.parent, cert.mu, cert.lam, cval)[cert.Dv]

def barcode0(graph, f):
    """Degree-0 barcode of the lower-star filtration of f (zero-length bars dropped), sorted."""
    f = np.ascontiguousarray(f, np.float64)
    death, isb = persistence0(graph.indptr, graph.indices, f, order_of(f))
    b = f[isb]; d = death[isb]; keep = d > b
    return sorted(zip(b[keep].tolist(), d[keep].tolist()))

def subbarcode_match(A, B):
    """Size of a maximum sub-barcode matching of bars A into bars B ([a,b) into [c,d) needs c <= a and b <= d).
    Greedy: A by increasing birth; among B bars born no later, take the smallest death >= b (best fit; exchange argument)."""
    A = sorted(A); B = sorted(B); avail = []; j = 0; matched = 0
    for a, b in A:
        while j < len(B) and B[j][0] <= a:
            bisect.insort(avail, B[j][1]); j += 1
        k = bisect.bisect_left(avail, b)
        if k < len(avail):
            avail.pop(k); matched += 1
    return matched

def is_subbarcode(A, B):
    return subbarcode_match(A, B) == len(A)

def count_bars(bars, tau):
    return sum(1 for b, d in bars if d - b > tau)

## 5 · Unit tests

There are three kinds of test, run before any experiment:
- known-answer tests on hand-solved instances;
- agreement between independent implementations, for example the bound against the mandatory minima of Günther, Salmon and Tierny on 1,000 random graphs (test T08);
- properties that follow from the definitions.

Tests that depend on exact arithmetic use integer data, and the compiled kernels are checked against exact rational arithmetic.

In [11]:
UNIT = []
def unit(name, passed, detail=""):
    UNIT.append(dict(test=name, result="PASS" if passed else "FAIL", detail=str(detail)))
    print(f"{'PASS' if passed else 'FAIL'}  {name}  {detail}")

def rand_int_box(rng, n, vmax=40, wmax=12):
    ell = rng.integers(0, vmax, n).astype(np.float64)
    return ell, ell + rng.integers(0, wmax, n).astype(np.float64)

def small_random_graph(rng, nmax=25):
    n = int(rng.integers(1, nmax))
    return graph_er(n, float(rng.uniform(0.05, 0.5)), rng)

_r = rng_for("unit")
# T01 persistence known answer: path 0-2-1-3 has Dgm = {(0,inf),(1,2)}
g = graph_path(4); f = np.array([0., 2., 1., 3.])
_, b, d = diagram0(g, f)
unit("T01 persistence known answer", sorted(zip(b.tolist(), d.tolist())) == [(0.0, math.inf), (1.0, 2.0)] and n_tau(g, f, 1.0) == 1)

# T02 staircase with three basins, known answer (frozen values)
g, f = staircase_path(2)
c = certify_graph(g, f, f, 1.0)
unit("T02 Prop.3(b) example: T_1=3, N_1=3, max rank=2",
     (c.T, n_tau(g, f, 1.0), rank_certificate(g, f, f, 1.0)) == (3, 3, 2), (c.T, n_tau(g, f, 1.0), rank_certificate(g, f, f, 1.0)))

# T03 staircase family: gap K-1
ok = True
for K in range(1, 9):
    g, f = staircase_path(K)
    ok &= certify_graph(g, f, f, 1.0).T == K + 1 and rank_certificate(g, f, f, 1.0) == min(2, K + 1)
unit("T03 staircase: T=K+1, max rank=2 (K=1..8)", ok)

# T04 comb: certificate exactly 1
ok = all(certify_graph(*comb_path(K), 0.5).T == 1 for K in range(2, 21))
unit("T04 comb: T_0.5 = 1 (K=2..20)", ok)

# T05 nested forest: T = D+1
ok = True
for D in range(1, 15):
    g, f = nested_path(D)
    ok &= certify_graph(g, f, f, 1.0).T == D + 1
unit("T05 nested forest: T_1 = D+1 (D=1..14)", ok)

# T06 degenerate box ell=u=f: T_tau = N_tau(f)
bad = 0
for _ in range(500):
    g = small_random_graph(_r); f = _r.integers(0, 30, g.n).astype(float); tau = float(_r.integers(0, 5))
    bad += certify_graph(g, f, f, tau).T != n_tau(g, f, tau)
unit("T06 degenerate box: T = N_tau(f)", bad == 0, f"mismatches={bad}/500")

# T07 B2 == B3 identity: T_tau(fhat-delta, fhat+delta) = N_{tau+2delta}(fhat)  
bad = 0
for _ in range(500):
    g = small_random_graph(_r); fh = _r.integers(0, 60, g.n).astype(float)
    delta = float(_r.integers(0, 6)); tau = float(_r.integers(0, 6))
    bad += certify_graph(g, fh - delta, fh + delta, tau).T != uniform_band_certificate(g, fh, delta, tau)
unit("T07 symmetric-box identity (B2 = B3)", bad == 0, f"mismatches={bad}/500")

# T08 GST14 = T_0
bad = 0
for _ in range(1000):
    g = small_random_graph(_r); l, u_ = rand_int_box(_r, g.n)
    bad += gst14_count(g, l, u_) != certify_graph(g, l, u_, 0.0).T
unit("T08 GST14 mandatory minima = T_0", bad == 0, f"mismatches={bad}/1000")

# T09 tau-curve: monotone tau_x on leaf-to-root paths and interval formula = direct recomputation
bad_curve = bad_mono = 0
for _ in range(500):
    g = small_random_graph(_r); l, u_ = rand_int_box(_r, g.n)
    c = certify_graph(g, l, u_, 0.0)
    tx = c.lam - c.mu
    has_p = c.parent >= 0
    bad_mono += int(np.any(tx[c.parent[has_p]] < tx[has_p]))
    taus = np.arange(0, 20, 0.5)
    direct = np.array([certify_graph(g, l, u_, t).T for t in taus])
    bad_curve += int(np.any(tau_curve(c, taus) != direct))
unit("T09 Cor.2 monotone tau_x", bad_mono == 0, f"violations={bad_mono}/500")
unit("T09 Cor.2 interval formula = recomputation", bad_curve == 0, f"mismatches={bad_curve}/500")

# T10 CSR kernel = implicit grid kernel (conn 4, 6, 8)
bad = 0
for conn in (4, 6, 8):
    for _ in range(60):
        H, W = int(_r.integers(1, 15)), int(_r.integers(1, 15))
        l, u_ = rand_int_box(_r, H * W); tau = float(_r.integers(0, 6))
        a = certify_graph(graph_grid(H, W, conn), l, u_, tau); b_ = certify_grid(l.reshape(H, W), u_.reshape(H, W), tau, conn)
        bad += (a.T != b_.T) or (not np.array_equal(a.fstar, b_.fstar))
unit("T10 CSR kernel = grid kernel (T and f*)", bad == 0, f"mismatches={bad}/180")
bad = 0
for conn in (4, 6, 8):
    for _ in range(60):
        H, W = int(_r.integers(1, 15)), int(_r.integers(1, 15))
        f = _r.integers(0, 20, H * W).astype(float)
        d1, i1 = persistence0(graph_grid(H, W, conn).indptr, graph_grid(H, W, conn).indices, f, order_of(f))
        _, d2, i2 = persistence_grid(f.reshape(H, W), conn)
        bad += (not np.array_equal(i1, i2)) or (not np.array_equal(np.nan_to_num(d1, nan=-7), np.nan_to_num(d2, nan=-7)))
unit("T10b CSR persistence = grid persistence", bad == 0, f"mismatches={bad}/180")

# T11 numba vs exact Fraction reference
bad = 0
for _ in range(200):
    g = small_random_graph(_r, 15); l, u_ = rand_int_box(_r, g.n); tau = int(_r.integers(0, 6))
    adj = _adjacency(g)
    Tr, fr = certify_reference(adj, [Fraction(int(x), 10) for x in l], [Fraction(int(x), 10) for x in u_], Fraction(tau, 10))
    c = certify_graph(g, l, u_, float(tau))
    bad += (Tr != c.T) or any(Fraction(int(round(a)), 10) != b_ for a, b_ in zip(c.fstar, fr))
unit("T11 numba = exact rational reference (T and f*)", bad == 0, f"mismatches={bad}/200")

# T12 invariances: translation, positive integer scaling, vertex relabelling
bad = 0
for _ in range(300):
    g = small_random_graph(_r); l, u_ = rand_int_box(_r, g.n); tau = float(_r.integers(0, 6))
    T0 = certify_graph(g, l, u_, tau).T
    s = int(_r.integers(2, 5)); cshift = float(_r.integers(-50, 50))
    perm = _r.permutation(g.n); inv = np.argsort(perm)
    src = np.repeat(np.arange(g.n), np.diff(g.indptr)); e = np.stack([src, g.indices], 1)
    gp = graph_from_edges(g.n, inv[e]) if len(e) else g
    lp = np.empty(g.n); up = np.empty(g.n); lp[inv] = l; up[inv] = u_
    bad += not (certify_graph(g, l + cshift, u_ + cshift, tau).T == T0 and
                certify_graph(g, s * l, s * u_, s * tau).T == T0 and certify_graph(gp, lp, up, tau).T == T0)
unit("T12 invariance (shift, scale, relabel)", bad == 0, f"violations={bad}/300")

# T13 monotonicity: shrinking the box cannot decrease T; T non-increasing in tau
bad = 0
for _ in range(300):
    g = small_random_graph(_r); l, u_ = rand_int_box(_r, g.n); tau = float(_r.integers(0, 6))
    l2 = np.minimum(l + _r.integers(0, 3, g.n), u_); u2 = np.maximum(u_ - _r.integers(0, 3, g.n), l2)
    bad += not (certify_graph(g, l2, u2, tau).T >= certify_graph(g, l, u_, tau).T >= certify_graph(g, l, u_, tau + 1).T)
unit("T13 monotonicity (box inclusion, tau)", bad == 0, f"violations={bad}/300")

# T14 rank certificate: sweep = brute force over (s,t); never above T
def rank_bruteforce(g, l, u_, tau):
    levels = sorted(set(l.tolist())); best = 0; adj = _adjacency(g)
    for k, L in enumerate(levels):
        Ln = levels[k + 1] if k + 1 < len(levels) else math.inf
        S = [v for v in range(g.n) if l[v] <= L]; Sset = set(S); seen = set(); cnt = 0
        for s0 in S:
            if s0 in seen:
                continue
            comp = [s0]; seen.add(s0); st = [s0]
            while st:
                x = st.pop()
                for y in adj[x]:
                    if y in Sset and y not in seen:
                        seen.add(y); st.append(y); comp.append(y)
            cnt += min(u_[comp]) < Ln - tau
        best = max(best, cnt)
    return best
bad_eq = bad_ub = 0
for _ in range(400):
    g = small_random_graph(_r, 18); l, u_ = rand_int_box(_r, g.n); tau = float(_r.integers(0, 6))
    rk = rank_certificate(g, l, u_, tau)
    bad_eq += rk != rank_bruteforce(g, l, u_, tau); bad_ub += rk > certify_graph(g, l, u_, tau).T
unit("T14 rank certificate sweep = brute force", bad_eq == 0, f"mismatches={bad_eq}/400")
unit("T14 rank certificate <= T (Prop.3 direction)", bad_ub == 0, f"violations={bad_ub}/400")

# T15 priority-flood: filled surface has no interior pits; #depressions(depth>tau) <= #tau-persistent pits
bad = 0
for _ in range(100):
    H, W = int(_r.integers(3, 25)), int(_r.integers(3, 25))
    z = grf_batch(_r, 1, H, W, 2.0)[0]; tau = float(_r.choice([0.0, 0.2, 0.5]))
    go = add_outlet(graph_grid(H, W, 8), grid_boundary(H, W))
    filled = _priority_flood(z.copy(), 8)
    pits_filled = n_tau(go, np.append(filled.ravel(), -1e9), 0.0) - 1
    pits = n_tau(go, np.append(z.ravel(), -1e9), tau) - 1
    bad += (pits_filled != 0) or (priority_flood_depressions(z, tau, 8) > pits)
unit("T15 priority-flood consistency", bad == 0, f"violations={bad}/100")

# T16 A1 check
try:
    certify_graph(graph_path(2), np.array([1.0, 0.0]), np.array([0.0, 1.0]), 0.0); ok = False
except ValueError:
    ok = True
unit("T16 A1 violation (ell>u) is rejected", ok)

# T17 trap-to-birth injection and quadrant matching are complete for f in B
bad = 0
for _ in range(300):
    g = small_random_graph(_r); l, u_ = rand_int_box(_r, g.n); tau = float(_r.integers(0, 6))
    c = certify_graph(g, l, u_, tau)
    for _k in range(3):
        f = l + _r.integers(0, 4, g.n) * (u_ - l) / 3.0
        bad += (lemma1_injection(c, g, f) != c.T) or (quadrant_matching(c, g, f, tau) != c.T)
unit("T17 forced quadrants populated for f in B (Lemma 1 map and matching)", bad == 0, f"failures={bad}/900")

# T18 ablation sanity: the post-filter variant is a valid lower bound and is exact on the staircase family
bad = 0
for _ in range(300):
    g = small_random_graph(_r); l, u_ = rand_int_box(_r, g.n); tau = float(_r.integers(0, 6))
    bad += postfilter_count(g, l, u_, tau) > certify_graph(g, l, u_, tau).T
for K in range(1, 9):
    g, f = staircase_path(K)
    bad += postfilter_count(g, f, f, 1.0) != K + 1
unit("T18 post-filter ablation <= T, exact on staircases", bad == 0, f"violations={bad}/308")

# T19 exact predicate: _lt_sum(a, b, c) equals rational arithmetic on the stored doubles, including near-ties
bad = naive_bad = 0
for _ in range(20000):
    a = float(_r.standard_normal() * 10.0 ** _r.integers(-3, 4)); b = float(_r.standard_normal() * 10.0 ** _r.integers(-3, 4))
    s = a + b; rr = _r.random()
    c = float(np.nextafter(s, np.inf)) if rr < 0.3 else (float(np.nextafter(s, -np.inf)) if rr < 0.6 else s)
    if _r.random() < 0.1:
        c = float(c + _r.standard_normal())
    truth = Fraction(a) + Fraction(b) < Fraction(c)
    bad += bool(_lt_sum(a, b, c)) != truth; naive_bad += (a + b < c) != truth
unit("T19 exact predicate a+b<c = rational arithmetic (near-ties)", bad == 0, f"mismatches={bad}/20000 (rounded test: {naive_bad})")

# T20 N_0(f) = number of minimal plateaus (the tau = 0 objective of Gray-Kammer-Loffler-Silveira)
bad = 0
for _ in range(1000):
    g = small_random_graph(_r); f = _r.integers(0, 4, g.n).astype(float)
    bad += n_tau(g, f, 0.0) != minimal_plateaus(g, f)
unit("T20 N_0(f) = #minimal plateaus", bad == 0, f"mismatches={bad}/1000")

# T21 rank-certificate characterisation: sweep rank certificate = forest formula max_s #min(U_s)
bad = 0
for _ in range(1000):
    g = small_random_graph(_r, 30); l, u_ = rand_int_box(_r, g.n); tau = float(_r.integers(0, 6))
    bad += rank_via_forest(certify_graph(g, l, u_, tau), tau) != rank_certificate(g, l, u_, tau)
unit("T21 max rank = max_s #minimal(U_s) (forest characterisation)", bad == 0, f"mismatches={bad}/1000")

# T22 window lemma: window clique <= max rank <= T, and a strict gap forces windows that do not all intersect
bad = 0
for _ in range(1000):
    g = small_random_graph(_r, 30); l, u_ = rand_int_box(_r, g.n); tau = float(_r.integers(0, 6))
    c = certify_graph(g, l, u_, tau); rk = rank_certificate(g, l, u_, tau); wc = window_clique(c, tau)
    bad += (not (wc <= rk <= c.T)) or (c.T > rk and wc >= c.T)
unit("T22 window lemma", bad == 0, f"violations={bad}/1000")

# T23 N-D implicit kernels = explicit CSR (2-D conn 4/6/8, 3-D conn 6/26; certificate and persistence)
bad = 0
for conn in (4, 6, 8):
    for _ in range(30):
        H, W = int(_r.integers(1, 12)), int(_r.integers(1, 12)); l, u_ = rand_int_box(_r, H * W); tau = float(_r.integers(0, 6))
        a = certify_graph(graph_grid(H, W, conn), l, u_, tau); b_ = certify_nd(l.reshape(H, W), u_.reshape(H, W), tau, conn)
        bad += (a.T != b_.T) or (not np.array_equal(a.fstar, b_.fstar))
for conn in (6, 26):
    for _ in range(30):
        D, H, W = (int(_r.integers(1, 6)) for _ in range(3)); l, u_ = rand_int_box(_r, D * H * W); tau = float(_r.integers(0, 6))
        g3 = graph_grid3(D, H, W, conn)
        a = certify_graph(g3, l, u_, tau); b_ = certify_nd(l.reshape(D, H, W), u_.reshape(D, H, W), tau, conn)
        bad += (a.T != b_.T) or (not np.array_equal(a.fstar, b_.fstar))
        f = _r.integers(0, 20, D * H * W).astype(float)
        d1, i1 = persistence0(g3.indptr, g3.indices, f, order_of(f)); _, d2, i2 = persistence_nd(f.reshape(D, H, W), conn)
        bad += (not np.array_equal(i1, i2)) or (not np.array_equal(np.nan_to_num(d1, nan=-7), np.nan_to_num(d2, nan=-7)))
unit("T23 N-D grid kernels = CSR (2-D and 3-D)", bad == 0, f"mismatches={bad}")

# T24 sweep-order relabelled kernel = original grid kernel (T and f*)
bad = 0
for conn in (4, 8):
    for _ in range(40):
        H, W = int(_r.integers(1, 15)), int(_r.integers(1, 15)); l, u_ = rand_int_box(_r, H * W); tau = float(_r.integers(0, 6))
        a = certify_grid(l.reshape(H, W), u_.reshape(H, W), tau, conn)
        b_ = certify_relabeled_grid(l.reshape(H, W), u_.reshape(H, W), tau, conn)
        bad += (a.T != b_.T) or (not np.array_equal(a.fstar, b_.fstar))
unit("T24 relabelled sweep = original kernel", bad == 0, f"mismatches={bad}/80")

# T25 the certificate is a barcode: the number of image bars longer than tau equals T_tau
bad = 0
for _ in range(500):
    g = small_random_graph(_r, 30); l, u_ = rand_int_box(_r, g.n)
    I = image_bars(certify_graph(g, l, u_, 0.0))
    for tau in (0.0, 1.0, 2.5, 5.0, 9.0):
        bad += count_bars(I, tau) != certify_graph(g, l, u_, tau).T
unit("T25 #image bars longer than tau = T_tau", bad == 0, f"mismatches={bad}")

# T26 universal realiser: f_U lies in the box and Bar0(f_U) equals the image barcode bar for bar (ties and floats)
bad = 0
for k in range(600):
    g = small_random_graph(_r, 30)
    if k % 2:
        l, u_ = rand_int_box(_r, g.n, 12, 5)
    else:
        l = _r.normal(size=g.n); u_ = l + _r.exponential(0.5, g.n)
    c0 = certify_graph(g, l, u_, 0.0); fU = universal_realizer(c0, l)
    bad += (not (np.all(fU >= l) and np.all(fU <= u_))) or barcode0(g, fU) != image_bars(c0)
unit("T26 Bar0(f_U) = image barcode and f_U in the box", bad == 0, f"failures={bad}/600")

# T27 one tau-free witness is optimal at every scale
bad = 0
for _ in range(400):
    g = small_random_graph(_r, 30); l, u_ = rand_int_box(_r, g.n)
    fU = universal_realizer(certify_graph(g, l, u_, 0.0), l)
    for tau in range(0, 13):
        bad += n_tau(g, fU, float(tau)) != certify_graph(g, l, u_, float(tau)).T
unit("T27 N_tau(f_U) = T_tau for every tau", bad == 0, f"mismatches={bad}")

# T28 sub-barcode validity (Chubet-Gardner-Sheehy, reproduced): the image barcode is a sub-barcode of Bar0(f) for sampled f in the box
bad = 0
for _ in range(600):
    g = small_random_graph(_r, 30); l, u_ = rand_int_box(_r, g.n, 20, 8); I = image_bars(certify_graph(g, l, u_, 0.0))
    for _ in range(3):
        bad += not is_subbarcode(I, barcode0(g, np.round(l + _r.random(g.n) * (u_ - l))))
unit("T28 image barcode is a sub-barcode of every sampled f (reproduced)", bad == 0, f"violations={bad}/1800")

# T29 the greedy sub-barcode matching is maximum (Kuhn's augmenting paths as reference)
def _kuhn(A, B):
    adj = [[j for j, (c, d) in enumerate(B) if c <= a and b <= d] for a, b in A]; match = [-1] * len(B)
    def aug(i, seen):
        for j in adj[i]:
            if not seen[j]:
                seen[j] = True
                if match[j] == -1 or aug(match[j], seen):
                    match[j] = i; return True
        return False
    return sum(aug(i, [False] * len(B)) for i in range(len(A)))
def _rand_bars(k):
    return [(float(x), float(x + y) if y < 9 else math.inf) for x, y in zip(_r.integers(0, 8, k), _r.integers(1, 10, k))]
bad = 0
for _ in range(2000):
    A, B = _rand_bars(int(_r.integers(0, 7))), _rand_bars(int(_r.integers(0, 7)))
    bad += subbarcode_match(A, B) != _kuhn(A, B)
unit("T29 greedy sub-barcode matching is maximum", bad == 0, f"mismatches={bad}/2000")

# T30 monotonicity (reproduced): widening the box yields a sub-barcode of the narrower box's image barcode
bad = 0
for _ in range(500):
    g = small_random_graph(_r, 30); l, u_ = rand_int_box(_r, g.n, 20, 6)
    a1, a2 = _r.integers(0, 4, g.n), _r.integers(0, 4, g.n)
    bad += not is_subbarcode(image_bars(certify_graph(g, l - a1, u_ + a2, 0.0)), image_bars(certify_graph(g, l, u_, 0.0)))
unit("T30 wider box gives a sub-barcode (monotonicity)", bad == 0, f"violations={bad}/500")

unit_df = pd.DataFrame(UNIT)
unit_df.to_csv(RESULTS / "tables" / "unit_tests.csv", index=False)
ALL_UNIT_PASS = bool((unit_df.result == "PASS").all())
set_status("unit_tests", "EXECUTED", f"{(unit_df.result=='PASS').sum()}/{len(unit_df)} PASS")

PASS  T01 persistence known answer  
PASS  T02 Prop.3(b) example: T_1=3, N_1=3, max rank=2  (3, 3, 2)
PASS  T03 staircase: T=K+1, max rank=2 (K=1..8)  
PASS  T04 comb: T_0.5 = 1 (K=2..20)  
PASS  T05 nested forest: T_1 = D+1 (D=1..14)  
PASS  T06 degenerate box: T = N_tau(f)  mismatches=0/500
PASS  T07 symmetric-box identity (B2 = B3)  mismatches=0/500
PASS  T08 GST14 mandatory minima = T_0  mismatches=0/1000
PASS  T09 Cor.2 monotone tau_x  violations=0/500
PASS  T09 Cor.2 interval formula = recomputation  mismatches=0/500
PASS  T10 CSR kernel = grid kernel (T and f*)  mismatches=0/180
PASS  T10b CSR persistence = grid persistence  mismatches=0/180
PASS  T11 numba = exact rational reference (T and f*)  mismatches=0/200
PASS  T12 invariance (shift, scale, relabel)  violations=0/300
PASS  T13 monotonicity (box inclusion, tau)  violations=0/300
PASS  T14 rank certificate sweep = brute force  mismatches=0/400
PASS  T14 rank certificate <= T (Prop.3 direction)  violations=0/400
PASS  T15 pr

## 6 · Degree 0: Theorem 5 and Lemma 2

- **τ = 0.** The number of image bars equals the minimum number of local minima over the box, the objective of Gray, Kammer, Löffler and Silveira. This is checked by exhaustive search over an extended grid of values on 500 small instances.
- **Attainment.** On 100,000 random boxes on paths, grids and random graphs, the check is that $f_U$ lies in the box and $\mathrm{Bar}_0(f_U)=\mathrm{Bar}_0(u\ge\ell)$ bar for bar. Then $N_\tau(f_U)=T_\tau$ for every τ, and every sampled function in the box has the image barcode as a sub-barcode (Theorem 1).
- **Regions (Lemma 2).** On smaller instances, exhaustive minimisation over a value grid that contains $f_U$ is compared with the image counts on 3,000 persistence regions and 4,000 quadrants.

In [12]:
def candidate_array(l, u_, tau, extended):
    n = len(l); vals = set()
    for w in range(n):
        if extended:
            for k in range(-2, 3):
                vals.update([l[w] + k * tau, u_[w] + k * tau])
        else:
            vals.update([l[w], l[w] - tau])
    vals = np.array(sorted(vals))
    if extended:                                   # also probe the open cells between candidate values
        vals = np.unique(np.concatenate([vals, (vals[:-1] + vals[1:]) / 2])) if len(vals) > 1 else vals
    cands = [vals[(vals >= l[v]) & (vals <= u_[v])] for v in range(n)]
    cands = [c if len(c) else np.array([l[v]]) for v, c in enumerate(cands)]
    width = max(len(c) for c in cands)
    arr = np.zeros((n, width)); cnt = np.array([len(c) for c in cands], np.int64)
    for v, c in enumerate(cands):
        arr[v, :len(c)] = c
    return arr, cnt

In [13]:
ck_gk = Checkpoint("p1_gkls_objective")
if not ck_gk.done("all"):
    rng = rng_for("gkls"); rows = []
    for i in range(CONFIG["gkls_instances"]):
        n = int(rng.integers(1, 7)); g = graph_er(n, float(rng.uniform(0.2, 0.9)), rng)
        l = rng.integers(0, 12, n).astype(float); u_ = l + rng.integers(0, 6, n)
        cand, cnt = candidate_array(l, u_, 0.0, True)
        if np.prod(cnt.astype(float)) > 2e6:
            continue
        rows.append(dict(i=i, n=n, T0=certify_graph(g, l, u_, 0.0).T,
                         min_plateaus=int(_brute_min_plateaus(g.indptr, g.indices, cand, cnt)), assignments=int(np.prod(cnt))))
    ck_gk.add("all", rows)
gk = ck_gk.frame(); gk["match"] = gk.T0 == gk.min_plateaus
gk.to_csv(RESULTS / "raw" / "gkls_objective.csv", index=False)
GKLS = dict(instances=int(len(gk)), matches=int(gk.match.sum()))
GKLS_VERDICT = "SUPPORTED" if GKLS["instances"] and GKLS["matches"] == GKLS["instances"] else "NOT SUPPORTED"
print("tau = 0 imprecise-terrain objective:", GKLS, GKLS_VERDICT)
set_status("gkls_objective", "EXECUTED", GKLS_VERDICT)

tau = 0 imprecise-terrain objective: {'instances': 500, 'matches': 500} SUPPORTED
[STATUS] gkls_objective: EXECUTED — SUPPORTED


In [14]:
def _chain_top(parent, elder, x):
    while True:
        p = parent[x]
        if p < 0 or elder[p] != x:
            return x
        x = p

ck_opt = Checkpoint("v3_optimality")
if not ck_opt.done("all"):
    rng = rng_for("v3opt"); rows = []
    for i in range(CONFIG["opt_instances"]):
        kind = ("path", "grid", "graph", "graph_float")[i % 4]
        if kind == "path":
            n = int(rng.integers(2, 80)); g = graph_path(n); l, u_ = rand_int_box(rng, n)
        elif kind == "grid":
            H, W = int(rng.integers(2, 16)), int(rng.integers(2, 16))
            g = graph_grid(H, W, int(rng.choice([4, 6, 8]))); l, u_ = rand_int_box(rng, H * W)
        elif kind == "graph":
            g = small_random_graph(rng, 60); l, u_ = rand_int_box(rng, g.n)
        else:
            g = small_random_graph(rng, 60); l = rng.normal(size=g.n); u_ = l + rng.exponential(0.5, g.n)
        cert = certify_graph(g, l, u_, 0.0); I = image_bars(cert); fU = universal_realizer(cert, l)
        in_box = bool(np.all(fU >= l) and np.all(fU <= u_)); FU = barcode0(g, fU)
        lengths = sorted({0.0} | {d - b for b, d in I if math.isfinite(d)})
        all_tau = all(count_bars(FU, t) == count_bars(I, t) for t in lengths)
        viol = 0
        for _ in range(3):
            f = l + rng.random(g.n) * (u_ - l)
            viol += not is_subbarcode(I, barcode0(g, f if kind == "graph_float" else np.round(f)))
        elder = _elder_children(len(cert.parent), cert.parent, cert.mu); Iset = collections.Counter(I)
        n_elder = n_fin = corner_bad = 0
        for M in cert.minimal_ids:
            p = cert.parent[M]
            if p >= 0:
                n_fin += 1; n_elder += int(elder[p] == M)
            corner_bad += Iset[(float(cert.mu[M]), float(cert.lam[_chain_top(cert.parent, elder, M)]))] == 0
        rows.append(dict(i=i, kind=kind, n=g.n, bars=len(I), in_box=in_box, equal=FU == I, all_tau=all_tau, validity_violations=viol,
                         minimal=len(cert.minimal_ids), minimal_finite=n_fin, elder_minimal=n_elder, corner_bad=corner_bad))
    ck_opt.add("all", rows)
opt = ck_opt.frame(); opt.to_csv(RESULTS / "raw" / "v3_optimality.csv", index=False)

ck_optb = Checkpoint("v3_optimality_regions")
if not ck_optb.done("all"):
    rng = rng_for("v3optb"); rows = []
    def _cnt(bars, R):
        return sum(1 for b, d in bars if ((b <= R[1] and d >= R[2]) if R[0] == "quadrant" else (d - b > R[1])))
    for i in range(CONFIG["opt_brute_instances"]):
        n = int(rng.integers(2, 7)); g = graph_er(n, float(rng.uniform(0.2, 0.9)), rng)
        l = rng.integers(0, 10, n) * 2.0; u_ = l + rng.integers(0, 5, n) * 2.0
        cert = certify_graph(g, l, u_, 0.0); I = image_bars(cert); fU = universal_realizer(cert, l)
        regions = ([("quadrant", float(rng.integers(0, 20)), float(rng.integers(0, 30))) for _ in range(4)]
                   + [("persistence", t, None) for t in (0.0, 2.0, 4.0)])
        best = [math.inf] * len(regions)
        for combo in itertools.product(*[sorted({l[v], u_[v], 0.5 * (l[v] + u_[v]), fU[v]}) for v in range(n)]):
            bars = barcode0(g, np.array(combo))
            for k, R in enumerate(regions):
                best[k] = min(best[k], _cnt(bars, R))
        rows += [dict(i=i, n=n, region=R[0], beta_or_tau=R[1], delta=R[2], min_over_grid=best[k], image_count=_cnt(I, R))
                 for k, R in enumerate(regions)]
    ck_optb.add("all", rows)
optb = ck_optb.frame(); optb["equal"] = optb.min_over_grid == optb.image_count
optb.to_csv(RESULTS / "raw" / "v3_optimality_regions.csv", index=False)

OPT = dict(instances=int(len(opt)), in_box=int(opt.in_box.sum()), equal=int(opt.equal.sum()), all_tau=int(opt.all_tau.sum()),
           validity_violations=int(opt.validity_violations.sum()), corner_bad=int(opt.corner_bad.sum()),
           elder_share=float(opt.elder_minimal.sum() / max(opt.minimal_finite.sum(), 1)),
           region_checks=int(len(optb)), region_equal=int(optb.equal.sum()))
OPT_VERDICT = ("SUPPORTED" if (OPT["in_box"] == OPT["equal"] == OPT["all_tau"] == OPT["instances"] and OPT["corner_bad"] == 0
                               and OPT["region_equal"] == OPT["region_checks"]) else "NOT SUPPORTED")
CGS_VERDICT = "SUPPORTED" if OPT["validity_violations"] == 0 else "NOT SUPPORTED"
print(opt.groupby("kind")[["in_box", "equal", "all_tau"]].mean().round(4).to_string())
print("optimality:", OPT, "|", OPT_VERDICT, "| sub-barcode validity (reproduced):", CGS_VERDICT)
set_status("v3_optimality", "EXECUTED", OPT_VERDICT)

             in_box  equal  all_tau
kind                               
graph           1.0    1.0      1.0
graph_float     1.0    1.0      1.0
grid            1.0    1.0      1.0
path            1.0    1.0      1.0
optimality: {'instances': 100000, 'in_box': 100000, 'equal': 100000, 'all_tau': 100000, 'validity_violations': 0, 'corner_bad': 0, 'elder_share': 0.6291153234286947, 'region_checks': 7000, 'region_equal': 7000} | SUPPORTED | sub-barcode validity (reproduced): SUPPORTED
[STATUS] v3_optimality: EXECUTED — SUPPORTED


## 7 · Exact degree-1 machinery and its self-checks

- **Lower-star persistence** in degrees 0 and 1 on a 2-complex, by column reduction over $\mathbb Z_2$.
- **The image barcode** $\mathrm{Bar}_k(u\ge\ell)$ for $k\in\{0,1\}$, computed by inclusion–exclusion on the exact rank invariant $\operatorname{rank}(H_k(\mathrm{Sub}_u(s))\to H_k(\mathrm{Sub}_\ell(t)))$. The rank invariant is computed by Gaussian elimination over $\mathbb Z_2$.
- **Exact decisions.** $\mathrm{Bar}_1(f)$ depends only on the order type of $f$, the ordered partition of the vertices into level sets, and on the values at the bar endpoints. The procedure:
  - enumerate every order type the box admits, using a least-value greedy with a symbolic ε;
  - match each order type's index barcode against the target;
  - decide whether the forced endpoint values are feasible.

  Minimum counts over a box are difference-constraint systems, solved exactly by Bellman–Ford in rational arithmetic.
- **Self-checks.** In degree 0 the exact decision must always find attainment, and an exact minimum may never fall below the image count. A second, independent enumeration, over every surjective rank map with realisability decided by Bellman–Ford, must agree with the primary one on 60 instances.

**KC vertices.** A vertex is a KC (kill-and-create) vertex if some full subcomplex of its link, that is, an induced subgraph, is disconnected and contains a cycle. Only such a vertex can both kill and create degree-1 classes when it enters the filtration (Lemma 16).

In [15]:
def lowerstar_diagrams(nv, triangles, f, edges=None):
    """Z2 persistence of the lower-star filtration of a 2-complex. Simplices are ordered by (max vertex rank, dimension,
    descending vertex ranks), which refines the lower-star order with (f, id) tie-breaking. Returns {dim: [(b, d)]}."""
    tri = [tuple(int(x) for x in t) for t in triangles]
    edges = sorted({e for a, b, c in tri for e in ((a, b), (a, c), (b, c))} | {tuple(sorted((int(a), int(b)))) for a, b in (edges or ())})
    r = np.empty(nv, np.int64); r[np.lexsort((np.arange(nv), f))] = np.arange(nv)
    simp = [((int(r[v]),), 0, (v,)) for v in range(nv)]
    simp += [(tuple(sorted((int(r[a]), int(r[b])), reverse=True)), 1, (a, b)) for a, b in edges]
    simp += [(tuple(sorted((int(r[x]) for x in t), reverse=True)), 2, t) for t in tri]
    simp.sort(key=lambda s: (s[0][0], s[1], s[0]))
    index = {frozenset(s[2]): i for i, s in enumerate(simp)}
    val = [max(f[x] for x in s[2]) for s in simp]; dim = [s[1] for s in simp]
    cols = [set() if s[1] == 0 else {index[frozenset(s[2]) - {y}] for y in s[2]} for s in simp]
    pivot_of = {}; paired = set(); dg = {0: [], 1: [], 2: []}
    for j in range(len(cols)):
        c = cols[j]
        while c:
            low = max(c)
            if low in pivot_of:
                c ^= cols[pivot_of[low]]
            else:
                pivot_of[low] = j; paired.update((low, j)); dg[dim[low]].append((val[low], val[j])); break
        cols[j] = c
    for i in range(len(simp)):
        if i not in paired:
            dg[dim[i]].append((val[i], math.inf))
    return dg

def count_pers(pairs, tau):
    return sum(1 for b, d in pairs if d - b > tau)

In [16]:
class Small2Complex:
    def __init__(self, nv, triangles, name, edges=()):
        self.nv, self.name = nv, name
        self.tris = sorted({tuple(sorted(int(x) for x in t)) for t in triangles})
        self.edges = sorted({e for a, b, c in self.tris for e in ((a, b), (a, c), (b, c))} | {tuple(sorted((int(a), int(b)))) for a, b in edges})
        self.eidx = {e: i for i, e in enumerate(self.edges)}
        self.tri_bd = [(1 << self.eidx[(a, b)]) | (1 << self.eidx[(a, c)]) | (1 << self.eidx[(b, c)]) for a, b, c in self.tris]
        self.graph = graph_from_edges(nv, np.array(self.edges, np.int64).reshape(-1, 2), name)

def _z2_rank(vectors):
    piv = {}
    for v in vectors:
        while v:
            h = v.bit_length() - 1
            if h in piv:
                v ^= piv[h]
            else:
                piv[h] = v; break
    return len(piv)

def _cycle_basis(edge_ids, edges):
    """Z2 cycle-space basis (as edge bitmasks) of the graph formed by the given edges."""
    piv = {}; cycles = []
    for i in edge_ids:
        a, b = edges[i]; vm = (1 << a) | (1 << b); em = 1 << i
        while vm:
            h = vm.bit_length() - 1
            if h in piv:
                pv, pe = piv[h]; vm ^= pv; em ^= pe
            else:
                piv[h] = (vm, em); break
        if vm == 0:
            cycles.append(em)
    return cycles

def _components_meeting(cx, lv, uv, s, t):
    """rank of H0(Sub_u(s)) -> H0(Sub_ell(t)): components of {ell <= t} containing a vertex with u <= s."""
    par = list(range(cx.nv))
    def find(x):
        while par[x] != x:
            par[x] = par[par[x]]; x = par[x]
        return x
    for a, b in cx.edges:
        if max(lv[a], lv[b]) <= t:
            ra, rb = find(a), find(b)
            if ra != rb:
                par[ra] = rb
    return len({find(v) for v in range(cx.nv) if uv[v] <= s})

DEG1_CHECKS = dict(zoo_ok=True, rank_vs_reduction_mismatch=0, image0_vs_sweep_mismatch=0, negative_multiplicity=0)

def image_barcode_rank(cx, lv, uv, degree):
    """Barcode of im(H_k(Sub_u) -> H_k(Sub_ell)), k in {0, 1}, by inclusion-exclusion on the exact Z2 rank invariant."""
    lv = [float(x) for x in lv]; uv = [float(x) for x in uv]
    crit = sorted(set(lv) | set(uv)); K = len(crit)
    ue = [max(uv[a], uv[b]) for a, b in cx.edges]; lt = [max(lv[a], lv[b], lv[c]) for a, b, c in cx.tris]
    memo = {}
    def rk(i, j):                               # rank(I(c_i) -> I(c_j)); i = 0 is -inf, j = K + 1 is +inf
        if i == 0:
            return 0
        if (i, j) not in memo:
            s = crit[i - 1]; t = crit[j - 1] if j <= K else math.inf
            if degree == 0:
                memo[(i, j)] = _components_meeting(cx, lv, uv, s, t)
            else:
                Z = _cycle_basis([e for e, v in enumerate(ue) if v <= s], cx.edges)
                Bt = [cx.tri_bd[q] for q, v in enumerate(lt) if v <= t]
                memo[(i, j)] = _z2_rank(Z + Bt) - _z2_rank(Bt)
        return memo[(i, j)]
    bars = []
    for i in range(1, K + 1):
        for j in range(i + 1, K + 1):
            m = rk(i, j - 1) - rk(i - 1, j - 1) - rk(i, j) + rk(i - 1, j)
            DEG1_CHECKS["negative_multiplicity"] += int(m < 0)
            bars += [(crit[i - 1], crit[j - 1])] * max(m, 0)
        m = rk(i, K + 1) - rk(i - 1, K + 1)
        DEG1_CHECKS["negative_multiplicity"] += int(m < 0)
        bars += [(crit[i - 1], math.inf)] * max(m, 0)
    return sorted(bars)

def barcodes_01(cx, f):
    dg = lowerstar_diagrams(cx.nv, cx.tris, np.asarray(f, np.float64))
    return tuple(sorted((float(b), float(d)) for b, d in dg[0] if d > b)), tuple(sorted((float(b), float(d)) for b, d in dg[1] if d > b))

_RP2_6 = [(0, 1, 2), (0, 2, 3), (0, 3, 4), (0, 4, 5), (0, 5, 1), (1, 2, 4), (2, 3, 5), (3, 4, 1), (4, 5, 2), (5, 1, 3)]

def _grid_torus(m=3):
    v = lambda i, j: (i % m) * m + (j % m)
    return [t for i in range(m) for j in range(m) for t in ((v(i, j), v(i + 1, j), v(i + 1, j + 1)), (v(i, j), v(i, j + 1), v(i + 1, j + 1)))]

def complex_zoo():
    """Search complexes (non-neighborly, so finite H1 bars exist) plus the neighborly minimal triangulations used only in self-checks."""
    return {"sphere": Small2Complex(6, [(0, 1, 2), (0, 2, 3), (0, 3, 4), (0, 4, 1), (5, 1, 2), (5, 2, 3), (5, 3, 4), (5, 4, 1)], "octahedron"),
            "torus": Small2Complex(9, _grid_torus(3), "torus_grid3x3"),
            "rp2": Small2Complex(7, [t for t in _RP2_6 if t != (0, 1, 2)] + [(0, 1, 6), (0, 2, 6), (1, 2, 6)], "rp2_7"),
            "mobius": Small2Complex(9, [(0, 3, 4), (0, 1, 4), (3, 6, 7), (3, 4, 7), (1, 4, 5), (1, 2, 5), (4, 7, 8), (4, 5, 8), (2, 5, 3), (2, 6, 3),
                                        (5, 8, 0), (5, 3, 0)], "mobius9"),          # three-row twisted strip; vertices 3-5 are interior
            "torus7": Small2Complex(7, [(i % 7, (i + 1) % 7, (i + 3) % 7) for i in range(7)] + [(i % 7, (i + 2) % 7, (i + 3) % 7) for i in range(7)], "torus7"),
            "rp2_6": Small2Complex(6, _RP2_6, "rp2_6"),
            "mobius5": Small2Complex(5, [(i % 5, (i + 1) % 5, (i + 2) % 5) for i in range(5)], "mobius5")}

def _draw_complex(ctype, rng):
    if ctype == "random":
        return random_2complex(rng)
    if ctype == "sphere_rand":
        g = triangulated_sphere(7, rng); return Small2Complex(g.n, g.triangles.tolist(), "sphere8")
    return zoo[ctype]

def random_2complex(rng, nv=6):
    allt = list(itertools.combinations(range(nv), 3)); pick = rng.choice(len(allt), size=int(rng.integers(4, 10)), replace=False)
    tris = [allt[j] for j in pick]; used = sorted({v for t in tris for v in t}); rel = {v: i for i, v in enumerate(used)}
    return Small2Complex(len(used), [tuple(rel[v] for v in t) for t in tris], "random")

def betti_z2(cx):
    par = list(range(cx.nv))
    def find(x):
        while par[x] != x:
            par[x] = par[par[x]]; x = par[x]
        return x
    for a, b in cx.edges:
        par[find(a)] = find(b)
    b0 = len({find(v) for v in range(cx.nv)}); rB = _z2_rank(cx.tri_bd)
    return b0, len(_cycle_basis(range(len(cx.edges)), cx.edges)) - rB, len(cx.tris) - rB

zoo = complex_zoo(); _rz = rng_for("deg1_selfcheck")
for name, want in (("sphere", (1, 0, 1)), ("torus", (1, 2, 1)), ("rp2", (1, 1, 1)), ("mobius", (1, 1, 0)), ("torus7", (1, 2, 1)),
                   ("rp2_6", (1, 1, 1)), ("mobius5", (1, 1, 0))):
    DEG1_CHECKS["zoo_ok"] &= betti_z2(zoo[name]) == want
for _ in range(60):                             # rank-invariant barcodes agree with the matrix reduction and with the sweep
    cx = zoo[["sphere", "torus", "rp2", "mobius"][int(_rz.integers(0, 4))]] if _rz.random() < 0.7 else random_2complex(_rz)
    f = _rz.integers(0, 8, cx.nv).astype(float); b0, b1 = barcodes_01(cx, f)
    DEG1_CHECKS["rank_vs_reduction_mismatch"] += (tuple(image_barcode_rank(cx, f, f, 0)) != b0) + (tuple(image_barcode_rank(cx, f, f, 1)) != b1)
    l = _rz.integers(0, 8, cx.nv).astype(float); u_ = l + _rz.integers(0, 5, cx.nv)
    DEG1_CHECKS["image0_vs_sweep_mismatch"] += image_barcode_rank(cx, l, u_, 0) != image_bars(certify_graph(cx.graph, l, u_, 0.0))
print("self-checks:", DEG1_CHECKS)

self-checks: {'zoo_ok': True, 'rank_vs_reduction_mismatch': 0, 'image0_vs_sweep_mismatch': 0, 'negative_multiplicity': 0}


In [17]:
# ---- exact degree-1 machinery ------------------------------------------------------------------------------------------------
def bars_cx(cx, f, deg):
    """Positive-length lower-star bars of f on the complex cx (its own 1-skeleton, including edges outside triangles), sorted."""
    return sorted((float(b), float(d)) for b, d in lowerstar_diagrams(cx.nv, cx.tris, np.asarray(f, float), edges=cx.edges)[deg] if d > b)

def realizable_weak_orders(nv, l, u, cap=None):
    """Every ordered partition (B_0 < ... < B_{m-1}) of the vertices induced by some f in the box (f constant on blocks, strictly increasing
    across blocks); feasibility by the least-value greedy with a symbolic epsilon, x = (value, k) meaning value + k*eps. Returns (orders, complete)."""
    out = []; complete = [True]
    def rec(rem, blocks, prev):
        if cap is not None and len(out) >= cap:
            complete[0] = False; return
        if not rem:
            out.append(tuple(blocks)); return
        rl = sorted(rem)
        for r in range(1, len(rl) + 1):
            for S in itertools.combinations(rl, r):
                L = max(l[v] for v in S); U = min(u[v] for v in S)
                if L > U:
                    continue
                x = (L, 0) if prev is None else max((L, 0), (prev[0], prev[1] + 1))
                if x[0] > U or (x[0] == U and x[1] > 0):
                    continue
                rest = rem - set(S)
                if any(u[w] <= x[0] for w in rest):
                    continue
                rec(rest, blocks + [S], x)
                if cap is not None and len(out) >= cap:
                    complete[0] = False; return
    rec(frozenset(range(nv)), [], None)
    return out, complete[0]

def index_barcode(cx, blocks, deg):
    f = np.zeros(cx.nv)
    for j, S in enumerate(blocks):
        f[list(S)] = j
    return bars_cx(cx, f, deg)

def _fix_values(ibars, I):
    if len(ibars) != len(I):
        return None
    fixed = {}
    for (i, j), (b, d) in zip(sorted(ibars), sorted(I)):
        for idx, val in ((i, b), (j, d)):
            if math.isinf(idx) or math.isinf(val):
                if math.isinf(idx) != math.isinf(val):
                    return None
                continue
            if fixed.setdefault(int(idx), val) != val:
                return None
    return fixed

def realise_with_fixed(blocks, l, u, fixed):
    """Strictly increasing block values in the block bounds with x_j = fixed[j]; exact verdict (symbolic epsilon) and concrete values."""
    prev = None; sym = []
    for j, S in enumerate(blocks):
        L = max(l[v] for v in S); U = min(u[v] for v in S)
        if j in fixed:
            c = fixed[j]
            if not (L <= c <= U) or (prev is not None and not ((c, 0) > prev)):
                return False, None
            x = (c, 0)
        else:
            x = (L, 0) if prev is None else max((L, 0), (prev[0], prev[1] + 1))
            if x[0] > U or (x[0] == U and x[1] > 0):
                return False, None
        sym.append(x); prev = x
    cand = sorted({float(v) for v in list(l) + list(u) + list(fixed.values())})
    gaps = [b - a for a, b in zip(cand[:-1], cand[1:]) if b > a]
    eps = (min(gaps) if gaps else 1.0) / (2.0 * (len(blocks) + 1))
    return True, [x0 + k * eps for x0, k in sym]

def exact_attainment(cx, l, u, I, deg, cap=None, orders=None):
    """Exact decision of 'some f in the box has Bar_deg(f) == I' over all realisable order types; a witness is re-verified by recomputing its
    barcode. attained is True, False (complete enumeration) or None (order-type cap reached)."""
    orders, complete = orders if orders is not None else realizable_weak_orders(cx.nv, l, u, cap)
    I = sorted(I)
    for blocks in orders:
        fixed = _fix_values(index_barcode(cx, blocks, deg), I)
        if fixed is None:
            continue
        ok, vals = realise_with_fixed(blocks, l, u, fixed)
        if ok:
            f = np.zeros(cx.nv)
            for j, S in enumerate(blocks):
                f[list(S)] = vals[j]
            verified = bool(bars_cx(cx, f, deg) == I and np.all(f >= l) and np.all(f <= u))
            return dict(attained=True, witness=f, verified=verified, n_orders=len(orders), complete=complete)
    return dict(attained=False if complete else None, witness=None, verified=None, n_orders=len(orders), complete=complete)

# exact minimum region counts: difference constraints x_i - x_j <= w with w = (rational, eps-coefficient), decided by Bellman-Ford
def _w(val, eps=0):
    return (Fraction(val).limit_denominator(10**12), eps)

def difference_feasible(nvars, cons):
    dist = [(Fraction(0), 0)] * (nvars + 1); edges = [(j, i, w) for i, j, w in cons]
    for _ in range(nvars + 1):
        changed = False
        for a, b, w in edges:
            nd = (dist[a][0] + w[0], dist[a][1] + w[1])
            if nd < dist[b]:
                dist[b] = nd; changed = True
        if not changed:
            return True
    return False

def _order_constraints(blocks, l, u):
    m = len(blocks); z = m; cons = []
    for j, S in enumerate(blocks):
        cons += [(j, z, _w(min(u[v] for v in S))), (z, j, _w(-max(l[v] for v in S)))]
        if j + 1 < m:
            cons.append((j, j + 1, (Fraction(0), -1)))
    return cons

def exact_min_persistence_count(orders, ibars, l, u, tau):
    """min over the box of #bars with death - birth > tau (exact; needs the complete list of realisable order types)."""
    tau = Fraction(tau).limit_denominator(10**12); best = None
    for blocks, ib in zip(orders, ibars):
        fin = [(int(b), int(d)) for b, d in ib if not math.isinf(d)]; ess = len(ib) - len(fin)
        if best is not None and ess >= best:
            continue
        base = _order_constraints(blocks, l, u)
        for size in range(len(fin), -1, -1):
            if best is not None and ess + len(fin) - size >= best:
                break
            if any(difference_feasible(len(blocks), base + [(d, b, (tau, 0)) for b, d in short])
                   for short in itertools.combinations(fin, size)):
                best = ess + len(fin) - size if best is None else min(best, ess + len(fin) - size)
                break
    return best

def exact_min_quadrant_count(orders, ibars, l, u, beta, delta):
    """min over the box of #bars with birth <= beta and death >= delta (exact)."""
    beta = Fraction(beta).limit_denominator(10**12); delta = Fraction(delta).limit_denominator(10**12); best = None
    for blocks, ib in zip(orders, ibars):
        m = len(blocks); z = m; base = _order_constraints(blocks, l, u)
        bl = [(int(b), None if math.isinf(d) else int(d)) for b, d in ib]
        for combo in itertools.product(*[[("in",), ("b",)] + ([("d",)] if d is not None else []) for b, d in bl]):
            cnt = sum(1 for c in combo if c[0] == "in")
            if best is not None and cnt >= best:
                continue
            cons = list(base)
            for (b, d), c in zip(bl, combo):
                if c[0] == "b":
                    cons.append((z, b, (-beta, -1)))
                elif c[0] == "d":
                    cons.append((d, z, (delta, -1)))
            if difference_feasible(m, cons):
                best = cnt
    return best

def _truthy_count(values):
    """Number of entries that are present (not None/NaN) and true; avoids object-dtype fillna downcasting."""
    return int(sum(1 for v in values if v is not None and v == v and bool(v)))

def bars_txt(I):
    """Barcode as text, e.g. '[2,5) [7,inf)'."""
    return " ".join(f"[{b:g},{'inf' if math.isinf(d) else format(d, 'g')})" for b, d in I)

def image_count_pers(I, tau):
    return sum(1 for b, d in I if d - b > tau)

def image_count_quad(I, beta, delta):
    return sum(1 for b, d in I if b <= beta and d >= delta)

# links, KC vertices, surfaces
def link_graph(cx, v):
    E = set(); nb = set()
    for t in cx.tris:
        if v in t:
            a, b = [w for w in t if w != v]; E.add((min(a, b), max(a, b)))
    for a, b in cx.edges:
        if a == v: nb.add(b)
        if b == v: nb.add(a)
    return sorted(nb), sorted(E)

def _graph_components(V, E):
    par = {x: x for x in V}
    def find(x):
        while par[x] != x:
            par[x] = par[par[x]]; x = par[x]
        return x
    for a, b in E:
        ra, rb = find(a), find(b)
        if ra != rb:
            par[ra] = rb
    return len({find(x) for x in V})

def is_kc_link(V, E):
    for r in range(4, len(V) + 1):
        for W in itertools.combinations(V, r):
            Ws = set(W); EW = [(a, b) for a, b in E if a in Ws and b in Ws]; c = _graph_components(W, EW)
            if c >= 2 and len(EW) - len(W) + c > 0:
                return True
    return False

def kc_vertices(cx):
    return [v for v in range(cx.nv) if is_kc_link(*link_graph(cx, v))]

def link_type(V, E):
    if not V:
        return "isolated"
    deg = {x: 0 for x in V}
    for a, b in E:
        deg[a] += 1; deg[b] += 1
    c = _graph_components(V, E)
    if c == 1 and len(E) == len(V) >= 3 and all(d == 2 for d in deg.values()):
        return "cycle"
    if c == 1 and len(E) == len(V) - 1 and all(d <= 2 for d in deg.values()):
        return "path"
    return "singular"

def edge_triangle_degrees(cx):
    cnt = {e: 0 for e in cx.edges}
    for a, b, c in cx.tris:
        for e in ((a, b), (a, c), (b, c)):
            cnt[e] += 1
    return cnt

def is_surface(cx):
    deg = edge_triangle_degrees(cx)
    return all(k in (1, 2) for k in deg.values()) and all(link_type(*link_graph(cx, v)) in ("cycle", "path") for v in range(cx.nv))

def boundary_cycles(cx):
    adj = {}
    for (a, b), k in edge_triangle_degrees(cx).items():
        if k == 1:
            adj.setdefault(a, []).append(b); adj.setdefault(b, []).append(a)
    seen = set(); cycles = []
    for s in sorted(adj):
        if s in seen:
            continue
        cyc = [s]; seen.add(s); prev, cur = None, s
        while True:
            nxt = [w for w in adj[cur] if w != prev]
            if not nxt or nxt[0] == s:
                break
            prev, cur = cur, nxt[0]; cyc.append(cur); seen.add(cur)
        cycles.append(cyc)
    return cycles

def is_orientable(cx):
    tris = [tuple(t) for t in cx.tris]; by_edge = {}
    for k, (a, b, c) in enumerate(tris):
        for e in ((a, b), (b, c), (a, c)):
            by_edge.setdefault(e, []).append(k)
    orient = [None] * len(tris)
    def directed(t, o):
        a, b, c = t if o == 1 else (t[0], t[2], t[1])
        return {(a, b), (b, c), (c, a)}
    for s in range(len(tris)):
        if orient[s] is not None:
            continue
        orient[s] = 1; stack = [s]
        while stack:
            k = stack.pop(); dk = directed(tris[k], orient[k])
            for e in ((tris[k][0], tris[k][1]), (tris[k][1], tris[k][2]), (tris[k][0], tris[k][2])):
                for j in by_edge[e]:
                    if j == k:
                        continue
                    o = 1 if not (dk & directed(tris[j], 1)) else -1
                    if dk & directed(tris[j], o):
                        return False
                    if orient[j] is None:
                        orient[j] = o; stack.append(j)
                    elif orient[j] != o:
                        return False
    return True

def surface_planar(cx):
    """Genus 0 test for a connected surface: orientable and chi = 2 - (number of boundary components)."""
    chi = cx.nv - len(cx.edges) + len(cx.tris)
    return bool(is_orientable(cx) and chi == 2 - len(boundary_cycles(cx)))

def complex_class(cx):
    if kc_vertices(cx):
        return "KC vertex"
    if any(k >= 3 for k in edge_triangle_degrees(cx).values()):
        return "non-manifold edge, no KC"
    if not is_surface(cx):
        return "cycle-or-paths links, not a surface"
    return "planar surface" if surface_planar(cx) else "non-planar surface"

def cone_boundaries(cx, l, u, M):
    tris = [tuple(t) for t in cx.tris]; nv = cx.nv; L = list(map(float, l)); U = list(map(float, u))
    for cyc in boundary_cycles(cx):
        w = nv; nv += 1; L.append(M); U.append(M)
        tris += [tuple(sorted((a, b, w))) for a, b in zip(cyc, cyc[1:] + cyc[:1])]
    return Small2Complex(nv, tris, "coned"), np.array(L), np.array(U)

def glue_along_boundary(P, lP, uP, Q, M):
    bp = boundary_cycles(P)[0]; bq = boundary_cycles(Q)[0]
    assert len(bp) == len(bq) == 3
    mp = dict(zip(bq, bp)); nv = P.nv
    for v in range(Q.nv):
        if v not in mp:
            mp[v] = nv; nv += 1
    tris = [tuple(t) for t in P.tris] + [tuple(sorted(mp[v] for v in t)) for t in Q.tris]
    assert len(set(tris)) == len(tris)
    return Small2Complex(nv, tris, "glued"), np.array(list(map(float, lP)) + [M] * (nv - P.nv)), np.array(list(map(float, uP)) + [M] * (nv - P.nv))

def dual_realiser(cx, l, u):
    return -universal_realizer(certify_graph(cx.graph, -u, -l, 0.0), -u)

def lower_link(cx, v, f):
    V, E = link_graph(cx, v); S = {w for w in V if f[w] < f[v]}
    return sorted(S), [(a, b) for a, b in E if a in S and b in S]

def kill_create_counts(cx, f):
    pos = {float(f[v]): v for v in range(cx.nv)}; kills = [0] * cx.nv; creates = [0] * cx.nv
    for b, d in bars_cx(cx, f, 1):
        creates[pos[b]] += 1
        if not math.isinf(d):
            kills[pos[d]] += 1
    return kills, creates

def one_free_vertex_attains(cx, l, u, x, I, deg):
    """Exact attainment when x is the only free vertex: test every fixed value in [l(x), u(x)], both ends, every target endpoint in the interval
    and one interior point between consecutive candidates (the barcode is constant between them up to endpoints equal to f(x))."""
    others = sorted({float(l[v]) for v in range(cx.nv) if v != x}); lo, hi = float(l[x]), float(u[x])
    pts = sorted({lo, hi} | {p for p in others if lo <= p <= hi})
    cands = set(pts) | {0.5 * (a + b) for a, b in zip(pts[:-1], pts[1:])} | {e for bar in I for e in bar if not math.isinf(e) and lo <= e <= hi}
    I = sorted(I)
    for c in sorted(cands):
        f = np.array(l, float); f[x] = c
        if bars_cx(cx, f, deg) == I:
            return True, c
    return False, None

def random_box(rng, n):
    d = int(rng.integers(0, 3))
    if d == 0:
        l = rng.integers(0, 8, n) * 2.0; u = l + rng.integers(0, 4, n) * 2.0
    elif d == 1:
        l = rng.integers(0, 6, n) * 1.0; u = l + rng.integers(0, 3, n) * 1.0
    else:
        l = rng.integers(0, 12, n) * 1.0; u = l + rng.integers(0, 6, n) * 1.0
    return l, u

def has_finite_bar(I):
    return any(math.isfinite(d) for _, d in I)

def random_pure_complex(rng, nv, k, max_edge_deg=None):
    allt = list(itertools.combinations(range(nv), 3)); order = rng.permutation(len(allt)); cnt = collections.Counter(); tris = []
    for idx in order:
        t = allt[idx]; es = [(t[0], t[1]), (t[0], t[2]), (t[1], t[2])]
        if max_edge_deg is None or all(cnt[e] < max_edge_deg for e in es):
            tris.append(t); cnt.update(es)
            if len(tris) == k:
                break
    used = sorted({v for t in tris for v in t}); rel = {v: i for i, v in enumerate(used)}
    return Small2Complex(len(used), [tuple(rel[v] for v in t) for t in tris], "random")

def decide_degree1(cx, l, u, I1, cap):
    """Witnesses first (f_D, f_U), then exact enumeration. Returns (status, how)."""
    if bars_cx(cx, dual_realiser(cx, l, u), 1) == I1:
        return "attained", "f_D"
    if bars_cx(cx, universal_realizer(certify_graph(cx.graph, l, u, 0.0), l), 1) == I1:
        return "attained", "f_U"
    ex = exact_attainment(cx, l, u, I1, 1, cap=cap)
    if ex["attained"] is True:
        return ("attained" if ex["verified"] else "WITNESS FAILED"), "exact"
    return ("NOT attained" if ex["attained"] is False else "undecided"), "exact"

In [18]:
def weak_orders_bruteforce(nv, l, u):
    out = []
    for m in range(1, nv + 1):
        for r in itertools.product(range(m), repeat=nv):
            if len(set(r)) == m:
                blocks = tuple(tuple(v for v in range(nv) if r[v] == j) for j in range(m))
                if difference_feasible(m, _order_constraints(blocks, l, u)):
                    out.append(blocks)
    return out

def attained_bellman_ford(cx, l, u, I, orders):
    I = sorted(I)
    for blocks in orders:
        fixed = _fix_values(index_barcode(cx, blocks, 1), I)
        if fixed is None:
            continue
        cons = _order_constraints(blocks, l, u) + [c for j, val in fixed.items() for c in ((j, len(blocks), _w(val)), (len(blocks), j, _w(-val)))]
        if difference_feasible(len(blocks), cons):
            return True
    return False

ck_d1x_sc = Checkpoint("v4_degree1_selfcheck")
if not ck_d1x_sc.done("all"):
    rng = rng_for("d1x_selfcheck"); rows = []
    for i in range(CONFIG["d1x_selfcheck"]):
        cx = random_pure_complex(rng, int(rng.integers(4, 7)), int(rng.integers(2, 11)))
        l, u = random_box(rng, cx.nv); orders = realizable_weak_orders(cx.nv, l, u, cap=20000)
        if not orders[1]:
            continue
        r = dict(i=i, nv=cx.nv, n_orders=len(orders[0]))
        I0 = image_barcode_rank(cx, l, u, 0); ex0 = exact_attainment(cx, l, u, I0, 0, orders=orders)
        r["deg0_attained"] = bool(ex0["attained"]) and bool(ex0["verified"])
        mism0 = lbv = 0
        for deg in (0, 1):
            I = I0 if deg == 0 else image_barcode_rank(cx, l, u, 1); ib = [index_barcode(cx, o, deg) for o in orders[0]]
            vals = sorted(set(l) | set(u))
            for tau in sorted({0.0} | {float(abs(a - b)) for a in vals[:4] for b in vals[-4:]})[:6]:
                mn = exact_min_persistence_count(orders[0], ib, l, u, tau); ic = image_count_pers(I, tau)
                lbv += mn < ic; mism0 += (deg == 0 and mn != ic)
            for be, de in ((vals[len(vals) // 3], vals[len(vals) // 2]), (vals[0], vals[-1]), (vals[len(vals) // 2], vals[len(vals) // 2])):
                mn = exact_min_quadrant_count(orders[0], ib, l, u, be, de); ic = image_count_quad(I, be, de)
                lbv += mn < ic; mism0 += (deg == 0 and mn != ic)
        r.update(deg0_region_mismatch=mism0, lower_bound_violations=lbv)
        if i < CONFIG["d1x_indep"] and cx.nv <= 6:
            I1 = image_barcode_rank(cx, l, u, 1); ob = weak_orders_bruteforce(cx.nv, l, u)
            r.update(indep_checked=True, indep_order_mismatch=int(len(ob) != len(orders[0])),
                     indep_decision_mismatch=int(attained_bellman_ford(cx, l, u, I1, ob) != bool(exact_attainment(cx, l, u, I1, 1, orders=orders)["attained"])))
        rows.append(r)
    ck_d1x_sc.add("all", rows)
d1x_sc = ck_d1x_sc.frame()
D1X_CHECKS = dict(instances=int(len(d1x_sc)), deg0_not_attained=int((~d1x_sc.deg0_attained.astype(bool)).sum()),
                  deg0_region_mismatches=int(d1x_sc.deg0_region_mismatch.sum()), lower_bound_violations=int(d1x_sc.lower_bound_violations.sum()),
                  indep_checked=_truthy_count(d1x_sc.get("indep_checked", pd.Series(dtype=float))),
                  indep_order_mismatches=int(d1x_sc.get("indep_order_mismatch", pd.Series(dtype=float)).fillna(0).sum()),
                  indep_decision_mismatches=int(d1x_sc.get("indep_decision_mismatch", pd.Series(dtype=float)).fillna(0).sum()))
print("exact machinery self-checks:", D1X_CHECKS)

exact machinery self-checks: {'instances': 400, 'deg0_not_attained': 0, 'deg0_region_mismatches': 0, 'lower_bound_violations': 0, 'indep_checked': 60, 'indep_order_mismatches': 0, 'indep_decision_mismatches': 0}


## 8 · Lemma 16: KC vertices

This section uses random injective functions on random 2-complexes. It checks that:
- a vertex that kills a degree-1 class has a cycle in its lower link;
- a vertex that creates one has a disconnected lower link;
- no vertex that is not KC does both.

As a positive control it also counts the entrances of KC vertices that do both, which shows that the check can fail.

In [19]:
ck_d1x_kc = Checkpoint("v4_degree1_kclemma")
if not ck_d1x_kc.done("all"):
    rng = rng_for("d1x_kclemma"); rows = []
    for i in range(CONFIG["d1x_lemma_complexes"]):
        nv = int(rng.integers(5, 9)); cx = random_pure_complex(rng, nv, int(rng.integers(3, 3 * nv)))
        kcv = set(kc_vertices(cx)); r = dict(i=i, kc=bool(kcv), functions=0, mv_kill=0, mv_create=0, lemma=0, control=0)
        for _ in range(4):
            f = rng.permutation(cx.nv) + rng.random(cx.nv) * 0.5; k_, c_ = kill_create_counts(cx, f); r["functions"] += 1
            for v in range(cx.nv):
                Vl, El = lower_link(cx, v, f); nc = _graph_components(Vl, El) if Vl else 0
                cyc = bool(Vl) and len(El) - len(Vl) + nc > 0
                r["mv_kill"] += k_[v] > 0 and not cyc; r["mv_create"] += c_[v] > 0 and nc < 2
                if k_[v] > 0 and c_[v] > 0:
                    r["lemma" if v not in kcv else "control"] += 1
        rows.append(r)
    ck_d1x_kc.add("all", rows)
d1x_kc = ck_d1x_kc.frame()
KC_LEMMA = dict(complexes=int(len(d1x_kc)), functions=int(d1x_kc.functions.sum()), kc_free_complexes=int((~d1x_kc.kc.astype(bool)).sum()),
                mv_violations=int(d1x_kc.mv_kill.sum() + d1x_kc.mv_create.sum()), lemma_violations=int(d1x_kc.lemma.sum()),
                positive_control_events=int(d1x_kc.control.sum()))
print("Lemma 16 (KC vertices):", KC_LEMMA)

Lemma 16 (KC vertices): {'complexes': 1500, 'functions': 6000, 'kc_free_complexes': 839, 'mv_violations': 0, 'lemma_violations': 0, 'positive_control_events': 283}


## 9 · Theorem 8 for $d = 2$: planar complexes

The witness is always the theorem's $f_D=-f_U(-u,-\ell)$. For a full subcomplex $K\subset L$ it is computed on the sphere $L$, with the vertices outside $K$ padded at $M$. For planar surfaces with boundary it is computed on the coned surface. For comparison, the section also reports $f_D$ computed on the complex itself, which no theorem covers.

In [20]:
ck_d1x_pl = Checkpoint("v4_degree1_planar")
if not ck_d1x_pl.done("all"):
    rng = rng_for("d1x_planar"); rows = []
    for fam in ("sphere", "full subcomplex of a sphere", "disk", "annulus", "sphere with 3 holes"):
        done = tries = 0
        while done < CONFIG["d1x_planar"] and tries < 40 * CONFIG["d1x_planar"]:
            tries += 1
            g = triangulated_sphere(int(rng.integers(5, 26)), rng); Lc = Small2Complex(g.n, [tuple(t) for t in g.triangles.tolist()], "S")
            if fam in ("sphere", "full subcomplex of a sphere"):
                l, u = random_box(rng, Lc.nv)
                if fam == "sphere":
                    K, keep = Lc, list(range(Lc.nv))
                else:
                    W = set(rng.choice(Lc.nv, size=int(rng.integers(1, max(2, Lc.nv // 2))), replace=False).tolist())
                    keep = [v for v in range(Lc.nv) if v not in W]; rel = {v: i for i, v in enumerate(keep)}; M = float(u.max() + 10)
                    l[list(W)] = M; u[list(W)] = M
                    K = Small2Complex(len(keep), [tuple(rel[x] for x in t) for t in Lc.tris if all(x in rel for x in t)], "K",
                                      edges=[(rel[a], rel[b]) for a, b in Lc.edges if a in rel and b in rel])
                lk, uk = l[keep], u[keep]; I1 = image_barcode_rank(K, lk, uk, 1)
                if not has_finite_bar(I1):
                    continue
                fD = dual_realiser(Lc, l, u)[keep]; ok_thm = bars_cx(K, fD, 1) == I1; ok_plain = bars_cx(K, dual_realiser(K, lk, uk), 1) == I1
            else:
                holes = {"disk": 1, "annulus": 2, "sphere with 3 holes": 3}[fam]
                tris = [tuple(sorted(t)) for t in Lc.tris]; used = set(); removed = []
                for idx in rng.permutation(len(tris)):
                    if used.isdisjoint(tris[idx]):
                        removed.append(tris[idx]); used |= set(tris[idx])
                        if len(removed) == holes:
                            break
                if len(removed) < holes:
                    continue
                K = Small2Complex(Lc.nv, [t for t in tris if t not in set(removed)], fam); l, u = random_box(rng, K.nv)
                I1 = image_barcode_rank(K, l, u, 1)
                if not has_finite_bar(I1):
                    continue
                C, lc, uc = cone_boundaries(K, l, u, float(u.max() + 10))
                ok_thm = bars_cx(K, dual_realiser(C, lc, uc)[:K.nv], 1) == I1; ok_plain = bars_cx(K, dual_realiser(K, l, u), 1) == I1
            done += 1
            lo_, hi_ = (lk, uk) if fam in ("sphere", "full subcomplex of a sphere") else (l, u)
            frand = lo_ + rng.random(K.nv) * (hi_ - lo_)                       # sub-barcode validity on a random realisation
            rows.append(dict(family=fam, nv=K.nv, bars=len(I1), theorem_witness=bool(ok_thm), plain_dual=bool(ok_plain),
                             validity=bool(is_subbarcode(list(I1), bars_cx(K, frand, 1)))))
    ck_d1x_pl.add("all", rows)
d1x_pl = ck_d1x_pl.frame()
d1x_pls = d1x_pl.groupby("family").agg(instances=("nv", "size"), max_vertices=("nv", "max"), theorem_witness=("theorem_witness", "mean"),
                                       plain_dual=("plain_dual", "mean"), validity=("validity", "mean")).reset_index()
PLANAR = dict(instances=int(len(d1x_pl)), theorem_failures=int((~d1x_pl.theorem_witness.astype(bool)).sum()),
              validity_violations=int((~d1x_pl.validity.astype(bool)).sum()))
print(d1x_pls.round(3).to_string(index=False)); print("Theorem 8 (planar complexes):", PLANAR)

                     family  instances  max_vertices  theorem_witness  plain_dual  validity
                    annulus        300            26              1.0       0.930       1.0
                       disk        300            26              1.0       1.000       1.0
full subcomplex of a sphere        300            25              1.0       0.827       1.0
                     sphere        300            26              1.0       1.000       1.0
        sphere with 3 holes        300            26              1.0       0.817       1.0
Theorem 8 (planar complexes): {'instances': 1500, 'theorem_failures': 0, 'validity_violations': 0}


## 10 · Theorem 11 and Examples 17–20: certificates, decided by complete enumeration

Each certificate is an explicit input from a hand proof. The code computes its image barcode, compares it with the proof's prediction, and decides attainment by enumerating every order type. For count certificates it compares the exact minimum τ-counts with the image counts.

| | Paper | Complex | Claim |
|---|---|---|---|
| A | Example 17 | a 2-complex with a KC vertex | the barcode is not attained |
| B | Example 18 | a 2-complex with a KC vertex | the counts are not attained either (τ ∈ [3, 4)) |
| C | Example 19 | no KC vertex; one edge lies in three triangles | the barcode is not attained |
| D | Theorem 11, Figure 2 | Möbius band | the barcode is not attained |
| E | Theorem 11, Appendix B | punctured torus | the barcode is not attained |
| F | Example 20 | no KC vertex: Example 19 doubled and coned at 11 | the counts are not attained (τ ∈ [3, 3.5)) |

**Local minimality.** For A–C, deleting any triangle or collapsing any free interval to an endpoint must destroy the property. For D and E only the collapses are tested, so the surfaces stay intact. Minimality is not claimed for F.

**Transfers (Theorem 11).** Coning or gluing D and E gives the projective plane, the torus, the Klein bottle, the genus-2 surface and the non-orientable surface of genus 3. Each is shown to be unattainable by complete enumeration.

**Example 20.** Its image barcode is $\{[4,6),[8,11)\}$, so no bar is longer than 3, yet every function in the box has a bar of length at least 3.5. No vertex is KC, so Lemma 16 does not apply; the failure comes from constraints at two filtration levels.

In [21]:
CERT_SPECS = {
    "A": dict(tris=[(0, 1, 2), (0, 2, 3), (0, 1, 3), (0, 4, 5), (1, 4, 5)], l=[5, 0, 1, 2, 3, 4], u=[7, 0, 1, 2, 3, 4],
              predicted=[(2.0, 5.0), (7.0, math.inf)], prop="barcode", note="local obstruction: KC vertex x = 0; every edge in at most two triangles"),
    "B": dict(tris=[(0, 1, 2), (0, 2, 3), (0, 1, 3), (0, 1, 5), (1, 4, 5), (0, 4, 5)], l=[5, 0, 1, 2, 3, 10], u=[7, 0, 1, 2, 3, 10],
              predicted=[(2.0, 5.0), (7.0, 10.0)], prop="count", note="count gap for tau in [3, 4)"),
    "C": dict(tris=[(0, 1, 4), (0, 3, 5), (0, 4, 5), (1, 2, 4), (1, 2, 5), (2, 3, 4), (2, 4, 5)], l=[2, 1, 1, 0, 3, 0], u=[2, 1, 4, 0, 5, 0],
              predicted=[(2.0, 3.0), (4.0, math.inf)], prop="barcode", note="no KC vertex; edge 24 in three triangles"),
    "D": dict(tris=[(0, 1, 4), (0, 2, 3), (0, 2, 5), (0, 4, 5), (1, 2, 4), (1, 2, 5), (1, 3, 5), (2, 3, 4), (3, 4, 5)], l=[0, 0, 1, 0, 1, 2],
              u=[0, 0, 3, 0, 4, 4], predicted=[(0.0, 2.0), (3.0, math.inf)], prop="barcode", note="Moebius band, boundary 0-1-3"),
    "E": dict(tris=[(0, 1, 3), (0, 1, 5), (0, 2, 3), (0, 2, 6), (0, 4, 6), (1, 2, 4), (1, 2, 6), (1, 3, 4), (1, 5, 6), (2, 3, 5), (2, 4, 5),
                    (3, 4, 6), (3, 5, 6)], l=[0, 4, 0, 4, 1, 2, 3], u=[0, 7, 0, 6, 1, 2, 5],
              predicted=[(1.0, math.inf), (2.0, 4.0), (5.0, math.inf)], prop="barcode", note="punctured torus, boundary 0-4-5"),
}
# F = Example 20: Example 19 with its box doubled, plus a vertex z = 6 fixed at 11 that cones all
# 13 edges of Example 19. Only the count property is claimed, not local minimality.
_t19 = [tuple(sorted(t)) for t in CERT_SPECS["C"]["tris"]]
_e19 = sorted({e for t in _t19 for e in itertools.combinations(t, 2)})
CERT_SPECS["F"] = dict(tris=_t19 + [(a, b, 6) for a, b in _e19], l=[2 * x for x in CERT_SPECS["C"]["l"]] + [11],
                       u=[2 * x for x in CERT_SPECS["C"]["u"]] + [11], predicted=[(4.0, 6.0), (8.0, 11.0)], prop="count", check_minimal=False,
                       note="Example 20: no KC vertex; image bars of lengths 2 and 3, yet every function has a bar of length >= 3.5")

def _make(tris, l, u):
    used = sorted({v for t in tris for v in t}); rel = {v: i for i, v in enumerate(used)}
    return (Small2Complex(len(used), [tuple(rel[v] for v in t) for t in tris], "cert"),
            np.array([l[v] for v in used], float), np.array([u[v] for v in used], float))

def cert_property(cx, l, u, prop):
    """'barcode': the image barcode has a finite bar and is not attained (exact). 'count': some tau has exact min count > image count."""
    I1 = image_barcode_rank(cx, l, u, 1)
    if not has_finite_bar(I1):
        return False
    orders = realizable_weak_orders(cx.nv, l, u, cap=200000)
    if not orders[1]:
        return None
    if prop == "barcode":
        return exact_attainment(cx, l, u, I1, 1, orders=orders)["attained"] is False
    ib = [index_barcode(cx, o, 1) for o in orders[0]]
    lens = sorted({d - b for b, d in I1 if math.isfinite(d)} | {0.0})
    taus = sorted({t + e for t in lens for e in (-0.5, 0.0, 0.5) if t + e >= 0})
    return any(exact_min_persistence_count(orders[0], ib, l, u, t) > image_count_pers(I1, t) for t in taus)

def free_vertex_sweep(cx, l, u, x, n_grid=41):
    """Bar_1(f_c) for f = l elsewhere and f(x) = c on a grid of c in [l(x), u(x)]."""
    return [(float(c), bars_cx(cx, np.where(np.arange(cx.nv) == x, c, l), 1)) for c in np.linspace(l[x], u[x], n_grid)]

CERTS = {}
for name, S in CERT_SPECS.items():
    cx, l, u = _make(S["tris"], S["l"], S["u"]); I1 = image_barcode_rank(cx, l, u, 1)
    orders = realizable_weak_orders(cx.nv, l, u); ib = [index_barcode(cx, o, 1) for o in orders[0]]
    ex = exact_attainment(cx, l, u, I1, 1, orders=orders)
    taus = sorted({0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.25, 3.5, 3.99, 4.0, 4.5, 5.0})
    pers = {t: (exact_min_persistence_count(orders[0], ib, l, u, t), image_count_pers(I1, t)) for t in taus}
    vals = sorted(set(l) | set(u)); grid = sorted(set(vals) | {0.5 * (a + b) for a, b in zip(vals[:-1], vals[1:])})
    qgaps = [(be, de, exact_min_quadrant_count(orders[0], ib, l, u, be, de), image_count_quad(I1, be, de)) for be in grid for de in grid]
    free = [v for v in range(cx.nv) if l[v] != u[v]]
    # local minimality (claimed for A-E only)
    minimal = True if S.get("check_minimal", True) else None; checks = 0
    if name in ("A", "B", "C"):
        for t in S["tris"]:
            sub = _make([s for s in S["tris"] if s != t], S["l"], S["u"]); checks += 1
            if cert_property(*sub, S["prop"]):
                minimal = False
    for v in (free if S.get("check_minimal", True) else []):
        for val in (l[v], u[v]):
            l2, u2 = l.copy(), u.copy(); l2[v] = u2[v] = val; checks += 1
            if cert_property(cx, l2, u2, S["prop"]):
                minimal = False
    C = dict(name=name, nv=cx.nv, ntri=len(cx.tris), betti=betti_z2(cx), chi=cx.nv - len(cx.edges) + len(cx.tris), kc_vertices=kc_vertices(cx),
             max_edge_degree=max(edge_triangle_degrees(cx).values()), surface=is_surface(cx), orientable=is_orientable(cx) if is_surface(cx) else None,
             planar_surface=surface_planar(cx) if is_surface(cx) else None, free_vertices=free, I1=I1, predicted=S["predicted"],
             matches_prediction=I1 == S["predicted"], n_orders=len(orders[0]), complete=orders[1], attained=ex["attained"],
             pers={t: (int(a), int(b)) for t, (a, b) in pers.items()}, pers_gap_taus=[t for t, (a, b) in pers.items() if a > b],
             quad_gaps=[q for q in qgaps if q[2] != q[3]], quad_checked=len(qgaps), lower_bound_violations=int(sum(a < b for a, b in pers.values()) +
             sum(q[2] < q[3] for q in qgaps)), locally_minimal=minimal, minimality_checks=checks, note=S["note"],
             sweep=free_vertex_sweep(cx, l, u, free[0]) if len(free) == 1 else None, l=l.tolist(), u=u.tolist(), tris=[list(t) for t in cx.tris])
    C["property_holds"] = (C["attained"] is False) and (bool(C["pers_gap_taus"]) if S["prop"] == "count" else True)
    CERTS[name] = C
    print(f"certificate {name}: V={C['nv']} T={C['ntri']} betti={C['betti']} KC={C['kc_vertices']} max edge degree {C['max_edge_degree']} "
          f"surface={C['surface']} | I1={I1} (as predicted: {C['matches_prediction']}) | attained={C['attained']} ({C['n_orders']} order types) | "
          f"count gaps at tau {C['pers_gap_taus']} | quadrant gaps {len(C['quad_gaps'])}/{C['quad_checked']} | locally minimal {minimal}")

# transfers of D and E to closed surfaces: cone the boundary, or glue a second surface along it (every vertex off the seam fixed at M)
TRANSFERS = []
_D = _make(CERT_SPECS["D"]["tris"], CERT_SPECS["D"]["l"], CERT_SPECS["D"]["u"]); _E = _make(CERT_SPECS["E"]["tris"], CERT_SPECS["E"]["l"], CERT_SPECS["E"]["u"])
for label, base, how in (("RP^2 (D + cone)", _D, "cone"), ("torus (E + cone)", _E, "cone"), ("Klein bottle (D + Moebius band)", _D, _D[0]),
                         ("genus 2 (E + punctured torus)", _E, _E[0]), ("non-orientable genus 3 (D + punctured torus)", _D, _E[0])):
    P, lP, uP = base; M = float(uP.max() + 3)
    S_, L_, U_ = cone_boundaries(P, lP, uP, M) if how == "cone" else glue_along_boundary(P, lP, uP, how, M)
    I1 = image_barcode_rank(S_, L_, U_, 1); ex = exact_attainment(S_, L_, U_, I1, 1)
    TRANSFERS.append(dict(surface=label, nv=S_.nv, ntri=len(S_.tris), betti=betti_z2(S_), chi=S_.nv - len(S_.edges) + len(S_.tris),
                          closed_surface=bool(is_surface(S_) and not boundary_cycles(S_)), orientable=is_orientable(S_), I1=bars_txt(I1),
                          n_orders=ex["n_orders"], complete=ex["complete"], attained=ex["attained"]))
transfers_df = pd.DataFrame(TRANSFERS); print(transfers_df.to_string(index=False))

certificate A: V=6 T=5 betti=(1, 1, 0) KC=[0] max edge degree 2 surface=False | I1=[(2.0, 5.0), (7.0, inf)] (as predicted: True) | attained=False (1 order types) | count gaps at tau [] | quadrant gaps 1/169 | locally minimal True
certificate B: V=6 T=6 betti=(1, 0, 0) KC=[0] max edge degree 3 surface=False | I1=[(2.0, 5.0), (7.0, 10.0)] (as predicted: True) | attained=False (1 order types) | count gaps at tau [3.0, 3.25, 3.5, 3.99] | quadrant gaps 1/169 | locally minimal True
certificate C: V=6 T=7 betti=(1, 1, 0) KC=[] max edge degree 3 surface=False | I1=[(2.0, 3.0), (4.0, inf)] (as predicted: True) | attained=False (6 order types) | count gaps at tau [] | quadrant gaps 1/121 | locally minimal True
certificate D: V=6 T=9 betti=(1, 1, 0) KC=[] max edge degree 2 surface=True | I1=[(0.0, 2.0), (3.0, inf)] (as predicted: True) | attained=False (13 order types) | count gaps at tau [] | quadrant gaps 1/81 | locally minimal True
certificate E: V=7 T=13 betti=(1, 2, 0) KC=[] max edge degree 

In [22]:
# Example 20. Certificate F above is Example 19 with its box doubled, plus a vertex z = 6 fixed
# at 11 that cones all 13 edges. Three checks: (a) no vertex is KC, so Lemma 16 does not apply; (b) the image barcode is {[4,6), [8,11)},
# so T_3 = 0; (c) every function in the box has a bar of length at least 3.5, so N_3(f) >= 1 > T_3 (exhaustive over order types, plus
# the inequality longest >= max(D - 4, 11 - D), D = max(f(2), f(4)), from the hand proof, on a grid).
S20 = CERT_SPECS["F"]; cx20, l20, u20 = _make(S20["tris"], S20["l"], S20["u"]); C20 = CERTS["F"]
_K19 = {tuple(sorted(t)) for t in CERT_SPECS["C"]["tris"]}
_E19 = sorted({e for t in _K19 for e in itertools.combinations(t, 2)})
assert len(_E19) == 13 and sorted(tuple(sorted(t)) for t in S20["tris"]) == sorted(_K19 | {(a, b, 6) for a, b in _E19})
orders20 = realizable_weak_orders(cx20.nv, l20, u20); ib20 = [index_barcode(cx20, o, 1) for o in orders20[0]]
EX20_TAUS = [2.0, 2.5, 3.0, 3.25, 3.49, 3.5, 4.0]
ex20_counts = {t: (exact_min_persistence_count(orders20[0], ib20, l20, u20, t), image_count_pers(C20["I1"], t)) for t in EX20_TAUS}

_step = CONFIG["ex20_grid_step"]; grid20 = []
for a in np.arange(l20[2], u20[2] + 1e-9, _step):
    for b in np.arange(l20[4], u20[4] + 1e-9, _step):
        f = l20.copy(); f[2], f[4] = a, b
        B = bars_cx(cx20, f, 1); D = max(a, b); longest = max((d - s for s, d in B), default=0.0)
        grid20.append(dict(f2=a, f4=b, D=D, bars=bars_txt(B), longest=longest, bound=max(D - 4.0, 11.0 - D),
                           bound_holds=longest >= max(D - 4.0, 11.0 - D) - 1e-12))
grid20 = pd.DataFrame(grid20); grid20.to_csv(RESULTS / "raw" / "example20_grid.csv", index=False)
EX20 = dict(kc_vertices=C20["kc_vertices"], max_edge_degree=C20["max_edge_degree"], image_barcode=bars_txt(C20["I1"]),
            matches_prediction=C20["matches_prediction"], order_types=len(orders20[0]), complete=orders20[1],
            min_count_vs_image={t: list(v) for t, v in ex20_counts.items()},
            count_gap_taus=[t for t, (a, b) in ex20_counts.items() if a > b],
            grid_points=int(len(grid20)), grid_min_longest=float(grid20.longest.min()),
            grid_argmin=grid20.loc[grid20.longest == grid20.longest.min(), ["f2", "f4"]].values.tolist(),
            bound_holds_on_grid=bool(grid20.bound_holds.all()), barcode_attained=C20["attained"])
EX20_VERDICT = ("SUPPORTED" if (not EX20["kc_vertices"] and EX20["matches_prediction"] and EX20["complete"] and EX20["bound_holds_on_grid"]
                                and 3.0 in EX20["count_gap_taus"] and 3.49 in EX20["count_gap_taus"] and 3.5 not in EX20["count_gap_taus"]
                                and abs(EX20["grid_min_longest"] - 3.5) < 1e-12) else "NOT SUPPORTED")
print("Example 20:", EX20, EX20_VERDICT)
set_status("example20", "EXECUTED", EX20_VERDICT)

Example 20: {'kc_vertices': [], 'max_edge_degree': 5, 'image_barcode': '[4,6) [8,11)', 'matches_prediction': True, 'order_types': 6, 'complete': True, 'min_count_vs_image': {2.0: [1, 1], 2.5: [1, 1], 3.0: [1, 0], 3.25: [1, 0], 3.49: [1, 0], 3.5: [0, 0], 4.0: [0, 0]}, 'count_gap_taus': [3.0, 3.25, 3.49], 'grid_points': 425, 'grid_min_longest': 3.5, 'grid_argmin': [[7.5, 7.5]], 'bound_holds_on_grid': True, 'barcode_attained': False} SUPPORTED
[STATUS] example20: EXECUTED — SUPPORTED


## 11 · Census: where degree-1 attainment fails (Table 2)

The census has three designs:
- **random pure 2-complexes**, classified by local structure (a KC vertex; a non-manifold edge but no KC vertex; links that are cycles or paths but not a surface; planar or non-planar surfaces);
- **small triangulations of nine surfaces:** the sphere, the disk and the annulus, which are planar, and the projective plane, the torus, the Klein bottle, the Möbius band, the punctured torus and the punctured Klein bottle;
- **pinched spheres**, with the pinch vertex as the only free vertex. Each is decided by an exact sweep, and the same sphere without the pinch serves as a control.

Each instance is decided in one of three ways:
- by a **verified witness**: $f_D$ or $f_U$, whose barcode is recomputed and compared with the image barcode;
- by **exhaustive enumeration** of the order types;
- by the **exact sweep**, for the pinched spheres.

The second cell gives the breakdown by method. Every "not attained" verdict comes from a complete enumeration or sweep. For each such instance, the exact minimum τ-counts are compared with the image counts.

In [23]:
def count_gap_taus(cx, l, u, I1, cap):
    """tau values (breakpoints and midpoints of the image bar lengths) where the exact minimum count exceeds the image count; None if the
    order-type enumeration is incomplete."""
    orders = realizable_weak_orders(cx.nv, l, u, cap=cap)
    if not orders[1]:
        return None
    ib = [index_barcode(cx, o, 1) for o in orders[0]]
    lens = sorted({d - b for b, d in I1 if math.isfinite(d)} | {0.0})
    taus = sorted({x for a, b in zip(lens, lens[1:] + [lens[-1] + 1.0]) for x in (a, 0.5 * (a + b))})
    return [t for t in taus if exact_min_persistence_count(orders[0], ib, l, u, t) > image_count_pers(I1, t)]

def _grid_tris(m, n, wrap_x, wrap_y, twist_y=False, flip_x=False):
    def vid(i, j):
        if j == n:
            if not wrap_x:
                return None
            j = 0
            if flip_x:
                i = m - 1 - i
        if i == m:
            if not wrap_y:
                return None
            i = 0
            if twist_y:
                j = (n - j) % n
        return i * n + j
    out = []
    for i in range(m if wrap_y else m - 1):
        for j in range(n if wrap_x else n - 1):
            a, b, c, d = vid(i, j), vid(i, j + 1), vid(i + 1, j), vid(i + 1, j + 1)
            if None not in (a, b, c, d):
                out += [(a, b, d), (a, c, d)]
    return out

RP2_6 = [(0, 1, 2), (0, 2, 3), (0, 3, 4), (0, 4, 5), (0, 1, 5), (1, 2, 4), (2, 3, 5), (1, 3, 4), (2, 4, 5), (1, 3, 5)]
RP2_7 = [t for t in RP2_6 if t != (0, 1, 2)] + [(0, 1, 6), (0, 2, 6), (1, 2, 6)]
TORUS_7 = [tuple(sorted((i % 7, (i + 1) % 7, (i + 3) % 7))) for i in range(7)] + [tuple(sorted((i % 7, (i + 2) % 7, (i + 3) % 7))) for i in range(7)]

def _minus(tris, k):
    T = [tuple(sorted(t)) for t in tris]
    return [t for j, t in enumerate(T) if j != k % len(T)]

def _sphere_minus(rng, k, holes):
    g = triangulated_sphere(k, rng); tris = [tuple(sorted(t)) for t in g.triangles.tolist()]; used = set(); removed = []
    for idx in rng.permutation(len(tris)):
        if used.isdisjoint(tris[idx]):
            removed.append(tris[idx]); used |= set(tris[idx])
            if len(removed) == holes:
                return [t for t in tris if t not in set(removed)]
    return None

def small_surface(kind, rng):
    if kind == "sphere":
        return [tuple(t) for t in triangulated_sphere(int(rng.integers(3, 8)), rng).triangles.tolist()]
    if kind == "disk":
        return _sphere_minus(rng, int(rng.integers(4, 8)), 1)
    if kind == "annulus":
        return _sphere_minus(rng, int(rng.integers(6, 9)), 2)
    # the 6-vertex RP^2 and the 7-vertex torus are neighborly: no function on them has a finite H1 bar, so they are not drawn
    if kind == "projective plane":
        return RP2_7
    if kind == "torus":
        return _grid_tris(3, 3, True, True)
    if kind == "Klein bottle":
        return _grid_tris(3, 3, True, True, twist_y=True)
    if kind == "Moebius band":
        return _minus(RP2_6, int(rng.integers(0, 10))) if rng.random() < 0.5 else _grid_tris(3, 3, True, False, flip_x=True)
    if kind == "punctured torus":
        return _minus(TORUS_7, int(rng.integers(0, 14)))
    if kind == "punctured Klein bottle":
        return _minus(_grid_tris(3, 3, True, True, twist_y=True), int(rng.integers(0, 18)))

SMALL_SURFACES = ["sphere", "disk", "annulus", "projective plane", "torus", "Klein bottle", "Moebius band", "punctured torus", "punctured Klein bottle"]

ck_d1x_cen = Checkpoint("v4_degree1_census")
for design in ["random"] + SMALL_SURFACES + ["pinched sphere"]:
    if ck_d1x_cen.done(design):
        continue
    rng = rng_for("d1x_census", design); rows = []; tries = 0; _t0 = time.perf_counter()
    target = CONFIG["d1x_census"] if design == "random" else CONFIG["d1x_pinched"] if design == "pinched sphere" else CONFIG["d1x_small_surfaces"]
    while len(rows) < target and tries < 60 * target:
        tries += 1
        if design == "random":
            nv = int(rng.integers(5, 9))
            cx = random_pure_complex(rng, nv, int(rng.integers(3, 2 * nv + 1)), max_edge_deg=2 if rng.random() < 0.5 else None)
        elif design == "pinched sphere":
            g = triangulated_sphere(int(rng.integers(10, 25)), rng); tris = [tuple(t) for t in g.triangles.tolist()]
            adj = {v: set() for v in range(g.n)}
            for a, b, c in tris:
                adj[a] |= {b, c}; adj[b] |= {a, c}; adj[c] |= {a, b}
            pair = next(((int(v), int(w)) for v, w in (rng.choice(g.n, 2, replace=False) for _ in range(200))
                         if w not in adj[v] and not (adj[v] & adj[w])), None)
            if pair is None:
                continue
            v, w = pair; T = [tuple(sorted(v if x == w else x for x in t)) for t in tris]
            if len(set(T)) != len(T):
                continue
            used = sorted({x for t in T for x in t}); rel = {x: i for i, x in enumerate(used)}
            cx = Small2Complex(len(used), [tuple(rel[x] for x in t) for t in T], "pinched"); x = rel[v]
            sph = Small2Complex(g.n, tris, "sphere")
            vals = rng.permutation(cx.nv).astype(float); l = vals.copy(); u = vals.copy()
            d = float(rng.integers(0, cx.nv - 2)) + 0.5; e = float(rng.integers(int(d) + 1, cx.nv)) + 0.5; l[x], u[x] = d, e
            I1 = image_barcode_rank(cx, l, u, 1)
            if not has_finite_bar(I1):
                continue
            att, c = one_free_vertex_attains(cx, l, u, x, I1, 1)
            # control: the same sphere without the pinch; v keeps the free interval, w is fixed above every other value
            ls, us = np.zeros(g.n), np.zeros(g.n); others = [y for y in range(g.n) if y not in (v, w)]
            ls[others] = us[others] = l[[rel[y] for y in others]]
            ls[v], us[v] = d, e; ls[w] = us[w] = float(cx.nv) + 0.5
            I1s = image_barcode_rank(sph, ls, us, 1)
            att_s = one_free_vertex_attains(sph, ls, us, v, I1s, 1)[0] if has_finite_bar(I1s) else None
            rows.append(dict(design=design, cls="pinched sphere (KC pinch vertex)", nv=cx.nv, ntri=len(cx.tris), status="attained" if att else "NOT attained",
                             how="one-free-vertex sweep", pinch_is_kc=x in kc_vertices(cx), betti=str(betti_z2(cx)),
                             unpinched_status=None if att_s is None else ("attained" if att_s else "NOT attained"), count_gap=None))
            continue
        else:
            tris = small_surface(design, rng)
            if tris is None:
                continue
            used = sorted({v for t in tris for v in t}); rel = {v: i for i, v in enumerate(used)}
            cx = Small2Complex(len(used), [tuple(rel[v] for v in t) for t in tris], design)
        l, u = random_box(rng, cx.nv); I1 = image_barcode_rank(cx, l, u, 1)
        if not has_finite_bar(I1):
            continue
        status, how = decide_degree1(cx, l, u, I1, CONFIG["d1x_order_cap"])
        gaps = count_gap_taus(cx, l, u, I1, CONFIG["d1x_order_cap"]) if status == "NOT attained" else None
        rows.append(dict(design=design, cls=complex_class(cx), nv=cx.nv, ntri=len(cx.tris), status=status, how=how,
                         count_gap=None if gaps is None else bool(gaps), surface_type=design if design != "random" else None,
                         example=json.dumps(dict(tris=[list(t) for t in cx.tris], l=l.tolist(), u=u.tolist())) if status == "NOT attained" else None))
    ck_d1x_cen.add(design, rows)
    print(f"census {design}: {len(rows)} instances ({time.perf_counter() - _t0:.0f}s)")
d1x_cen = ck_d1x_cen.frame()
d1x_cen["not_attained"] = d1x_cen.status == "NOT attained"; d1x_cen["undecided"] = d1x_cen.status == "undecided"
d1x_cen["witness_failed"] = d1x_cen.status == "WITNESS FAILED"
d1x_cls = (d1x_cen.groupby(["design", "cls"]).agg(instances=("nv", "size"), max_vertices=("nv", "max"), proved_not_attained=("not_attained", "sum"),
                                                   undecided=("undecided", "sum"), witness_failed=("witness_failed", "sum"),
                                                   count_gaps=("count_gap", lambda x: int(np.nansum([bool(v) for v in x if v is not None]))))
           .reset_index())
_dorder = {d: i for i, d in enumerate(SMALL_SURFACES + ["random", "pinched sphere"])}
d1x_cls = d1x_cls.sort_values(["design", "cls"], key=lambda c: c.map(_dorder) if c.name == "design" else c).reset_index(drop=True)
d1x_cls["share_not_attained"] = d1x_cls.proved_not_attained / d1x_cls.instances
_planar_designs = {"sphere", "disk", "annulus"}
_planar_rows = d1x_cen[(d1x_cen.design.isin(_planar_designs)) | (d1x_cen.cls == "planar surface")]
_pin = d1x_cen[d1x_cen.design == "pinched sphere"]
CENSUS = dict(instances=int(len(d1x_cen)), planar_instances=int(len(_planar_rows)), planar_not_attained=int(_planar_rows.not_attained.sum()),
              nonplanar_surface_not_attained=int(d1x_cen[(d1x_cen.design.isin(set(SMALL_SURFACES) - _planar_designs)) |
                                                         (d1x_cen.cls == "non-planar surface")].not_attained.sum()),
              kc_not_attained=int(d1x_cen[d1x_cen.cls == "KC vertex"].not_attained.sum()),
              nonmanifold_not_attained=int(d1x_cen[d1x_cen.cls == "non-manifold edge, no KC"].not_attained.sum()),
              undecided=int(d1x_cen.undecided.sum()), witness_failures=int(d1x_cen.witness_failed.sum()),
              count_gaps_total=int(d1x_cls.count_gaps.sum()),
              count_gaps_without_kc=_truthy_count(d1x_cen[(d1x_cen.cls != "KC vertex") & (d1x_cen.design != "pinched sphere")].count_gap),
              pinched=int(len(_pin)), pinched_not_attained=int(_pin.not_attained.sum()), pinch_kc=_truthy_count(_pin.pinch_is_kc),
              unpinched=int(_pin.unpinched_status.notna().sum()), unpinched_not_attained=int((_pin.unpinched_status == "NOT attained").sum()))
print(d1x_cls.round(3).to_string(index=False)); print("census:", CENSUS)

census random: 3000 instances (191s)
census sphere: 400 instances (5s)
census disk: 400 instances (1s)
census annulus: 400 instances (9s)
census projective plane: 400 instances (6s)
census torus: 400 instances (12s)
census Klein bottle: 400 instances (9s)
census Moebius band: 400 instances (5s)
census punctured torus: 400 instances (7s)
census punctured Klein bottle: 400 instances (94s)
census pinched sphere: 200 instances (5s)
                design                                 cls  instances  max_vertices  proved_not_attained  undecided  witness_failed  count_gaps  share_not_attained
                sphere                      planar surface        400             8                    0          0               0           0               0.000
                  disk                      planar surface        400             8                    0          0               0           0               0.000
               annulus                      planar surface        400       

In [24]:
# How each census instance was decided. A witness (f_D or f_U) is accepted only after its barcode is recomputed and found equal to the
# image barcode; "exhaustive" means that every realisable order type of the box was enumerated; pinched spheres use the exact one-free-vertex
# sweep. Every NOT attained verdict comes from a complete enumeration or sweep.
_METHOD = {"f_D": "verified witness f_D", "f_U": "verified witness f_U", "exact": "exhaustive enumeration",
           "one-free-vertex sweep": "one-free-vertex sweep"}
d1x_how = (d1x_cen.assign(method=d1x_cen.how.map(_METHOD).fillna(d1x_cen.how))
           .groupby(["method", "status"]).size().rename("instances").reset_index())
CENSUS_METHODS = dict(instances=int(len(d1x_cen)),
                      witness_fD=int((d1x_cen.how == "f_D").sum()), witness_fU=int((d1x_cen.how == "f_U").sum()),
                      exhaustive_attained=int(((d1x_cen.how == "exact") & (d1x_cen.status == "attained")).sum()),
                      exhaustive_not_attained=int(((d1x_cen.how == "exact") & (d1x_cen.status == "NOT attained")).sum()),
                      sweep_attained=int(((d1x_cen.how == "one-free-vertex sweep") & (d1x_cen.status == "attained")).sum()),
                      sweep_not_attained=int(((d1x_cen.how == "one-free-vertex sweep") & (d1x_cen.status == "NOT attained")).sum()))
CENSUS_METHODS["witness"] = CENSUS_METHODS["witness_fD"] + CENSUS_METHODS["witness_fU"]
CENSUS_METHODS["exhaustive"] = CENSUS_METHODS["exhaustive_attained"] + CENSUS_METHODS["exhaustive_not_attained"]
CENSUS_METHODS["sweep"] = CENSUS_METHODS["sweep_attained"] + CENSUS_METHODS["sweep_not_attained"]
CENSUS_METHODS["reconciles"] = CENSUS_METHODS["witness"] + CENSUS_METHODS["exhaustive"] + CENSUS_METHODS["sweep"] == CENSUS_METHODS["instances"]
d1x_how_design = d1x_cen.assign(method=d1x_cen.how.map(_METHOD).fillna(d1x_cen.how)).pivot_table(
    index="design", columns="method", values="nv", aggfunc="size", fill_value=0)
d1x_how.to_csv(RESULTS / "processed" / "degree1_census_methods.csv", index=False)
d1x_how_design.to_csv(RESULTS / "processed" / "degree1_census_methods_by_design.csv")
print(d1x_how.to_string(index=False)); print(d1x_how_design.to_string()); print("census decision methods:", CENSUS_METHODS)

                method       status  instances
exhaustive enumeration NOT attained        607
exhaustive enumeration     attained        502
 one-free-vertex sweep NOT attained        105
 one-free-vertex sweep     attained         95
  verified witness f_D     attained       5437
  verified witness f_U     attained         54
method                  exhaustive enumeration  one-free-vertex sweep  verified witness f_D  verified witness f_U
design                                                                                                           
Klein bottle                                 0                      0                   400                     0
Moebius band                                 7                      0                   393                     0
annulus                                     37                      0                   363                     0
disk                                         0                      0                   400          

## 12 · Degree-1 verdicts

A verdict reads `SUPPORTED` only if the computations of Sections 7–11 agree with the statements of the paper. Otherwise it reads `NOT SUPPORTED`, and the validation in Section 21 lists it again.

In [25]:
D1X_VERDICT_MACHINERY = "SUPPORTED" if all(D1X_CHECKS[k] == 0 for k in ("deg0_not_attained", "deg0_region_mismatches", "lower_bound_violations",
                                                                         "indep_order_mismatches", "indep_decision_mismatches")) else "NOT SUPPORTED"
KC_LEMMA_VERDICT = ("SUPPORTED" if KC_LEMMA["mv_violations"] == 0 and KC_LEMMA["lemma_violations"] == 0 and KC_LEMMA["positive_control_events"] > 0
                    else "NOT SUPPORTED")
PLANAR_VERDICT = ("SUPPORTED" if PLANAR["theorem_failures"] == 0 and PLANAR["validity_violations"] == 0 and CENSUS["planar_not_attained"] == 0
                  else "NOT SUPPORTED")
_cert_ok = all(C["property_holds"] and C["matches_prediction"] and C["complete"]
               and (C["locally_minimal"] is True or (C["locally_minimal"] is None and not CERT_SPECS[C["name"]].get("check_minimal", True)))
               and C["lower_bound_violations"] == 0
               for C in CERTS.values())
_tr_ok = all(t["attained"] is False and t["complete"] and t["closed_surface"] for t in TRANSFERS)
D1X_VERDICT_CERT = "SUPPORTED" if (_cert_ok and _tr_ok) else "NOT SUPPORTED"
D1X_COUNT_NOTE = (f"count gaps without a KC vertex in the census: {CENSUS['count_gaps_without_kc']}; counts are attained on every surface "
                  f"(Theorem 10), and Example 20 (certificate F) is a KC-free 2-complex whose counts are not attained")
D1X = dict(machinery=D1X_CHECKS, kc_lemma=KC_LEMMA, planar=PLANAR, census=CENSUS,
           certificates={k: dict(attained=C["attained"], n_orders=C["n_orders"], matches_prediction=C["matches_prediction"], locally_minimal=C["locally_minimal"],
                                 pers_gap_taus=C["pers_gap_taus"], quad_gaps=len(C["quad_gaps"]), kc_vertices=C["kc_vertices"], surface=C["surface"])
                         for k, C in CERTS.items()},
           transfers=[dict(surface=t["surface"], attained=t["attained"], n_orders=t["n_orders"]) for t in TRANSFERS])
print("degree 1 verdicts | machinery:", D1X_VERDICT_MACHINERY, "| Lemma 16 (KC):", KC_LEMMA_VERDICT, "| Theorem 8 (planar):", PLANAR_VERDICT,
      "| certificates A-F and transfers (Theorem 11, Examples 17-20):", D1X_VERDICT_CERT, "|", D1X_COUNT_NOTE)
save_json(dict(D1X, certificates_full=CERTS, transfers_full=TRANSFERS), RESULTS / "raw" / "degree1_exact.json")
set_status("degree1_v4", "EXECUTED", f"machinery {D1X_VERDICT_MACHINERY}; planar {PLANAR_VERDICT}; certificates {D1X_VERDICT_CERT}")

degree 1 verdicts | machinery: SUPPORTED | Lemma 16 (KC): SUPPORTED | Theorem 8 (planar): SUPPORTED | certificates A-F and transfers (Theorem 11, Examples 17-20): SUPPORTED | count gaps without a KC vertex in the census: 0; counts are attained on every surface (Theorem 10), and Example 20 (certificate F) is a KC-free 2-complex whose counts are not attained
[STATUS] degree1_v4: EXECUTED — machinery SUPPORTED; planar SUPPORTED; certificates SUPPORTED


## 13 · Theorem 10: finite bars and counts on every compact surface

This section draws random boxes on eighteen surface families, closed and with boundary, planar and not. For each box it checks that:
- $f_D$, computed on the capped surface $\hat\Sigma$, lies in the box;
- $\mathrm{Bar}_1(f_D)$ and the exact image barcode have the same finite bars and the same number of essential bars;
- the two barcodes differ in at most $\hat b$ essential births;
- all τ-counts agree.

It also checks Corollary 14's algorithm against the exact image barcode. Every surface instance that the census proved unattainable is re-checked: its counts must still be attained by $f_D$.

In [26]:
def capped_surface(K, l, u):
    """(S, lS, uS, M): K itself if it is closed, otherwise K with every boundary component coned to a new vertex fixed at M = max u + 3."""
    l = np.asarray(l, float); u = np.asarray(u, float)
    if boundary_cycles(K):
        M = float(u.max()) + 3.0
        S, lS, uS = cone_boundaries(K, l, u, M)
        return S, np.asarray(lS, float), np.asarray(uS, float), M
    return K, l, u, None

def _alive_counts(bars, vals, strict=False):
    """For each threshold c: number of bars (b, d) alive, b <= c < d (or b < c <= d when strict)."""
    if not bars:
        return np.zeros(len(vals), np.int64)
    b = np.sort(np.array([x for x, _ in bars], float)); d = np.sort(np.array([y for _, y in bars], float))
    side = "left" if strict else "right"
    return np.searchsorted(b, vals, side=side) - np.searchsorted(d, vals, side=side)

def essential_births_closed(nv, edges, tris, u, graph):
    """Births of im(H1(S[u <= t]) -> H1(S)) on a closed connected surface S over Z2 (Theorem 10, claim 3; Corollary 14).
    For a proper sublevel complex A_t: dim = beta0(A_t) - chi(A_t) - (beta0(S[u > t]) - 1); at the top it is b1(S) = 2 - chi(S)."""
    u = np.asarray(u, float); vals = np.unique(u)
    b0 = _alive_counts(barcode0(graph, u), vals)
    b0c = _alive_counts(barcode0(graph, -u), -vals, strict=True)             # components of {-u < -t} = {u > t}
    E = np.asarray(edges, np.int64).reshape(-1, 2); T = np.asarray(tris, np.int64).reshape(-1, 3)
    ev = np.sort(np.maximum(u[E[:, 0]], u[E[:, 1]])); tv = np.sort(u[T].max(1)); vv = np.sort(u)
    chi = np.searchsorted(vv, vals, "right") - np.searchsorted(ev, vals, "right") + np.searchsorted(tv, vals, "right")
    dim = b0 - chi - (b0c - 1)
    dim[-1] = 2 - (nv - len(E) + len(T))
    inc = np.diff(np.concatenate([[0], dim]))
    if np.any(inc < 0):
        raise ValueError("essential dimension decreased: the input is not a closed connected surface")
    return np.repeat(vals, inc).tolist()

def degree1_image_barcode_arrays(nv, edges, tris, ell, u, M=None, closed_surface=True, graph=None, witness=False):
    """Corollary 14: Bar_1(u >= ell) on a closed connected surface given by arrays, in O(n log n + m alpha(n)).
    The finite part is the reflected degree-0 image barcode of the box (-u, -ell); the global births come from essential_births_closed.
    If the surface is a capped one whose cone vertices are fixed at M, bars ending at M are reported as essential (they are essential on the
    uncapped surface). With witness=True also returns f_D = -f_U(-u, -ell)."""
    ell = np.asarray(ell, float); u = np.asarray(u, float)
    g = graph if graph is not None else graph_from_edges(nv, np.asarray(edges, np.int64).reshape(-1, 2), "S")
    cert = certify_graph(g, -u, -ell, 0.0)
    bars = [(-d, (math.inf if (M is not None and -b == M) else -b)) for b, d in image_bars(cert) if math.isfinite(d)]
    if closed_surface:
        bars += [(t, math.inf) for t in essential_births_closed(nv, edges, tris, u, g)]
    out = sorted(bars)
    return (out, -universal_realizer(cert, -u)) if witness else out

def degree1_image_barcode_surface(K, l, u, witness=False):
    S, lS, uS, M = capped_surface(K, l, u)
    res = degree1_image_barcode_arrays(S.nv, S.edges, S.tris, lS, uS, M, witness=witness)
    return (res[0], res[1][:K.nv]) if witness else res

def _fin(B):
    return sorted((float(b), float(d)) for b, d in B if math.isfinite(d))

def _ess(B):
    return sorted(float(b) for b, d in B if not math.isfinite(d))

def theorem_c_check(K, l, u):
    """All claims of Theorem 10 and Corollary 14 on one instance (exact reference: the rank invariant)."""
    l = np.asarray(l, float); u = np.asarray(u, float)
    I1 = sorted(image_barcode_rank(K, l, u, 1)); S = capped_surface(K, l, u)[0]; bhat = betti_z2(S)[1]
    fast, fD = degree1_image_barcode_surface(K, l, u, witness=True); B = bars_cx(K, fD, 1)
    lens = sorted({d - b for b, d in _fin(I1) + _fin(B)} | {0.0})
    taus = sorted({x + e for x in lens for e in (-0.25, 0.0, 0.25) if x + e >= 0})
    mism = sum((collections.Counter(_ess(B)) - collections.Counter(_ess(I1))).values())
    return dict(bhat=int(bhat), closed=not boundary_cycles(K), bars=len(I1), finite_bars=len(_fin(I1)),
                in_box=bool(np.all(fD >= l - 1e-12) and np.all(fD <= u + 1e-12)), finite_equal=_fin(B) == _fin(I1),
                essential_count_equal=len(_ess(B)) == len(_ess(I1)), essential_mismatch=int(mism), mismatch_within_bhat=bool(mism <= bhat),
                counts_equal=all(count_bars(B, t) == count_bars(I1, t) for t in taus), barcode_equal=B == I1, fast_equal_exact=fast == I1)

def _glue_any(P, Q):
    """P and Q (each with one boundary 3-cycle) glued along their boundaries: a closed surface for random boxes on all vertices."""
    bp = boundary_cycles(P)[0]; bq = boundary_cycles(Q)[0]
    mp = dict(zip(bq, bp)); nv = P.nv
    for v in range(Q.nv):
        if v not in mp:
            mp[v] = nv; nv += 1
    return Small2Complex(nv, [tuple(t) for t in P.tris] + [tuple(sorted(mp[v] for v in t)) for t in Q.tris], "glued")

def _cx(tris):
    if not tris:
        return None
    used = sorted({v for t in tris for v in t}); rel = {v: i for i, v in enumerate(used)}
    return Small2Complex(len(used), [tuple(rel[v] for v in t) for t in tris], "surface")

_Dcx = _make(CERT_SPECS["D"]["tris"], CERT_SPECS["D"]["l"], CERT_SPECS["D"]["u"])[0]
_Ecx = _make(CERT_SPECS["E"]["tris"], CERT_SPECS["E"]["l"], CERT_SPECS["E"]["u"])[0]
_T4 = [tuple(sorted(t)) for t in _grid_tris(4, 4, True, True)]
THMC_FAMILIES = {
    "RP^2 (7 vertices)": lambda r: _cx(RP2_7),
    "torus 3x3": lambda r: _cx(_grid_tris(3, 3, True, True)), "torus 4x4": lambda r: _cx(_grid_tris(4, 4, True, True)),
    "torus 3x5": lambda r: _cx(_grid_tris(3, 5, True, True)),
    "Klein bottle 3x3": lambda r: _cx(_grid_tris(3, 3, True, True, twist_y=True)),
    "Klein bottle 4x4": lambda r: _cx(_grid_tris(4, 4, True, True, twist_y=True)),
    "Klein bottle (D+D)": lambda r: _glue_any(_Dcx, _Dcx), "genus 2 (E+E)": lambda r: _glue_any(_Ecx, _Ecx),
    "N3 (D+E)": lambda r: _glue_any(_Dcx, _Ecx),
    "Moebius band (6 v.)": lambda r: _cx(_minus(RP2_6, int(r.integers(0, 10)))),
    "Moebius band (grid)": lambda r: _cx(_grid_tris(3, 3, True, False, flip_x=True)),
    "punctured torus (7 v.)": lambda r: _cx(_minus(TORUS_7, int(r.integers(0, 14)))),
    "punctured torus 4x4": lambda r: _cx(_minus(_T4, int(r.integers(0, 32)))),
    "punctured Klein bottle": lambda r: _cx(_minus(_grid_tris(3, 3, True, True, twist_y=True), int(r.integers(0, 18)))),
    "torus with two holes": lambda r: _cx([t for k, t in enumerate(sorted(_T4)) if k not in (0, 17)]),
    "disk": lambda r: _cx(_sphere_minus(r, int(r.integers(5, 12)), 1)),
    "annulus": lambda r: _cx(_sphere_minus(r, int(r.integers(7, 12)), 2)),
    "sphere": lambda r: _cx([tuple(t) for t in triangulated_sphere(int(r.integers(4, 12)), r).triangles.tolist()]),
}

def _thmc_box(rng, n):
    if rng.random() < 0.7:                                    # integer bounds with heavy ties (as in the census)
        return random_box(rng, n)
    l = rng.random(n) * 10.0                                  # continuous bounds, some vertices fixed
    return l, l + rng.random(n) * 3.0 * (rng.random(n) < 0.6)

ck_thmc = Checkpoint("v5_theorem_c")
for fam, gen in THMC_FAMILIES.items():
    if ck_thmc.done(fam):
        continue
    rng = rng_for("v5_thmc", fam); rows = []
    for i in range(CONFIG["d1x_thmc"]):
        K = gen(rng)
        if K is None or not is_surface(K):
            continue
        l, u = _thmc_box(rng, K.nv)
        rows.append(dict(family=fam, i=i, nv=K.nv, ntri=len(K.tris), planar=bool(surface_planar(K)), **theorem_c_check(K, l, u)))
    ck_thmc.add(fam, rows)
thmc = ck_thmc.frame()
thmc_s = (thmc.groupby("family", sort=False).agg(instances=("nv", "size"), max_vertices=("nv", "max"), bhat=("bhat", "max"), closed=("closed", "max"),
                                                 in_box=("in_box", "mean"), finite_equal=("finite_equal", "mean"),
                                                 essential_count_equal=("essential_count_equal", "mean"), max_mismatch=("essential_mismatch", "max"),
                                                 counts_equal=("counts_equal", "mean"), fast_equal_exact=("fast_equal_exact", "mean"),
                                                 barcode_attained_by_fD=("barcode_equal", "mean")).reset_index())

# census re-check: surface instances proved NOT attained in Section 11 must still have their counts attained by f_D
_surface_designs = set(SMALL_SURFACES)
_na = d1x_cen[d1x_cen.not_attained & (d1x_cen.design.isin(_surface_designs) | d1x_cen.cls.isin(["planar surface", "non-planar surface"]))]
recheck = []
for ex in _na.example.dropna():
    E_ = json.loads(ex); K = Small2Complex(max(v for t in E_["tris"] for v in t) + 1, [tuple(t) for t in E_["tris"]], "census")
    recheck.append(theorem_c_check(K, np.array(E_["l"]), np.array(E_["u"])))
recheck = pd.DataFrame(recheck)
THMC = dict(instances=int(len(thmc)), families=int(thmc.family.nunique()), max_vertices=int(thmc.nv.max()),
            failures=int((~(thmc.in_box & thmc.finite_equal & thmc.essential_count_equal & thmc.mismatch_within_bhat & thmc.counts_equal)).sum()),
            fast_mismatches=int((~thmc.fast_equal_exact.astype(bool)).sum()),
            planar_barcode_failures=int((thmc.planar & ~thmc.barcode_equal.astype(bool)).sum()),
            nonplanar_barcode_defects=int((~thmc.planar & ~thmc.barcode_equal.astype(bool)).sum()),
            census_not_attained_rechecked=int(len(recheck)),
            census_recheck_failures=int((~(recheck.counts_equal & recheck.finite_equal & recheck.mismatch_within_bhat)).sum()) if len(recheck) else 0)
THMC_VERDICT = ("SUPPORTED" if THMC["failures"] == 0 and THMC["fast_mismatches"] == 0 and THMC["planar_barcode_failures"] == 0
                and THMC["census_recheck_failures"] == 0 else "NOT SUPPORTED")
thmc.to_csv(RESULTS / "raw" / "v5_theorem_c.csv", index=False)
print(thmc_s.round(3).to_string(index=False)); print("Theorem 10 (surfaces):", THMC, THMC_VERDICT)
set_status("theorem_c", "EXECUTED", THMC_VERDICT)

                family  instances  max_vertices  bhat  closed  in_box  finite_equal  essential_count_equal  max_mismatch  counts_equal  fast_equal_exact  barcode_attained_by_fD
     RP^2 (7 vertices)        400             7     1    True     1.0           1.0                    1.0             1           1.0               1.0                   0.930
             torus 3x3        400             9     2    True     1.0           1.0                    1.0             2           1.0               1.0                   0.915
             torus 4x4        400            16     2    True     1.0           1.0                    1.0             2           1.0               1.0                   0.982
             torus 3x5        400            15     2    True     1.0           1.0                    1.0             2           1.0               1.0                   0.940
      Klein bottle 3x3        400             9     2    True     1.0           1.0                    1.0         

## 14 · Proposition 15: graphs

On $K_5$, $K_{3,3}$ and random graphs, all without triangles, the exact degree-1 image barcode must equal $\mathrm{Bar}_1(u)$, so $f=u$ attains it. The test graphs include non-planar ones.

In [27]:
ck_pg = Checkpoint("v5_prop_g")
if not ck_pg.done("all"):
    rng = rng_for("v5_prop_g"); rows = []
    for i in range(CONFIG["d1x_graphs"]):
        kind = ("K5", "K3,3", "random")[i % 3]
        if kind == "K5":
            n, E = 5, list(itertools.combinations(range(5), 2))
        elif kind == "K3,3":
            n, E = 6, [(a, b) for a in range(3) for b in range(3, 6)]
        else:
            n = int(rng.integers(5, 11)); p = float(rng.uniform(0.3, 0.8))
            E = [(a, b) for a, b in itertools.combinations(range(n), 2) if rng.random() < p]
        K = Small2Complex(n, [], kind, edges=E); l, u = random_box(rng, n)
        I1 = sorted(image_barcode_rank(K, l, u, 1)); Bu = bars_cx(K, u, 1)
        rows.append(dict(kind=kind, nv=n, edges=len(E), bars=len(I1), attained_by_u=bool(Bu == I1)))
    ck_pg.add("all", rows)
pg = ck_pg.frame()
pg_s = pg.groupby("kind").agg(instances=("nv", "size"), mean_bars=("bars", "mean"), attained_by_u=("attained_by_u", "mean")).reset_index()
PROPG = dict(instances=int(len(pg)), failures=int((~pg.attained_by_u.astype(bool)).sum()),
             nonplanar_instances=int(pg.kind.isin(["K5", "K3,3"]).sum()), with_bars=int((pg.bars > 0).sum()))
PROPG_VERDICT = "SUPPORTED" if PROPG["failures"] == 0 and PROPG["nonplanar_instances"] > 0 else "NOT SUPPORTED"
print(pg_s.round(3).to_string(index=False)); print("Proposition 15 (graphs):", PROPG, PROPG_VERDICT)
set_status("prop_g", "EXECUTED", PROPG_VERDICT)

  kind  instances  mean_bars  attained_by_u
  K3,3        200      4.000            1.0
    K5        200      6.000            1.0
random        200      8.035            1.0
Proposition 15 (graphs): {'instances': 600, 'failures': 0, 'nonplanar_instances': 400, 'with_bars': 587} SUPPORTED
[STATUS] prop_g: EXECUTED — SUPPORTED


## 15 · Corollary 14: near-linear degree-1 certificates (Figure 3b, Figure 4b)

The algorithm needs one degree-0 image sweep of the dual box and two union–find sweeps. It runs on two families:
- **terrains:** $H\times W$ Freudenthal grids whose boundary is coned to a vertex at $M$, so $\hat b=0$;
- **tori:** periodic grids, with $\hat b=2$.

Boxes are a Gaussian random field plus or minus a heteroscedastic half-width. There are three checks:
1. exact agreement with the rank-invariant computation on small grids, with both timed;
2. agreement of the witness $f_D$ at moderate size;
3. wall time at scale. These timings illustrate the proved bound on this machine.

In [28]:
def terrain_arrays(H, W):
    """Freudenthal H x W grid with its boundary cycle coned to vertex H*W: (nv, edges, triangles) of a triangulated sphere."""
    g = graph_grid(H, W, 6); n = H * W
    ring = ([(0, j) for j in range(W)] + [(i, W - 1) for i in range(1, H)] + [(H - 1, j) for j in range(W - 2, -1, -1)]
            + [(i, 0) for i in range(H - 2, 0, -1)])
    ring = np.array([i * W + j for i, j in ring], np.int64)
    src = np.repeat(np.arange(n), np.diff(g.indptr)); dst = g.indices; m = src < dst
    E = np.concatenate([np.stack([src[m], dst[m]], 1), np.stack([np.full(len(ring), n), ring], 1)])
    T = np.concatenate([g.triangles, np.stack([np.full(len(ring), n), ring, np.roll(ring, -1)], 1)])
    return n + 1, E, T

def torus_arrays(H, W):
    """Periodic Freudenthal H x W grid (H, W >= 3): (nv, edges, triangles) of a triangulated torus."""
    idx = np.arange(H * W, dtype=np.int64).reshape(H, W)
    a = idx; b = np.roll(idx, -1, 1); c = np.roll(idx, -1, 0); d = np.roll(b, -1, 0)
    E = np.concatenate([np.stack([a.ravel(), b.ravel()], 1), np.stack([a.ravel(), c.ravel()], 1), np.stack([a.ravel(), d.ravel()], 1)])
    T = np.concatenate([np.stack([a.ravel(), b.ravel(), d.ravel()], 1), np.stack([a.ravel(), c.ravel(), d.ravel()], 1)])
    return H * W, E, T

def d1f_instance(kind, side, rng, corr):
    """Arrays and box for one instance. Terrains carry the cone vertex at M; the returned (l, u) are on the original vertices."""
    H = W = side; n = H * W
    f = grf_batch(rng, 1, H, W, corr)[0].ravel(); w = CONFIG["d1f_halfwidth"] * (1.0 + 3.0 * rng.random(n))
    l, u = f - w, f + w
    if kind == "terrain":
        nv, E, T = terrain_arrays(H, W); M = float(u.max()) + 3.0
        return dict(nv=nv, E=E, T=T, L=np.append(l, M), U=np.append(u, M), M=M, l=l, u=u, n=n)
    nv, E, T = torus_arrays(H, W)
    return dict(nv=nv, E=E, T=T, L=l, U=u, M=None, l=l, u=u, n=n)

ck_d1f = Checkpoint("v5_degree1_fast")
if not ck_d1f.done("exact"):
    rng = rng_for("v5_d1f_exact"); rows = []
    for kind in ("terrain", "torus"):
        for side in CONFIG["d1f_exact_sides"]:
            for rep in range(CONFIG["d1f_exact_reps"]):
                I = d1f_instance(kind, side, rng, 0.8)
                t0 = time.perf_counter(); fast = degree1_image_barcode_arrays(I["nv"], I["E"], I["T"], I["L"], I["U"], I["M"]); t1 = time.perf_counter()
                tris = (graph_grid(side, side, 6).triangles if kind == "terrain" else I["T"]).tolist()
                K = Small2Complex(I["n"], [tuple(t) for t in tris], kind)
                t2 = time.perf_counter(); exact = sorted(image_barcode_rank(K, I["l"], I["u"], 1)); t3 = time.perf_counter()
                rows.append(dict(kind=kind, side=side, n=I["n"], rep=rep, bars=len(exact), agree=fast == exact, t_fast=t1 - t0, t_exact=t3 - t2))
    ck_d1f.add("exact", rows)
if not ck_d1f.done("witness"):
    rng = rng_for("v5_d1f_witness"); rows = []
    for kind in ("terrain", "torus"):
        for side in CONFIG["d1f_witness_sides"]:
            I = d1f_instance(kind, side, rng, CONFIG["d1f_corr"])
            fast, fD = degree1_image_barcode_arrays(I["nv"], I["E"], I["T"], I["L"], I["U"], I["M"], witness=True)
            fK = fD[:I["n"]]
            tris = (graph_grid(side, side, 6).triangles if kind == "terrain" else I["T"]).tolist()
            t0 = time.perf_counter(); B = sorted((float(b), float(d)) for b, d in lowerstar_diagrams(I["n"], tris, fK)[1] if d > b)
            t1 = time.perf_counter()
            mism = sum((collections.Counter(_ess(B)) - collections.Counter(_ess(fast))).values())
            ok = (B == fast) if kind == "terrain" else (_fin(B) == _fin(fast) and len(_ess(B)) == len(_ess(fast)) and mism <= 2)
            rows.append(dict(kind=kind, side=side, n=I["n"], bars=len(fast), finite_bars=len(_fin(fast)), in_box=bool(np.all(fK >= I["l"] - 1e-12)
                             and np.all(fK <= I["u"] + 1e-12)), agree=bool(ok), barcode_equal=B == fast, t_lowerstar=t1 - t0))
    ck_d1f.add("witness", rows)
for kind in ("terrain", "torus"):
    for side in CONFIG["d1f_sides"]:
        key = f"time|{kind}|{side}"
        if ck_d1f.done(key):
            continue
        rng = rng_for("v5_d1f_time", kind, side); rows = []
        for rep in range(CONFIG["d1f_reps"]):
            t0 = time.perf_counter(); I = d1f_instance(kind, side, rng, CONFIG["d1f_corr"])
            g = graph_from_edges(I["nv"], I["E"], kind); t1 = time.perf_counter()
            cert = certify_graph(g, -I["U"], -I["L"], 0.0)
            fin = [(-d, -b) for b, d in image_bars(cert) if math.isfinite(d)]; t2 = time.perf_counter()
            ess_b = essential_births_closed(I["nv"], I["E"], I["T"], I["U"], g); t3 = time.perf_counter()
            fD = -universal_realizer(cert, -I["U"]); t4 = time.perf_counter()
            rows.append(dict(kind=kind, side=side, n=I["n"], rep=rep, t_build=t1 - t0, t_finite=t2 - t1, t_global=t3 - t2, t_witness=t4 - t3,
                             finite_bars=len(fin), global_bars=len(ess_b),
                             in_box=bool(np.all(fD[:I["n"]] >= I["l"] - 1e-12) and np.all(fD[:I["n"]] <= I["u"] + 1e-12))))
            del I, g, cert, fD; gc.collect()
        ck_d1f.add(key, rows)
        print(f"degree-1 certificate, {kind} n={side * side}: done", flush=True)
_d = ck_d1f.frame()
d1f_exact = _d[_d.t_exact.notna()].reset_index(drop=True) if "t_exact" in _d else pd.DataFrame()
d1f_wit = _d[_d.t_lowerstar.notna()].reset_index(drop=True) if "t_lowerstar" in _d else pd.DataFrame()
d1f_time = _d[_d.t_finite.notna()].reset_index(drop=True) if "t_finite" in _d else pd.DataFrame()
d1f_ts = (d1f_time.assign(t_total=d1f_time.t_finite + d1f_time.t_global)
          .groupby(["kind", "n"]).agg(t_build=("t_build", "median"), t_finite=("t_finite", "median"), t_global=("t_global", "median"),
                                      t_total=("t_total", "median"), t_witness=("t_witness", "median"), finite_bars=("finite_bars", "median"),
                                      global_bars=("global_bars", "max"), in_box=("in_box", "mean"), reps=("rep", "size")).reset_index())
_sl = {k: loglog_slope(d.n, d.t_total) if len(d) >= 2 else float("nan") for k, d in d1f_ts.groupby("kind")}
D1FAST = dict(exact_checks=int(len(d1f_exact)), exact_agree=int(d1f_exact.agree.astype(bool).sum()), exact_max_n=int(d1f_exact.n.max()),
              exact_max_seconds=float(d1f_exact.t_exact.max()), fast_seconds_at_exact_max=float(d1f_exact.t_fast.max()),
              witness_checks=int(len(d1f_wit)), witness_agree=int(d1f_wit.agree.astype(bool).sum()), witness_in_box=int(d1f_wit.in_box.astype(bool).sum()),
              witness_max_n=int(d1f_wit.n.max()), max_n=int(d1f_ts.n.max()),
              seconds_at_max_n={k: float(d[d.n == d.n.max()].t_total.iloc[0]) for k, d in d1f_ts.groupby("kind")},
              slope_terrain=float(_sl.get("terrain", float("nan"))), slope_torus=float(_sl.get("torus", float("nan"))),
              timing_in_box=bool(d1f_ts.in_box.min() == 1.0), torus_global_bars=int(d1f_ts[d1f_ts.kind == "torus"].global_bars.max()))
D1FAST_VERDICT = ("SUPPORTED" if D1FAST["exact_agree"] == D1FAST["exact_checks"] > 0 and D1FAST["witness_agree"] == D1FAST["witness_checks"] > 0
                  and D1FAST["witness_in_box"] == D1FAST["witness_checks"] and D1FAST["timing_in_box"] else "NOT SUPPORTED")
_d.to_csv(RESULTS / "raw" / "v5_degree1_fast.csv", index=False); d1f_ts.to_csv(RESULTS / "processed" / "v5_degree1_fast_timing.csv", index=False)
print(d1f_exact.groupby(["kind", "n"]).agg(agree=("agree", "mean"), t_fast=("t_fast", "median"), t_exact=("t_exact", "median"), bars=("bars", "mean"))
      .reset_index().round(4).to_string(index=False))
print(d1f_wit[["kind", "n", "bars", "finite_bars", "agree", "barcode_equal", "in_box", "t_lowerstar"]].round(3).to_string(index=False))
print(d1f_ts.round(4).to_string(index=False))
print("Corollary 14 (near-linear degree 1):", D1FAST, D1FAST_VERDICT)
set_status("degree1_fast", "EXECUTED", D1FAST_VERDICT)

degree-1 certificate, terrain n=10000: done
degree-1 certificate, terrain n=40000: done
degree-1 certificate, terrain n=160000: done
degree-1 certificate, terrain n=640000: done
degree-1 certificate, terrain n=2560000: done
degree-1 certificate, torus n=10000: done
degree-1 certificate, torus n=40000: done
degree-1 certificate, torus n=160000: done
degree-1 certificate, torus n=640000: done
degree-1 certificate, torus n=2560000: done
   kind   n agree  t_fast  t_exact   bars
terrain  16   1.0  0.0003   0.0053 0.0000
terrain  25   1.0  0.0003   0.0224 0.6667
terrain  36   1.0  0.0004   0.0793 0.3333
terrain  49   1.0  0.0004   0.1665 0.6667
terrain  64   1.0  0.0005   0.4050 1.3333
terrain  81   1.0  0.0005   0.9134 2.3333
terrain 100   1.0  0.0005   1.9388 2.6667
  torus  16   1.0  0.0003   0.0097 2.0000
  torus  25   1.0  0.0003   0.0367 2.0000
  torus  36   1.0  0.0004   0.1092 2.6667
  torus  49   1.0  0.0004   0.2953 2.3333
  torus  64   1.0  0.0005   0.6162 3.3333
  torus  81   1.

## 16 · Theorem 13 and Lemma 12: deciding attainment on surfaces

All boxes are integer: $\ell$ is uniform in $\{0,\dots,11\}$ and $u-\ell$ is uniform in $\{0,\dots,5\}$ at every vertex. The surfaces are the six-vertex projective plane, the Möbius band obtained by deleting one of its triangles, the seven-vertex torus, and the punctured torus obtained by deleting one of its triangles. There are three checks.
- **Decision** (Theorem 13). Compute $f_D$, on $\hat\Sigma$ when the surface has boundary. If $f_D$ fails, try every peak function $f_p$ on closed surfaces, and decide by enumerating all order types. The claim to check: with boundary, the bound is attained if and only if $f_D$ attains it; on closed surfaces, if and only if $f_D$ or some $f_p$ attains it.
- **Maximality, with boundary** (Lemma 12). For every order type that attains the bound, take its *near-supremum representative*: the block values are as large as the box allows, each a symbolic ε below the next block, realised with a concrete small ε. The supremum of the order cell may itself lie outside the strict order type, which is why the representative is taken ε below it. The check is that every such representative lies below $f_D$.
- **Maximality, closed surfaces** (Lemma 12 and Step 1 of the proof of Theorem 13). For each near-supremum representative $g$, check that (i) $g\le f_D$ wherever $g\le\max\ell$, and (ii) $\{g>t\}$ is connected or empty for $t\ge\max\ell$. When $\max g>\max\ell$, also check (iii) $g\le f_p$ for a vertex $p$ maximising $g$.

The **reference runs** use the surfaces, box counts and seeds that the paper cites, and the FULL profile must reproduce their logged counts exactly. The **extended runs** use ten times as many boxes and fresh seeds. The QUICK profile uses the reference seeds with fewer boxes.

The code below runs in its own namespace. It reimplements the $\mathbb Z_2$ algebra, the persistence, the image barcode, $f_D$ and the enumeration, so this check is independent of Sections 4–15.

In [29]:
# The checks of Theorem 13 and Lemma 12 run in a separate namespace with their own implementations of every ingredient (GF(2)
# algebra, lower-star persistence, the degree-1 image barcode, f_D, order-type enumeration). They share no code with Sections 4-15.
T13_SRC = r'''
# Self-contained checks of Theorem 13 (deciding attainment on surfaces) and Lemma 12 (maximality), degree 1.
# Independent of the rest of the notebook: own GF(2) algebra, own lower-star persistence, own image barcode, own enumeration.
import itertools, math, sys, collections
import numpy as np

# ---------- GF(2) linear algebra on Python-int bitsets ----------
def rank_gf2(rows):
    rows = [r for r in rows if r]; rank = 0; piv = {}
    for r in rows:
        while r:
            h = r.bit_length() - 1
            if h in piv: r ^= piv[h]
            else: piv[h] = r; rank += 1; break
    return rank

def kernel_gf2(cols, nrows):
    # columns: list of int bitsets (vectors in GF(2)^nrows); return basis of kernel as bitsets over column indices
    n = len(cols); piv = {}; basis = []
    for j, c in enumerate(cols):
        comb = 1 << j; v = c
        while v:
            h = v.bit_length() - 1
            if h in piv: pv, pc = piv[h]; v ^= pv; comb ^= pc
            else: piv[h] = (v, comb); break
        if v == 0: basis.append(comb)
    return basis

# ---------- complexes ----------
class Cx:
    def __init__(self, nv, tris):
        self.nv = nv; self.tris = [tuple(sorted(t)) for t in tris]
        es = set()
        for a, b, c in self.tris: es |= {(a, b), (a, c), (b, c)}
        self.edges = sorted(es); self.eidx = {e: i for i, e in enumerate(self.edges)}
        self.nbr = collections.defaultdict(set)
        for a, b in self.edges: self.nbr[a].add(b); self.nbr[b].add(a)

def boundary_edges(cx):
    cnt = collections.Counter()
    for a, b, c in cx.tris: cnt.update([(a, b), (a, c), (b, c)])
    return [e for e, k in cnt.items() if k == 1]

def boundary_components(cx):
    be = boundary_edges(cx); adj = collections.defaultdict(set)
    for a, b in be: adj[a].add(b); adj[b].add(a)
    seen = set(); comps = []
    for v in adj:
        if v in seen: continue
        stack = [v]; comp = []
        while stack:
            x = stack.pop()
            if x in seen: continue
            seen.add(x); comp.append(x); stack.extend(adj[x] - seen)
        comps.append((sorted(comp), [e for e in be if e[0] in comp]))
    return comps

def cap(cx, l, u, M):
    tris = list(cx.tris); l2 = list(l); u2 = list(u); nv = cx.nv
    for comp, es in boundary_components(cx):
        c = nv; nv += 1; l2.append(M); u2.append(M)
        tris += [(c, a, b) for a, b in es]
    return Cx(nv, tris), np.array(l2, float), np.array(u2, float)

# ---------- lower-star persistence over Z2 (ties by (f, id)) ----------
def lowerstar(nv, tris, f):
    cx = Cx(nv, tris); key = {v: (f[v], v) for v in range(nv)}
    simp = [((v,), 0) for v in range(nv)] + [(e, 1) for e in cx.edges] + [(t, 2) for t in cx.tris]
    def k(s): return (max(key[v] for v in s[0]), s[1], sorted((key[v] for v in s[0]), reverse=True))
    simp.sort(key=k); pos = {s[0]: i for i, s in enumerate(simp)}
    val = [f[max(s[0], key=lambda v: key[v])] for s in simp]
    cols = []
    for s, d in simp:
        if d == 0: cols.append(0)
        else: cols.append(sum(1 << pos[fc] for fc in itertools.combinations(s, d)))
    low_of = {}; red = list(cols); pairs = {0: [], 1: []}; paired = set()
    for j in range(len(simp)):
        c = red[j]
        while c and (c.bit_length() - 1) in low_of: c ^= red[low_of[c.bit_length() - 1]]
        red[j] = c
        if c:
            i = c.bit_length() - 1; low_of[i] = j; paired |= {i, j}
            dg = simp[i][1]; pairs[dg].append((float(val[i]), float(val[j])))
    for i, (s, d) in enumerate(simp):
        if i not in paired and d <= 1: pairs[d].append((float(val[i]), math.inf))
    return [pairs[0], pairs[1]]

def bars(cx, f, deg):
    return sorted((b, d) for b, d in lowerstar(cx.nv, cx.tris, f)[deg] if d > b)

# ---------- image barcode in degree 1 via the rank invariant ----------
def sub(cx, g, t):
    S = {v for v in range(cx.nv) if g[v] <= t}
    E = [cx.eidx[e] for e in cx.edges if e[0] in S and e[1] in S]
    T = [tr for tr in cx.tris if all(v in S for v in tr)]
    return S, E, T

def rank_map(cx, A, B):
    _, EA, _ = A; _, EB, TB = B
    cols = [(1 << cx.edges[i][0]) | (1 << cx.edges[i][1]) for i in EA]
    Z = [sum(1 << EA[j] for j in range(len(EA)) if comb >> j & 1) for comb in kernel_gf2(cols, cx.nv)]
    Bd = [sum(1 << cx.eidx[e] for e in itertools.combinations(t, 2)) for t in TB]
    return rank_gf2(Z + Bd) - rank_gf2(Bd)

def image_barcode1(cx, l, u):
    C = sorted(set(map(float, l)) | set(map(float, u))); k = len(C)
    subsU = [sub(cx, u, c) for c in C]; subsL = [sub(cx, l, c) for c in C]
    r = {}
    def R(i, j):
        if i < 0: return 0
        if (i, j) not in r: r[(i, j)] = rank_map(cx, subsU[i], subsL[j])
        return r[(i, j)]
    out = []
    for i in range(k):
        for j in range(i + 1, k + 1):
            if j == k: m = R(i, k - 1) - R(i - 1, k - 1); d = math.inf
            else: m = R(i, j - 1) - R(i - 1, j - 1) - R(i, j) + R(i - 1, j); d = C[j]
            out += [(C[i], d)] * m
    return sorted(out)

# ---------- reconstruction and the dual realiser ----------
def recon_erosion(cx, lo, mk):
    # R^eps_lo(mk): lower level sets = components of {lo<=t} meeting {mk<=t}
    d = np.array(mk, float).copy()
    changed = True
    while changed:
        changed = False
        for v in range(cx.nv):
            m = min([d[v]] + [d[w] for w in cx.nbr[v]])
            nvl = max(lo[v], m)
            if nvl < d[v]: d[v] = nvl; changed = True
    return d

def f_D(cx, l, u):
    return -recon_erosion(cx, -np.asarray(u, float), -np.asarray(l, float))

# ---------- exact attainment (copied from the reviewed d1x_core functions) ----------
def realizable_weak_orders(nv, l, u, cap=None):
    out = []; complete = [True]
    def rec(rem, blocks, prev):
        if cap is not None and len(out) >= cap: complete[0] = False; return
        if not rem: out.append(tuple(blocks)); return
        rl = sorted(rem)
        for r_ in range(1, len(rl) + 1):
            for S in itertools.combinations(rl, r_):
                L = max(l[v] for v in S); U = min(u[v] for v in S)
                if L > U: continue
                x = (L, 0) if prev is None else max((L, 0), (prev[0], prev[1] + 1))
                if x[0] > U or (x[0] == U and x[1] > 0): continue
                rest = rem - set(S)
                if any(u[w] <= x[0] for w in rest): continue
                rec(rest, blocks + [S], x)
                if cap is not None and len(out) >= cap: complete[0] = False; return
    rec(frozenset(range(nv)), [], None)
    return out, complete[0]

def _fix_values(ibars, I):
    if len(ibars) != len(I): return None
    fixed = {}
    for (i, j), (b, d) in zip(sorted(ibars), sorted(I)):
        for idx, v in ((i, b), (j, d)):
            if math.isinf(idx) or math.isinf(v):
                if math.isinf(idx) != math.isinf(v): return None
                continue
            if fixed.setdefault(int(idx), v) != v: return None
    return fixed

def realise_with_fixed(blocks, l, u, fixed):
    m = len(blocks); prev = None; sym = []
    for j, S in enumerate(blocks):
        L = max(l[v] for v in S); U = min(u[v] for v in S)
        if j in fixed:
            c = fixed[j]
            if not (L <= c <= U) or (prev is not None and not ((c, 0) > prev)): return False, None
            x = (c, 0)
        else:
            x = (L, 0) if prev is None else max((L, 0), (prev[0], prev[1] + 1))
            if x[0] > U or (x[0] == U and x[1] > 0): return False, None
        sym.append(x); prev = x
    cand = sorted({float(v) for v in list(l) + list(u) + list(fixed.values())})
    gaps = [b - a for a, b in zip(cand[:-1], cand[1:]) if b > a]
    eps = (min(gaps) if gaps else 1.0) / (2.0 * (m + 1))
    return True, [x0 + k_ * eps for x0, k_ in sym]

def all_attaining(cx, l, u, I, cap=None):
    orders, complete = realizable_weak_orders(cx.nv, l, u, cap)
    wits = []
    for blocks in orders:
        f = np.zeros(cx.nv)
        for j, S in enumerate(blocks): f[list(S)] = j
        ib = bars(cx, f, 1); fixed = _fix_values(ib, I)
        if fixed is None: continue
        ok, vals = realise_with_fixed(blocks, l, u, fixed)
        if ok:
            g = np.zeros(cx.nv)
            for j, S in enumerate(blocks): g[list(S)] = vals[j]
            if bars(cx, g, 1) == I: wits.append(g)
    return wits, complete, len(orders)

# ---------- surfaces ----------
RP2 = [(0,1,2),(0,2,3),(0,3,4),(0,4,5),(0,5,1),(1,2,4),(2,3,5),(3,4,1),(4,5,2),(5,1,3)]
T7 = [tuple(sorted(((i)%7, (i+1)%7, (i+3)%7))) for i in range(7)] + [tuple(sorted(((i)%7, (i+2)%7, (i+3)%7))) for i in range(7)]

def check_surface(tris, nv):
    cx = Cx(nv, tris); cnt = collections.Counter()
    for a, b, c in cx.tris: cnt.update([(a, b), (a, c), (b, c)])
    return cx, len(cx.edges), len(cx.tris), set(cnt.values())
# ---------- the three checks (each consumes its random stream exactly as the stand-alone scripts did) ----------
SURFACES = {"mobius": (RP2[1:], 6), "rp2": (RP2, 6), "torus": (T7, 7), "ptorus": (T7[1:], 7)}


def _box(rng, nv):
    l = rng.integers(0, 12, nv).astype(float); u = l + rng.integers(0, 6, nv).astype(float)
    return l, u


def recon_dilation(cx, mask, marker):
    d = np.minimum(np.array(marker, float), mask); changed = True
    while changed:
        changed = False
        for v in range(cx.nv):
            m = max([d[v]] + [d[w] for w in cx.nbr[v]]); nvl = min(mask[v], m)
            if nvl > d[v]: d[v] = nvl; changed = True
    return d


def f_peak(cx, l, u, fd, p):
    """Closed surfaces, Theorem 13(b): f_D raised to the reconstruction by dilation of the peak marker u(p) at p, above max l."""
    mk = np.full(cx.nv, -1e18); mk[p] = u[p]
    R = recon_dilation(cx, np.asarray(u, float), mk)
    return np.maximum(fd, np.where(R > l.max(), R, -1e18))


def realise_max(blocks, l, u, fixed):
    """Near-supremum representative of an order type: block values as large as the box allows, each a symbolic eps below the next block
    (the supremum of the order cell itself may lie outside the strict order type). None if the order type cannot carry `fixed`."""
    m = len(blocks); nxt = None; sym = [None] * m
    for j in range(m - 1, -1, -1):
        S = blocks[j]; L = max(l[v] for v in S); U = min(u[v] for v in S)
        if j in fixed:
            c = fixed[j]
            if not (L <= c <= U) or (nxt is not None and not ((c, 0) < nxt)): return None
            x = (c, 0)
        else:
            x = (U, 0) if nxt is None else min((U, 0), (nxt[0], nxt[1] - 1))
            if x[0] < L or (x[0] == L and x[1] < 0): return None
        sym[j] = x; nxt = x
    cand = sorted({float(v) for v in list(l) + list(u) + list(fixed.values())})
    gaps = [b - a for a, b in zip(cand[:-1], cand[1:]) if b > a]
    eps = (min(gaps) if gaps else 1.0) / (2.0 * (m + 1))
    return [x0 + k * eps for x0, k in sym]


def connected_or_empty(cx, S):
    S = set(S)
    if not S: return True
    start = next(iter(S)); seen = {start}; stack = [start]
    while stack:
        v = stack.pop()
        for w in cx.nbr[v]:
            if w in S and w not in seen: seen.add(w); stack.append(w)
    return seen == S


def _attaining_max_realisations(cx, l, u, I1):
    """Yield the near-supremum representative g of every realisable order type that attains I1 (re-verified by recomputing Bar_1(g))."""
    orders, complete = realizable_weak_orders(cx.nv, l, u)
    if not complete:
        yield None; return
    for blocks in orders:
        f = np.zeros(cx.nv)
        for j, B in enumerate(blocks): f[list(B)] = j
        fixed = _fix_values(bars(cx, f, 1), I1)
        if fixed is None: continue
        vals = realise_max(blocks, l, u, fixed)
        if vals is None: continue
        g = np.zeros(cx.nv)
        for j, B in enumerate(blocks): g[list(B)] = vals[j]
        if not (np.all(g >= l - 1e-9) and np.all(g <= u + 1e-9)) or bars(cx, g, 1) != I1: continue
        yield g


def run_decide(fam, N, seed):
    """Theorem 13: f_D on the capped surface decides attainment (boundary); on closed surfaces f_D or a peak function f_p does."""
    base, nv = SURFACES[fam]; cx = Cx(nv, base); closed = not boundary_edges(cx); rng = np.random.default_rng(seed)
    st = collections.Counter(); notes = []
    for it in range(N):
        l, u = _box(rng, nv)
        I1 = image_barcode1(cx, l, u)
        if closed: fd = f_D(cx, l, u)
        else:
            M = float(u.max()) + 3.0; S, lS, uS = cap(cx, l, u, M); fd = f_D(S, lS, uS)[:nv]
        fin = lambda B: sorted(x for x in B if math.isfinite(x[1]))
        Bd = bars(cx, fd, 1); st['finite_equal'] += fin(Bd) == fin(I1)
        if Bd == I1: st['fD_attains'] += 1; continue
        peak_ok = False
        if closed:
            for p in range(nv):
                if u[p] > l.max():
                    g = f_peak(cx, l, u, fd, p)
                    if np.all(g >= l - 1e-9) and np.all(g <= u + 1e-9) and bars(cx, g, 1) == I1: peak_ok = True; break
        wits, complete, nord = all_attaining(cx, l, u, I1)
        if not complete: st['incomplete'] += 1; continue
        att = bool(wits)
        key = ('attained' if att else 'NOT attained') + (' | peak witness found' if peak_ok else '')
        st['fD fails -> ' + key] += 1
        if att and not closed: notes.append(f"COUNTEREXAMPLE (boundary): box {it}")
        if att and closed and not peak_ok: notes.append(f"closed case not covered by peak witnesses: box {it}")
        if not att and peak_ok: notes.append(f"BUG: peak witness attains but enumeration says not: box {it}")
    st['instances'] = N
    return dict(st), notes


def run_max(fam, N, seed):
    """Lemma 12 (surfaces with boundary): every attaining order type's near-supremum representative lies below f_D."""
    base, nv = SURFACES[fam]; cx = Cx(nv, base); rng = np.random.default_rng(seed); st = collections.Counter(); worst = 0.0
    for it in range(N):
        l, u = _box(rng, nv)
        I1 = image_barcode1(cx, l, u); M = float(u.max()) + 3.0
        S, lS, uS = cap(cx, l, u, M); fd = f_D(S, lS, uS)[:nv]
        att = 0; incomplete = False
        for g in _attaining_max_realisations(cx, l, u, I1):
            if g is None: st['incomplete'] += 1; incomplete = True; break
            att += 1; st['attaining_max_realisations'] += 1
            exc = float(np.max(g - fd))
            if exc > 1e-9: st['ABOVE_fD'] += 1; worst = max(worst, exc)
        if incomplete: continue
        st['attained_instances'] += att > 0
        st['fD_attains'] += bars(cx, fd, 1) == I1
        st['agree(attained <-> fD attains)'] += (att > 0) == (bars(cx, fd, 1) == I1)
    st['instances'] = N; st['max_excess_over_fD'] = worst
    return dict(st), []


def run_max_closed(fam, N, seed):
    """Lemma 12 and Step 1 of Theorem 13(b) on closed surfaces, for every attaining order type's near-supremum representative g:
    (i) g <= f_D wherever g <= L* = max l; (ii) {g > t} connected or empty for t >= L*; (iii) g <= f_p for p maximising g when max g > L*."""
    base, nv = SURFACES[fam]; cx = Cx(nv, base); assert not boundary_edges(cx)
    rng = np.random.default_rng(seed); st = collections.Counter()
    for it in range(N):
        l, u = _box(rng, nv)
        I1 = image_barcode1(cx, l, u); fd = f_D(cx, l, u); Ls = float(l.max())
        att = 0; incomplete = False
        for g in _attaining_max_realisations(cx, l, u, I1):
            if g is None: st['incomplete'] += 1; incomplete = True; break
            att += 1; st['attaining_max_realisations'] += 1
            low = g <= Ls + 1e-12
            if np.any(g[low] > fd[low] + 1e-9): st['VIOLATION (i) g > f_D below L*'] += 1
            for t in sorted({Ls} | {float(x) for x in g if x >= Ls}):
                if not connected_or_empty(cx, [v for v in range(nv) if g[v] > t]): st['VIOLATION (ii) superlevel disconnected'] += 1; break
            if g.max() > Ls:
                st['max_above_Lstar'] += 1
                p = int(np.argmax(g)); fp = f_peak(cx, l, u, fd, p)
                if np.any(g > fp + 1e-9): st['VIOLATION (iii) g > f_p'] += 1
        if incomplete: continue
        st['attained_instances'] += att > 0
    st['instances'] = N
    return dict(st), []


RUNNERS = {"decide": run_decide, "max": run_max, "max_closed": run_max_closed}


def run_job(job):
    import time as _time
    kind, fam, N, seed = job; t0 = _time.perf_counter()
    st, notes = RUNNERS[kind](fam, N, seed)
    return dict(check=kind, surface=fam, boxes=N, seed=seed, seconds=round(_time.perf_counter() - t0, 2), notes=notes, **st)
'''
t13 = types.ModuleType("t13core")                       # its own namespace: none of the notebook's functions are used or overwritten
exec(compile(T13_SRC, "t13core", "exec"), t13.__dict__)

# Counts logged by the stand-alone scripts that the paper cites (same surfaces, box sizes and seeds). FULL must reproduce them exactly.
T13_REFERENCE = {
    ("decide", "mobius", 400, 11): {"fD_attains": 393, "fD fails -> NOT attained": 7},
    ("decide", "rp2", 300, 12): {"fD_attains": 255, "fD fails -> attained | peak witness found": 45},
    ("decide", "ptorus", 200, 21): {"fD_attains": 193, "fD fails -> NOT attained": 7},
    ("decide", "torus", 200, 22): {"fD_attains": 156, "fD fails -> attained | peak witness found": 44},
    ("max", "mobius", 150, 32): {"attaining_max_realisations": 2745, "attained_instances": 148, "agree(attained <-> fD attains)": 150},
    ("max", "ptorus", 60, 31): {"attaining_max_realisations": 1285, "attained_instances": 59, "agree(attained <-> fD attains)": 60},
    ("max_closed", "rp2", 150, 41): {"attaining_max_realisations": 3930, "max_above_Lstar": 3207, "attained_instances": 150},
    ("max_closed", "torus", 60, 42): {"attaining_max_realisations": 2316, "max_above_Lstar": 1923, "attained_instances": 60},
}

_t13_jobs = ([("decide", *j, "main") for j in CONFIG["t13_decide"]] + [("max", *j, "main") for j in CONFIG["t13_max"]] +
             [("max_closed", *j, "main") for j in CONFIG["t13_max_closed"]] + [(*j, "extended") for j in CONFIG["t13_extended"]])
ck_t13 = Checkpoint("v6_theorem13")
for kind, fam, N, seed, run in _t13_jobs:
    key = f"{kind}:{fam}:{N}:{seed}"
    if not ck_t13.done(key):
        r = t13.run_job((kind, fam, N, seed)); r["notes"] = "; ".join(r["notes"])
        r["run"] = "reference" if (kind, fam, N, seed) in T13_REFERENCE else ("extended" if run == "extended" else "reduced")
        ck_t13.add(key, [r])
        print(f"{run:9s} {kind:10s} {fam:6s} boxes={N:5d} seed={seed:4d}: " +
              ", ".join(f"{k}={v}" for k, v in r.items() if k not in ("check", "surface", "boxes", "seed", "run", "notes")))
t13_df = ck_t13.frame()


def _t13_ref(row):
    ref = T13_REFERENCE.get((row["check"], row["surface"], int(row["boxes"]), int(row["seed"])))
    if ref is None:
        return None
    return all(int(row.get(k, 0) if row.get(k, 0) == row.get(k, 0) else 0) == v for k, v in ref.items())


def _t13_ok(row):
    get = lambda k: (row.get(k, 0) if row.get(k, 0) == row.get(k, 0) else 0) or 0
    if get("incomplete") or (isinstance(row.get("notes"), str) and row["notes"]):
        return False
    if row["check"] == "decide":
        closed = row["surface"] in ("rp2", "torus")
        bad = get("fD fails -> attained") + (0 if closed else get("fD fails -> attained | peak witness found"))
        return bad == 0 and get("fD fails -> NOT attained | peak witness found") == 0
    if row["check"] == "max":
        return get("ABOVE_fD") == 0 and get("agree(attained <-> fD attains)") == get("instances")
    return all(get(k) == 0 for k in row.index if str(k).startswith("VIOLATION"))


t13_df["matches_reference"] = t13_df.apply(_t13_ref, axis=1)
t13_df["consistent_with_theorem"] = t13_df.apply(_t13_ok, axis=1)
t13_df.to_csv(RESULTS / "raw" / "theorem13_checks.csv", index=False)
_cols = [c for c in ["run", "check", "surface", "boxes", "seed", "fD_attains", "fD fails -> NOT attained", "fD fails -> attained | peak witness found",
                     "attaining_max_realisations", "attained_instances", "agree(attained <-> fD attains)", "max_above_Lstar",
                     "matches_reference", "consistent_with_theorem", "seconds"] if c in t13_df]
print(t13_df[_cols].to_string(index=False))
_ref_rows = t13_df[t13_df.matches_reference.notna()]
T13 = dict(runs=int(len(t13_df)), boxes=int(t13_df.boxes.sum()), inconsistent=int((~t13_df.consistent_with_theorem.astype(bool)).sum()),
           reference_runs=int(len(_ref_rows)), reference_matches=int(_ref_rows.matches_reference.astype(bool).sum()),
           attaining_representatives=int(t13_df.get("attaining_max_realisations", pd.Series(dtype=float)).fillna(0).sum()))
T13_VERDICT = ("SUPPORTED" if T13["inconsistent"] == 0 and (PROFILE != "FULL" or T13["reference_matches"] == len(T13_REFERENCE))
               else "NOT SUPPORTED")
print("Theorem 13 and Lemma 12:", T13, T13_VERDICT)
set_status("theorem13", "EXECUTED", T13_VERDICT)

main      decide     mobius boxes=  400 seed=  11: seconds=0.51, finite_equal=400, fD_attains=393, fD fails -> NOT attained=7, instances=400
main      decide     rp2    boxes=  300 seed=  12: seconds=2.98, finite_equal=300, fD_attains=255, fD fails -> attained | peak witness found=45, instances=300
main      decide     ptorus boxes=  200 seed=  21: seconds=1.18, finite_equal=200, fD_attains=193, fD fails -> NOT attained=7, instances=200
main      decide     torus  boxes=  200 seed=  22: seconds=14.92, finite_equal=200, fD_attains=156, fD fails -> attained | peak witness found=44, instances=200
main      max        mobius boxes=  150 seed=  32: seconds=3.63, attaining_max_realisations=2745, attained_instances=148, fD_attains=148, agree(attained <-> fD attains)=150, instances=150, max_excess_over_fD=0.0
main      max        ptorus boxes=   60 seed=  31: seconds=4.21, attaining_max_realisations=1285, attained_instances=59, fD_attains=59, agree(attained <-> fD attains)=60, instances=60, ma

## 17 · Theorem 21: middle degrees fail on spheres

**Claim (Theorem 21).** Let $d\ge3$ and $1\le k\le d-2$. Some triangulation of $S^d$ carries a box with three free vertices whose degree-$k$ image barcode is not attained, and some carries a box with one free vertex and a $\tau$ for which every function in the box has more degree-$k$ bars longer than $\tau$ than the image barcode. The same holds for the $d$-ball. In degrees $0$, $d-1$ and $d$ the bound is attained for every triangulation of $S^d$ and every box: by Theorem 5, by Theorem 8, and, in degree $d$, by $u$ itself.

**Construction**, checked here step by step:
1. **The Möbius band of Theorem 11 in a 3-sphere.** Every triangle of the band has two vertices in $\{a,b,c\}$ and one in $\{p,q,r\}$, or the reverse, so the band lies in the join $\partial(abc)*\partial(pqr)$, a 6-vertex $S^3$. Stellar subdivisions of the simplices outside the band make it a full subcomplex. The new vertices are padded at $M$, and by Lemma 4 attainment is unchanged. Deleting the open star of one padding vertex gives a 3-ball.
2. **Example 18 in a 3-sphere**, inside the join $\partial(aby)*\partial(cxp)$. All of its bars end by 10, which is below $M=11$, so padding leaves the barcodes unchanged and the counts still fail.
3. **Raising the degree.** Suspension with both apexes fixed below every value shifts $H_k$ to $H_{k+1}$ naturally, so $(d,k)\mapsto(d+1,k+1)$.
4. **Raising the dimension.** A join with $S^0$ whose two vertices are fixed at $M$ keeps every barcode below $M$, so $(d,k)\mapsto(d+1,k)$.

Starting from $(3,1)$, these two steps reach every $(d,k)$ with $1\le k\le d-2$. The checks below cover $(3,1)$, $(4,2)$ and $(4,1)$:
- that each complex is a sphere or a ball ($\mathbb Z_2$ Betti numbers, closed pseudomanifold);
- that the subcomplex is full;
- the exact image barcode, from the rank invariant in the relevant degree;
- that the suspension isomorphism holds for every order type;
- attainment, or the exact minimum counts, by complete enumeration of order types.

Vertices fixed at a common value always share a block of every order type, so the enumeration merges them first. The self-check confirms that this gives the same order types.

In [30]:
# ---- general-degree simplicial tools over Z2 (used in Sections 17 and 18) -----------------------------------------------------------
class SComplex:
    """Simplicial complex generated by `facets` (any dimension). simplices[k] is the sorted list of k-simplices (vertex tuples)."""
    def __init__(self, facets, nv=None, name=""):
        S = set()
        for f in facets:
            f = tuple(sorted(int(x) for x in f))
            for k in range(1, len(f) + 1):
                S.update(itertools.combinations(f, k))
        self.dim = max(len(s) for s in S) - 1
        self.simplices = [sorted(s for s in S if len(s) == k + 1) for k in range(self.dim + 1)]
        self.facets = sorted({tuple(sorted(int(x) for x in f)) for f in facets})
        self.nv = int(nv) if nv is not None else max(s[0] for s in self.simplices[0]) + 1
        self.index = [{s: i for i, s in enumerate(sk)} for sk in self.simplices]
        self.name = name

    def two_skeleton(self):
        """The 2-skeleton as a Small2Complex: degree-0 and degree-1 lower-star homology only depend on it."""
        return Small2Complex(self.nv, self.simplices[2] if self.dim >= 2 else [], self.name + " (2-skeleton)",
                             edges=self.simplices[1] if self.dim >= 1 else [])


def _rank_gf2(rows):
    piv = {}
    for r in rows:
        while r:
            h = r.bit_length() - 1
            if h in piv:
                r ^= piv[h]
            else:
                piv[h] = r; break
    return len(piv)


def _kernel_gf2(cols):
    """Basis of the kernel of the GF(2) matrix with the given columns, as bitsets over column indices."""
    piv = {}; basis = []
    for j, c in enumerate(cols):
        comb = 1 << j; v = c
        while v:
            h = v.bit_length() - 1
            if h in piv:
                pv, pc = piv[h]; v ^= pv; comb ^= pc
            else:
                piv[h] = (v, comb); break
        if v == 0:
            basis.append(comb)
    return basis


def _bd_cols(K, k, idxs):
    """Columns of the boundary map d_k on the k-simplices with indices idxs, as bitsets over the (k-1)-simplices."""
    out = []
    for i in idxs:
        s = K.simplices[k][i]; c = 0
        for face in itertools.combinations(s, k):
            c |= 1 << K.index[k - 1][face]
        out.append(c)
    return out


def betti_z2_general(K):
    rk = [0] * (K.dim + 2)
    for k in range(1, K.dim + 1):
        rk[k] = _rank_gf2(_bd_cols(K, k, range(len(K.simplices[k]))))
    return [len(K.simplices[k]) - rk[k] - rk[k + 1] for k in range(K.dim + 1)]


def is_pseudomanifold(K):
    """Every ridge in exactly two facets, all facets of top dimension (closed pseudomanifold)."""
    d = K.dim
    if any(len(f) != d + 1 for f in K.facets):
        return False
    cnt = collections.Counter(r for f in K.facets for r in itertools.combinations(f, d))
    return set(cnt.values()) == {2}


def lowerstar_bars_general(K, f, deg):
    """Positive-length bars in degree deg of the lower-star filtration of f on K (Z2 column reduction). Simplices are ordered by
    (value, dimension), so every face precedes its cofaces; the barcode does not depend on how the remaining ties are broken."""
    f = np.asarray(f, float)
    simp = [(s, k) for k in range(min(deg + 1, K.dim) + 1) for s in K.simplices[k]]
    val = {s: max(f[v] for v in s) for s, _ in simp}
    simp.sort(key=lambda sk: (val[sk[0]], sk[1], sk[0]))
    pos = {s: i for i, (s, _) in enumerate(simp)}
    red = []; low_of = {}; paired = set(); out = []
    for j, (s, k) in enumerate(simp):
        c = 0
        if k > 0:
            for face in itertools.combinations(s, k):
                c |= 1 << pos[face]
        while c:
            h = c.bit_length() - 1
            if h in low_of:
                c ^= red[low_of[h]]
            else:
                break
        red.append(c)
        if c:
            h = c.bit_length() - 1; low_of[h] = j; paired.update((h, j))
            sb, kb = simp[h]
            if kb == deg and val[s] > val[sb]:
                out.append((float(val[sb]), float(val[s])))
    out += [(float(val[s]), math.inf) for i, (s, k) in enumerate(simp) if k == deg and i not in paired]
    return sorted(out)


def image_barcode_general(K, l, u, deg):
    """Barcode of im(H_deg(Sub_u) -> H_deg(Sub_l)) by inclusion-exclusion on the exact Z2 rank invariant
    rank(H_deg(Sub_u(s)) -> H_deg(Sub_l(t))), s <= t (independent of the lower-star reduction above)."""
    l = np.asarray(l, float); u = np.asarray(u, float)
    C = sorted(set(l.tolist()) | set(u.tolist())); n = len(C)
    val_u = [[max(u[v] for v in s) for s in K.simplices[k]] for k in range(K.dim + 1)]
    val_l = [[max(l[v] for v in s) for s in K.simplices[k]] for k in range(K.dim + 1)]
    memo = {}
    def R(i, j):
        if i < 0:
            return 0
        if (i, j) not in memo:
            A = [q for q, x in enumerate(val_u[deg]) if x <= C[i]]
            if deg == 0:
                Z = [1 << q for q in A]
            else:
                Z = [sum(1 << A[p] for p in range(len(A)) if comb >> p & 1) for comb in _kernel_gf2(_bd_cols(K, deg, A))]
            Bd = _bd_cols(K, deg + 1, [q for q, x in enumerate(val_l[deg + 1]) if x <= C[j]]) if deg + 1 <= K.dim else []
            memo[(i, j)] = _rank_gf2(Z + Bd) - _rank_gf2(Bd)
        return memo[(i, j)]
    out = []
    for i in range(n):
        for j in range(i + 1, n + 1):
            if j == n:
                m = R(i, n - 1) - R(i - 1, n - 1); d = math.inf
            else:
                m = R(i, j - 1) - R(i - 1, j - 1) - R(i, j) + R(i - 1, j); d = C[j]
            assert m >= 0, "negative multiplicity"
            out += [(C[i], d)] * m
    return sorted(out)


def realizable_weak_orders_merged(nv, l, u, cap=None):
    """The order types of realizable_weak_orders (Section 7), enumerated after merging the vertices fixed at a common value (l = u = c):
    such vertices carry the same value, so every order type puts them in one block. This keeps complexes with many fixed (padding)
    vertices tractable. Same set of order types; checked against the unmerged enumeration in the self-check below."""
    groups = {}; units = []
    for v in range(nv):
        if l[v] == u[v]:
            groups.setdefault(float(l[v]), []).append(v)
        else:
            units.append([v])
    units = list(groups.values()) + units
    gl = [float(max(l[v] for v in U)) for U in units]; gu = [float(min(u[v] for v in U)) for U in units]
    orders, complete = realizable_weak_orders(len(units), gl, gu, cap)
    return [tuple(tuple(sorted(v for i in B for v in units[i])) for B in blocks) for blocks in orders], complete


def exact_attainment_general(K, l, u, I, deg, orders):
    """Exact decision of 'some f in the box has Bar_deg(f) = I' (same procedure as exact_attainment in Section 7, with the
    general-degree reduction): for every order type, match bar endpoints to blocks, then realise with a symbolic epsilon."""
    orders, complete = orders; I = sorted(I)
    for blocks in orders:
        f = np.zeros(K.nv)
        for j, S in enumerate(blocks):
            f[list(S)] = j
        fixed = _fix_values(lowerstar_bars_general(K, f, deg), I)
        if fixed is None:
            continue
        ok, vals = realise_with_fixed(blocks, l, u, fixed)
        if ok:
            g = np.zeros(K.nv)
            for j, S in enumerate(blocks):
                g[list(S)] = vals[j]
            return dict(attained=True, witness=g, verified=bool(lowerstar_bars_general(K, g, deg) == I), n_orders=len(orders), complete=complete)
    return dict(attained=False if complete else None, witness=None, verified=None, n_orders=len(orders), complete=complete)


# self-checks: the general reduction and rank invariant agree with the degree-0/1 code of Section 7, and merging fixed vertices
# does not change the set of order types
GEN_CHECKS = dict(instances=0, bars_mismatch=0, image_mismatch=0, merged_order_mismatch=0, sphere_betti_ok=True)
_rg = rng_for("general_degree_selfcheck")
for _ in range(60 if PROFILE == "FULL" else 15):
    cx = random_pure_complex(_rg, int(_rg.integers(5, 8)), int(_rg.integers(3, 10)))
    K = SComplex(cx.tris, cx.nv)
    f = _rg.integers(0, 6, cx.nv).astype(float)
    l = _rg.integers(0, 6, cx.nv).astype(float); u = l + _rg.integers(0, 3, cx.nv)
    l[_rg.random(cx.nv) < 0.5] = 2.0; u = np.maximum(u, l)
    GEN_CHECKS["instances"] += 1
    GEN_CHECKS["bars_mismatch"] += sum(lowerstar_bars_general(K, f, k) != bars_cx(cx, f, k) for k in (0, 1))
    GEN_CHECKS["image_mismatch"] += sum(image_barcode_general(K, l, u, k) != image_barcode_rank(cx, l, u, k) for k in (0, 1))
    fixed_u = np.where(_rg.random(cx.nv) < 0.6, l, u)
    a = {tuple(tuple(sorted(B)) for B in o) for o in realizable_weak_orders(cx.nv, l, fixed_u)[0]}
    b = set(realizable_weak_orders_merged(cx.nv, l, fixed_u)[0])
    GEN_CHECKS["merged_order_mismatch"] += a != b
for d in (2, 3, 4):
    Sd = SComplex(list(itertools.combinations(range(d + 2), d + 1)))           # boundary of the (d+1)-simplex
    GEN_CHECKS["sphere_betti_ok"] &= betti_z2_general(Sd) == [1] + [0] * (d - 1) + [1] and is_pseudomanifold(Sd)
print("general-degree self-checks:", GEN_CHECKS)
assert GEN_CHECKS["bars_mismatch"] == GEN_CHECKS["image_mismatch"] == GEN_CHECKS["merged_order_mismatch"] == 0 and GEN_CHECKS["sphere_betti_ok"]

general-degree self-checks: {'instances': 60, 'bars_mismatch': 0, 'image_mismatch': 0, 'merged_order_mismatch': 0, 'sphere_betti_ok': True}


In [31]:
# ---- 18.2 constructions: joins, stellar subdivision to make a subcomplex full, suspension, padding ----------------------------------
def join_facets(A, B):
    """Facets of the join of two complexes given by their facets (vertex sets must be disjoint)."""
    return [tuple(sorted(tuple(a) + tuple(b))) for a in A for b in B]


def simplex_boundary(vs):
    return [tuple(c) for c in itertools.combinations(sorted(vs), len(vs) - 1)]


def all_faces(facets):
    S = set()
    for f in facets:
        for k in range(1, len(f) + 1):
            S.update(frozenset(c) for c in itertools.combinations(f, k))
    return S


def stellar(facets, tau, w):
    """Stellar subdivision of the simplex tau with the new vertex w."""
    out = []
    for s in facets:
        s = frozenset(s)
        if tau <= s:
            out += [(s - {v}) | {w} for v in tau]
        else:
            out.append(s)
    return out


def make_full(facets, K_faces, VK, nv):
    """Stellar-subdivide, highest dimension first, every simplex whose vertices all lie in VK but which is not a face of K.
    Afterwards K is a full subcomplex. Returns (facets, number of vertices)."""
    facets = [frozenset(f) for f in facets]
    while True:
        bad = [s for s in all_faces(facets) if s <= VK and s not in K_faces]
        if not bad:
            return [tuple(sorted(f)) for f in facets], nv
        facets = stellar(facets, max(bad, key=lambda s: (len(s), sorted(s))), nv); nv += 1


def is_full(K_faces, VK, facets):
    return all(s in K_faces for s in all_faces(facets) if s <= VK)


def suspend(facets, nv):
    """Suspension with two new apexes nv, nv+1."""
    return [tuple(f) + (nv,) for f in facets] + [tuple(f) + (nv + 1,) for f in facets], nv + 2


def remove_open_star(facets, v):
    return [f for f in facets if v not in f]


def embed_full(name, tris, A, B, l, u, M):
    """Put the 2-complex `tris` into the 3-sphere join(boundary A, boundary B), make it full, pad the box with M."""
    K_faces = all_faces([tuple(t) for t in tris]); VK = frozenset(range(len(l)))
    J = join_facets(simplex_boundary(A), simplex_boundary(B))
    assert all(frozenset(t) in all_faces(J) for t in tris), "not a subcomplex of the join"
    facets, nv = make_full(J, K_faces, VK, len(l))
    L = np.array(list(l) + [M] * (nv - len(l)), float); U = np.array(list(u) + [M] * (nv - len(u)), float)
    return SComplex(facets, nv, name), L, U, K_faces, VK


def _block_values(nv, blocks):
    f = np.zeros(nv)
    for j, S in enumerate(blocks):
        f[list(S)] = j
    return f


MD_ROWS = []
def md_record(case, d, k, construction, K, L, U, I, decision, **extra):
    row = dict(case=case, d=d, k=k, construction=construction, vertices=K.nv, facets=len(K.facets),
               betti=str(betti_z2_general(K)), pseudomanifold=is_pseudomanifold(K), free_vertices=int(np.sum(L != U)),
               image_barcode=bars_txt(I), **decision, **extra)
    MD_ROWS.append(row)
    print(f"{case:34s} d={d} k={k} | {K.nv} vertices, {len(K.facets)} facets, Z2 Betti {row['betti']} | image barcode {row['image_barcode']} | "
          + ", ".join(f"{a}={b}" for a, b in decision.items()))
    return row


_t_md = time.perf_counter()
# (1) Theorem 11's Moebius band (Figure 2, certificate D) lies in the join of the triangle boundaries abc and pqr, a 3-sphere.
#     Vertex labels of certificate D: 0=a 1=b 2=p 3=c 4=q 5=r.
_SD = CERT_SPECS["D"]
S3m, Lm, Um, KDm, VDm = embed_full("Moebius band in S^3", _SD["tris"], (0, 1, 3), (2, 4, 5), _SD["l"], _SD["u"], 5.0)
assert is_full(KDm, VDm, S3m.facets)
I1m = image_barcode_general(S3m, Lm, Um, 1)
_cxm = S3m.two_skeleton()
assert I1m == image_barcode_rank(_cxm, Lm, Um, 1)                     # two independent rank computations agree
ordm = realizable_weak_orders_merged(S3m.nv, Lm, Um)
exm = exact_attainment(_cxm, Lm, Um, I1m, 1, orders=ordm)
md_record("Moebius band (Thm 11) in S^3", 3, 1, "join of two triangle boundaries + stellar subdivisions + padding", S3m, Lm, Um, I1m,
          dict(order_types=exm["n_orders"], complete=exm["complete"], attained=exm["attained"]))
# the 3-ball: remove the open star of one padding vertex
B3m = SComplex(remove_open_star(S3m.facets, S3m.nv - 1), S3m.nv - 1, "Moebius band in B^3")
Lb, Ub = Lm[:-1], Um[:-1]; I1b = image_barcode_general(B3m, Lb, Ub, 1)
exb = exact_attainment(B3m.two_skeleton(), Lb, Ub, I1b, 1, orders=realizable_weak_orders_merged(B3m.nv, Lb, Ub))
md_record("Moebius band (Thm 11) in B^3", 3, 1, "as above, open star of one padding vertex removed", B3m, Lb, Ub, I1b,
          dict(order_types=exb["n_orders"], complete=exb["complete"], attained=exb["attained"]))

# (2) Example 18 (certificate B) lies in the join of aby and cxp. Labels of certificate B: 0=x 1=a 2=b 3=c 4=p 5=y.
_SB = CERT_SPECS["B"]
S3e, Le, Ue, KBe, VBe = embed_full("Example 18 in S^3", _SB["tris"], (1, 2, 5), (3, 0, 4), _SB["l"], _SB["u"], 11.0)
assert is_full(KBe, VBe, S3e.facets)
I1e = image_barcode_general(S3e, Le, Ue, 1)
assert I1e == image_barcode_rank(S3e.two_skeleton(), Le, Ue, 1)
orde = realizable_weak_orders_merged(S3e.nv, Le, Ue); assert orde[1]
MD_TAUS = [3.0, 3.5, 3.99]
ib_e = [index_barcode(S3e.two_skeleton(), blocks, 1) for blocks in orde[0]]
gap_e = {t: (exact_min_persistence_count(orde[0], ib_e, Le, Ue, t), image_count_pers(I1e, t)) for t in MD_TAUS}
# the minimum over the box of the longest bar: one free vertex x, so every f has bars [2, f(x)) and [f(x), 10)
_x = int(np.flatnonzero(Le != Ue)[0]); _grid = np.linspace(Le[_x], Ue[_x], 201)
_longest = [max(d - b for b, d in lowerstar_bars_general(S3e, np.where(np.arange(S3e.nv) == _x, c, Le), 1)) for c in _grid]
md_record("Example 18 in S^3 (counts)", 3, 1, "join of two triangle boundaries + stellar subdivisions + padding", S3e, Le, Ue, I1e,
          dict(order_types=len(orde[0]), complete=orde[1], count_gap_taus=str([t for t, (a, b) in gap_e.items() if a > b]),
               min_count_vs_image=str(gap_e)), min_longest_bar_on_grid=float(min(_longest)))

# (3) raise the degree: suspension with both apexes fixed below every value shifts H_k to H_{k+1}; (d, k) = (3, 1) -> (4, 2)
def suspended(K, L, U, low):
    F, nv = suspend(K.facets, K.nv)
    return SComplex(F, nv, "suspension of " + K.name), np.append(L, [low, low]), np.append(U, [low, low])


def check_suspension(K, L, U, K2, L2, U2, orders):
    """Bar_{k+1} on the suspension equals Bar_k on K for one representative of every order type (the suspension isomorphism)."""
    bad = 0
    for blocks in orders:
        f = _block_values(K.nv, blocks); f2 = np.append(f, [-1.0, -1.0])
        bad += lowerstar_bars_general(K2, f2, 2) != lowerstar_bars_general(K, f, 1)
    return bad


S4m, L4m, U4m = suspended(S3m, Lm, Um, -1.0)
I2m = image_barcode_general(S4m, L4m, U4m, 2)
ord4m = realizable_weak_orders_merged(S4m.nv, L4m, U4m)
ex4m = exact_attainment_general(S4m, L4m, U4m, I2m, 2, ord4m)
md_record("Moebius band, suspended", 4, 2, "suspension, apexes fixed at -1", S4m, L4m, U4m, I2m,
          dict(order_types=ex4m["n_orders"], complete=ex4m["complete"], attained=ex4m["attained"]),
          same_image_as_degree_1=I2m == I1m, suspension_iso_failures=check_suspension(S3m, Lm, Um, S4m, L4m, U4m, ordm[0]))

S4e, L4e, U4e = suspended(S3e, Le, Ue, -1.0)
I2e = image_barcode_general(S4e, L4e, U4e, 2)
ord4e = realizable_weak_orders_merged(S4e.nv, L4e, U4e); assert ord4e[1]
ib4e = [lowerstar_bars_general(S4e, _block_values(S4e.nv, b), 2) for b in ord4e[0]]
gap4e = {t: (exact_min_persistence_count(ord4e[0], ib4e, L4e, U4e, t), image_count_pers(I2e, t)) for t in MD_TAUS}
md_record("Example 18, suspended (counts)", 4, 2, "suspension, apexes fixed at -1", S4e, L4e, U4e, I2e,
          dict(order_types=len(ord4e[0]), complete=ord4e[1], count_gap_taus=str([t for t, (a, b) in gap4e.items() if a > b]),
               min_count_vs_image=str(gap4e)), same_image_as_degree_1=I2e == I1e)

# (4) raise the dimension, keep the degree: join with S^0 (two vertices fixed at the padding value M); (d, k) = (3, 1) -> (4, 1)
def joined_S0(K, L, U, M):
    F = join_facets(K.facets, [(K.nv,), (K.nv + 1,)])
    return SComplex(F, K.nv + 2, "join of " + K.name + " with S^0"), np.append(L, [M, M]), np.append(U, [M, M])


J4m, LJm, UJm = joined_S0(S3m, Lm, Um, 5.0)
assert is_full(KDm, VDm, J4m.facets)
I1Jm = image_barcode_general(J4m, LJm, UJm, 1)
exJm = exact_attainment(J4m.two_skeleton(), LJm, UJm, I1Jm, 1, orders=realizable_weak_orders_merged(J4m.nv, LJm, UJm))
md_record("Moebius band, joined with S^0", 4, 1, "join with S^0 fixed at M", J4m, LJm, UJm, I1Jm,
          dict(order_types=exJm["n_orders"], complete=exJm["complete"], attained=exJm["attained"]), same_image_as_S3=I1Jm == I1m)

J4e, LJe, UJe = joined_S0(S3e, Le, Ue, 11.0)
I1Je = image_barcode_general(J4e, LJe, UJe, 1)
ordJe = realizable_weak_orders_merged(J4e.nv, LJe, UJe); assert ordJe[1]
ibJe = [index_barcode(J4e.two_skeleton(), b, 1) for b in ordJe[0]]
gapJe = {t: (exact_min_persistence_count(ordJe[0], ibJe, LJe, UJe, t), image_count_pers(I1Je, t)) for t in MD_TAUS}
md_record("Example 18, joined with S^0 (counts)", 4, 1, "join with S^0 fixed at M", J4e, LJe, UJe, I1Je,
          dict(order_types=len(ordJe[0]), complete=ordJe[1], count_gap_taus=str([t for t, (a, b) in gapJe.items() if a > b]),
               min_count_vs_image=str(gapJe)), same_image_as_S3=I1Je == I1e)

md_df = pd.DataFrame(MD_ROWS)
md_df.to_csv(RESULTS / "raw" / "middle_degrees.csv", index=False)
_spheres_ok = bool(all(r["pseudomanifold"] for r in MD_ROWS if "B^3" not in r["case"]) and
                   all(r["betti"] == str([1] + [0] * (r["d"] - 1) + [1]) for r in MD_ROWS if "B^3" not in r["case"]) and
                   all(r["betti"] == str([1] + [0] * r["d"]) for r in MD_ROWS if "B^3" in r["case"]))
_barcode_fail = all(r.get("attained") is False and r.get("complete") for r in MD_ROWS if "counts" not in r["case"])
_count_fail = all(r.get("count_gap_taus") not in (None, "[]") and r.get("complete") for r in MD_ROWS if "counts" in r["case"])
_lift_ok = bool(md_df.get("suspension_iso_failures", pd.Series([0])).fillna(0).sum() == 0 and
                all(v is not False for v in md_df.get("same_image_as_degree_1", pd.Series([], dtype=object)).dropna()) and
                all(v is not False for v in md_df.get("same_image_as_S3", pd.Series([], dtype=object)).dropna()))
MIDDLE = dict(cases=len(MD_ROWS), spheres_and_ball_ok=_spheres_ok, barcode_not_attained_all=_barcode_fail, count_gaps_all=_count_fail,
              lifts_consistent=_lift_ok, moebius_s3_vertices=int(S3m.nv), moebius_s3_order_types=int(exm["n_orders"]),
              ex18_s3_vertices=int(S3e.nv), ex18_min_longest_bar=float(min(_longest)), seconds=round(time.perf_counter() - _t_md, 1))
MIDDLE_VERDICT = "SUPPORTED" if (_spheres_ok and _barcode_fail and _count_fail and _lift_ok) else "NOT SUPPORTED"
print("middle degrees:", MIDDLE, MIDDLE_VERDICT)
set_status("middle_degrees", "EXECUTED", MIDDLE_VERDICT)

Moebius band (Thm 11) in S^3       d=3 k=1 | 24 vertices, 72 facets, Z2 Betti [1, 0, 0, 1] | image barcode [0,2) [3,5) | order_types=13, complete=True, attained=False
Moebius band (Thm 11) in B^3       d=3 k=1 | 23 vertices, 66 facets, Z2 Betti [1, 0, 0, 0] | image barcode [0,2) [3,5) | order_types=13, complete=True, attained=False
Example 18 in S^3 (counts)         d=3 k=1 | 31 vertices, 112 facets, Z2 Betti [1, 0, 0, 1] | image barcode [2,5) [7,10) | order_types=1, complete=True, count_gap_taus=[3.0, 3.5, 3.99], min_count_vs_image={3.0: (1, 0), 3.5: (1, 0), 3.99: (1, 0)}
Moebius band, suspended            d=4 k=2 | 26 vertices, 144 facets, Z2 Betti [1, 0, 0, 0, 1] | image barcode [0,2) [3,5) | order_types=13, complete=True, attained=False
Example 18, suspended (counts)     d=4 k=2 | 33 vertices, 224 facets, Z2 Betti [1, 0, 0, 0, 1] | image barcode [2,5) [7,10) | order_types=1, complete=True, count_gap_taus=[3.0, 3.5, 3.99], min_count_vs_image={3.0: (1, 0), 3.5: (1, 0), 3.99: (1, 0)}


## 18 · 3-D Freudenthal grids

**Theorem 8 with $d=3$ (voids).** A 3-D image is a Freudenthal-triangulated grid, a 3-ball. Coning its boundary from a vertex fixed at $M$ gives a 3-sphere that contains the grid as a full subcomplex. For random integer boxes, most of them with one to three raised interior vertices so that voids occur, the section checks that:
- $f_D$ of the coned grid attains the degree-2 image barcode on the sphere;
- its restriction attains the image barcode of the grid itself (Lemma 4);
- the two image barcodes agree below $M$.

For comparison, it also reports how often $f_D$ computed on the grid without the cone attains, which no theorem covers.

**Degree 1, the middle degree for $d=3$.** Degree-1 homology depends only on the 2-skeleton, so the exact machinery of Section 7 applies. Boxes fix every vertex at a random integer in 0..7 except one to six free vertices, whose intervals have length 1 to 4. Each box is decided like the census of Section 11:
1. try verified witnesses: $f_D$ of the coned grid, $f_D$ of the grid, and $f_U$;
2. otherwise enumerate the order types exhaustively.

For every unattained box, the exact minimum τ-counts are compared with the image counts.

In [32]:
# ---- 18 · 3-D Freudenthal grids: Theorem 8 with d = 3 (degree 2), and an exact degree-1 census --------------------------------------
def freudenthal_tets(shape):
    """Freudenthal (Kuhn) triangulation of a grid with shape (a, b, c) vertices: each unit cube is cut into the 6 tetrahedra along its main
    diagonal (monotone lattice paths from the lowest to the highest corner). Vertex id = (i * b + j) * c + k."""
    a, b, c = shape
    vid = lambda i, j, k: (i * b + j) * c + k
    tets = []
    for i in range(a - 1):
        for j in range(b - 1):
            for k in range(c - 1):
                for perm in itertools.permutations(range(3)):
                    p = [i, j, k]; path = [vid(*p)]
                    for axis in perm:
                        p[axis] += 1; path.append(vid(*p))
                    tets.append(tuple(sorted(path)))
    return tets


def coned_ball(tets, nv):
    """The ball plus the cone over its boundary from a new apex nv: a triangulated 3-sphere containing the ball as a full subcomplex."""
    cnt = collections.Counter(t for T in tets for t in itertools.combinations(T, 3))
    return tets + [t + (nv,) for t, k in cnt.items() if k == 1], nv + 1


def void_box(rng, shape):
    """Degree-2 test box: integer values in 0..19 with widths 0..3; with probability 0.7, one to three interior vertices are raised to
    14..19 so that voids (bounded complementary components) appear in the image barcode."""
    a, b, c = shape; n = a * b * c
    l = rng.integers(0, 16, n).astype(float); u = l + rng.integers(0, 4, n)
    interior = [(i * b + j) * c + k for i in range(1, a - 1) for j in range(1, b - 1) for k in range(1, c - 1)]
    if interior and rng.random() < 0.7:
        for v in rng.choice(interior, size=min(len(interior), int(rng.integers(1, 4))), replace=False):
            l[v] = float(rng.integers(14, 20)); u[v] = l[v] + rng.integers(0, 4)
    return l, u


def grid_box(rng, n, free, vmax=7):
    """Integer box: every vertex fixed at a random value in 0..vmax except `free` vertices, whose intervals have length 1..4."""
    l = rng.integers(0, vmax + 1, n).astype(float); u = l.copy()
    idx = rng.choice(n, size=min(free, n), replace=False); u[idx] = l[idx] + rng.integers(1, 5, len(idx))
    return l, u


def dual_from_graph(cx, l, u):
    """f_D = -f_U(-u, -l), computed on the 1-skeleton of cx (Section 4)."""
    return -universal_realizer(certify_graph(cx.graph, -np.asarray(u, float), -np.asarray(l, float), 0.0), -np.asarray(u, float))


# 18.1 Theorem 8 with d = 3: on the coned grid (a 3-sphere) f_D attains the degree-2 image barcode; on the grid itself (a 3-ball, a full
# subcomplex of it padded at M) the restriction of that f_D attains, by Lemma 4. The f_D computed on the ball without the cone is shown
# for comparison only (no theorem covers it).
ck_g3 = Checkpoint("v6_grid3d_degree2")
for shape in CONFIG["g3_dual_shapes"]:
    key = "x".join(map(str, shape))
    if ck_g3.done(key):
        continue
    rng = rng_for("g3_dual", key); tets = freudenthal_tets(shape); n = int(np.prod(shape))
    Ball = SComplex(tets, n, f"grid {key}"); F3, n3 = coned_ball(tets, n); Sph = SComplex(F3, n3, f"coned grid {key}")
    cxB, cxS = Ball.two_skeleton(), Sph.two_skeleton(); rows = []; _t0 = time.perf_counter()
    for i in range(CONFIG["g3_dual_boxes"]):
        l, u = void_box(rng, shape)
        M = float(u.max() + 1); L3, U3 = np.append(l, M), np.append(u, M)
        I2S = image_barcode_general(Sph, L3, U3, 2); I2B = image_barcode_general(Ball, l, u, 2)
        fdS = dual_from_graph(cxS, L3, U3); fdB = dual_from_graph(cxB, l, u)
        in_box = bool(np.all(fdS >= L3) and np.all(fdS <= U3) and np.all(fdB >= l) and np.all(fdB <= u))
        rows.append(dict(shape=key, n=n, box=i, image_bars_sphere=len(I2S), image_bars_ball=len(I2B), in_box=in_box,
                         sphere_fD_attains=lowerstar_bars_general(Sph, fdS, 2) == I2S,
                         ball_restricted_fD_attains=lowerstar_bars_general(Ball, fdS[:n], 2) == I2B,
                         ball_own_fD_attains=lowerstar_bars_general(Ball, fdB, 2) == I2B,
                         padding_lemma_ok=sorted((b, math.inf if d == M else d) for b, d in I2S) == I2B))
    ck_g3.add(key, rows)
    print(f"grid {key}: {len(rows)} boxes in {time.perf_counter() - _t0:.0f}s")
g3d = ck_g3.frame()
g3d_s = (g3d.groupby("shape", sort=False).agg(n=("n", "first"), boxes=("box", "size"), mean_image_bars=("image_bars_ball", "mean"),
                                               in_box=("in_box", "sum"), sphere_fD_attains=("sphere_fD_attains", "sum"),
                                               ball_restricted_fD_attains=("ball_restricted_fD_attains", "sum"),
                                               ball_own_fD_attains=("ball_own_fD_attains", "sum")).reset_index())
G3_DUAL = dict(boxes=int(len(g3d)), shapes=int(g3d["shape"].nunique()) if len(g3d) else 0,
               theorem_failures=int((~g3d.sphere_fD_attains.astype(bool)).sum() + (~g3d.ball_restricted_fD_attains.astype(bool)).sum()),
               not_in_box=int((~g3d.in_box.astype(bool)).sum()), ball_own_fD_failures=int((~g3d.ball_own_fD_attains.astype(bool)).sum()),
               padding_lemma_failures=int((~g3d.padding_lemma_ok.astype(bool)).sum()),
               boxes_with_image_bars=int((g3d.image_bars_ball > 0).sum()))
print(g3d_s.to_string(index=False)); print("Theorem 8 with d = 3, degree 2:", G3_DUAL)

# 18.2 exact degree-1 census on 3-D grids (the middle degree for d = 3). Degree-1 homology only depends on the 2-skeleton, so the exact
# machinery of Section 7 applies: witnesses first (f_D of the coned grid, f_D of the grid, f_U), then exhaustive enumeration of order types.
def decide_degree1_grid(cxB, cxS, l, u, I1, cap):
    n = cxB.nv; M = float(u.max() + 1); L3, U3 = np.append(l, M), np.append(u, M)
    for how, f in (("f_D (coned)", dual_from_graph(cxS, L3, U3)[:n]), ("f_D", dual_from_graph(cxB, l, u)),
                   ("f_U", universal_realizer(certify_graph(cxB.graph, l, u, 0.0), l))):
        if bars_cx(cxB, f, 1) == I1:
            return "attained", how, None
    orders = realizable_weak_orders_merged(n, l, u, cap)
    ex = exact_attainment(cxB, l, u, I1, 1, orders=orders)
    if ex["attained"] is True:
        return ("attained" if ex["verified"] else "WITNESS FAILED"), "exact", None
    if ex["attained"] is None:
        return "undecided", "exact", None
    ib = [index_barcode(cxB, o, 1) for o in orders[0]]
    lens = sorted({d - b for b, d in I1 if math.isfinite(d)} | {0.0})
    taus = sorted({x for a, b in zip(lens, lens[1:] + [lens[-1] + 1.0]) for x in (a, 0.5 * (a + b))})
    gaps = [t for t in taus if exact_min_persistence_count(orders[0], ib, l, u, t) > image_count_pers(I1, t)]
    return "NOT attained", "exact", bool(gaps)


ck_g3c = Checkpoint("v6_grid3d_degree1_census")
for shape, free in CONFIG["g3_census_jobs"]:
    key = "x".join(map(str, shape)) + f"/free{free}"
    if ck_g3c.done(key):
        continue
    rng = rng_for("g3_census", key); tets = freudenthal_tets(shape); n = int(np.prod(shape))
    cxB = SComplex(tets, n).two_skeleton(); F3, n3 = coned_ball(tets, n); cxS = SComplex(F3, n3).two_skeleton()
    rows = []; tries = 0; _t0 = time.perf_counter()
    while len(rows) < CONFIG["g3_census_boxes"] and tries < 50 * CONFIG["g3_census_boxes"]:
        tries += 1
        l, u = grid_box(rng, n, free); I1 = image_barcode_rank(cxB, l, u, 1)
        if not has_finite_bar(I1):
            continue
        status, how, gap = decide_degree1_grid(cxB, cxS, l, u, I1, CONFIG["d1x_order_cap"])
        rows.append(dict(shape="x".join(map(str, shape)), n=n, free=free, status=status, how=how, count_gap=gap, bars=len(I1),
                         example=json.dumps(dict(l=l.tolist(), u=u.tolist())) if status == "NOT attained" else None))
    ck_g3c.add(key, rows)
    print(f"census {key}: {len(rows)} boxes ({time.perf_counter() - _t0:.0f}s)")
g3c = ck_g3c.frame()
g3c_s = (g3c.assign(not_attained=g3c.status == "NOT attained", undecided=g3c.status == "undecided",
                    by_witness=g3c.how.isin(["f_D (coned)", "f_D", "f_U"]), by_enumeration=g3c.how == "exact",
                    count_gap=g3c.count_gap.map(lambda v: bool(v) if v is not None and v == v else False))
         .groupby(["shape", "free"], sort=False)
         .agg(n=("n", "first"), boxes=("status", "size"), by_witness=("by_witness", "sum"), by_enumeration=("by_enumeration", "sum"),
              not_attained=("not_attained", "sum"), count_gaps=("count_gap", "sum"), undecided=("undecided", "sum")).reset_index())
G3_CENSUS = dict(boxes=int(len(g3c)), not_attained=int((g3c.status == "NOT attained").sum()), undecided=int((g3c.status == "undecided").sum()),
                 witness_failures=int((g3c.status == "WITNESS FAILED").sum()), count_gaps=int(g3c.count_gap.map(lambda v: v is True).sum()),
                 decided_by_witness=int(g3c.how.isin(["f_D (coned)", "f_D", "f_U"]).sum()))
print(g3c_s.to_string(index=False)); print("3-D degree-1 census:", G3_CENSUS)
G3_VERDICT = ("SUPPORTED" if (G3_DUAL["theorem_failures"] == 0 and G3_DUAL["not_in_box"] == 0 and G3_DUAL["padding_lemma_failures"] == 0
                              and G3_CENSUS["witness_failures"] == 0) else "NOT SUPPORTED")
g3d.to_csv(RESULTS / "raw" / "grid3d_degree2.csv", index=False); g3c.to_csv(RESULTS / "raw" / "grid3d_degree1_census.csv", index=False)
set_status("grid3d", "EXECUTED", G3_VERDICT)

grid 3x3x3: 100 boxes in 4s
grid 3x3x4: 100 boxes in 7s
grid 4x4x4: 100 boxes in 27s
grid 4x4x5: 100 boxes in 42s
grid 5x5x5: 100 boxes in 121s
shape   n  boxes  mean_image_bars  in_box  sphere_fD_attains  ball_restricted_fD_attains  ball_own_fD_attains
3x3x3  27    100             0.21     100                100                         100                  100
3x3x4  36    100             0.40     100                100                         100                  100
4x4x4  64    100             0.50     100                100                         100                  100
4x4x5  80    100             0.60     100                100                         100                  100
5x5x5 125    100             0.72     100                100                         100                  100
Theorem 8 with d = 3, degree 2: {'boxes': 500, 'shapes': 5, 'theorem_failures': 0, 'not_in_box': 0, 'ball_own_fD_failures': 0, 'padding_lemma_failures': 0, 'boxes_with_image_bars': 216}
census 2x3

## 19 · Scaling (Figure 3a)

The fields are Gaussian random fields on 4-connected grids with implicit neighbours, so $m\approx2n$. Each size is repeated with pinned single threads, and the phases are timed separately: sort, sweep, top-down pass and τ-curve. A hardware-independent count of union–find operations is recorded alongside.

The second cell relabels the grid in sweep order and times plain degree-0 persistence on the same input. The third measures the time per vertex of the bound, of persistence, and of a random memory access. It also reads the cache sizes, to separate the algorithm's cost from the memory hierarchy.

In [33]:
BYTES_PER_VERTEX_GRID = 8 * 2 + 8 + 8 * 12 + 2 + 8 * 2 + 8 * 2 + 8   # inputs, order, 12 int64/float64 node+UF arrays, flags, val/fs, sort temp

def scaling_field(rng, H, W):
    ell = ndimage.uniform_filter(rng.standard_normal((H, W)), size=7, mode="nearest")
    u = ell + 0.02 + 0.2 * rng.random((H, W)) * (1 + np.linspace(0, 3, W)[None, :])
    return ell, u

def time_certify_grid(ell, u, tau, conn=4):
    cert, tm = certify_grid(ell, u, tau, conn, return_timing=True)
    t0 = time.perf_counter(); lo, hi = cert.tau_intervals(); np.sort(lo); np.sort(hi); tm["t_curve"] = time.perf_counter() - t0
    tm["t_total"] += tm["t_curve"]
    nb = ell.nbytes + u.nbytes + sum(a.nbytes for a in (cert.parent, cert.mu, cert.lam, cert.minell, cert.trap, cert.minimal,
                                                          cert.maxchild_tau, cert.vertex, cert.Dv, cert.fstar)) + 8 * ell.size * 5
    return cert, tm, nb

_warm = scaling_field(np.random.default_rng(0), 20, 20); time_certify_grid(*_warm, 0.1)     # JIT warm-up (not timed)

ck_p2 = Checkpoint("p2_scaling"); SKIPPED_SIZES = []
for n in CONFIG["p2_sizes"]:
    if ck_p2.done(n):
        continue
    need = BYTES_PER_VERTEX_GRID * n * 1.15
    if need > 0.85 * psutil.virtual_memory().available:
        SKIPPED_SIZES.append(n); print(f"skip n={n}: needs ~{need/1e9:.1f} GB"); continue
    H = int(round(math.sqrt(n))); W = int(round(n / H))
    ell, u = scaling_field(rng_for("p2", n), H, W)
    rows = []
    for rep in range(CONFIG["p2_reps"]):
        gc.collect()
        cert, tm, nb = time_certify_grid(ell, u, 0.1)
        rows.append(dict(n=H * W, m=2 * H * W - H - W, rep=rep, T=cert.T, ops=cert.ops, array_bytes=nb,
                         peak_rss_gb=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, **tm))
        del cert
    ck_p2.add(n, rows)
    print(f"n={H*W:>10d}  median total {np.median([r['t_total'] for r in rows]):.3f}s  ops/n {rows[0]['ops']/(H*W):.3f}")
    del ell, u; gc.collect()
p2 = ck_p2.frame()
p2.to_csv(RESULTS / "raw" / "p2_scaling_runs.csv", index=False)
p2m = p2.groupby("n").agg(t_total=("t_total", "median"), t_total_q1=("t_total", lambda x: x.quantile(0.25)),
                          t_total_q3=("t_total", lambda x: x.quantile(0.75)), t_sort=("t_sort", "median"),
                          t_sweep=("t_sweep", "median"), t_topdown=("t_topdown", "median"), t_curve=("t_curve", "median"),
                          ops=("ops", "median"), array_bytes=("array_bytes", "median"), peak_rss_gb=("peak_rss_gb", "max"),
                          T=("T", "first"), reps=("rep", "size")).reset_index()
p2m.to_csv(RESULTS / "processed" / "scaling.csv", index=False)

def loglog_slope(x, y):
    x = np.log(np.asarray(x, float)); y = np.log(np.asarray(y, float))
    A = np.stack([x, np.ones_like(x)], 1); coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    return float(coef[0])

SLOPES = {k: loglog_slope(p2m.n, p2m[k]) for k in ["t_total", "t_sort", "t_sweep", "t_topdown", "t_curve", "ops", "array_bytes"]}
SLOPES["t_total_over_nlogn"] = loglog_slope(p2m.n, p2m.t_total / (p2m.n * np.log(p2m.n)))
lo_s, hi_s = CONFIG["p2_slope_range"]
full_range = len(p2m) >= 4 and p2m.n.max() >= 0.9 * max(CONFIG["p2_sizes"]) and max(CONFIG["p2_sizes"]) >= 5e7
if SLOPES["t_total"] > CONFIG["p2_slope_falsify"]:
    P2_VERDICT = "NOT SUPPORTED"
elif lo_s <= SLOPES["t_total"] <= hi_s and full_range:
    P2_VERDICT = "SUPPORTED"
else:
    P2_VERDICT = "INCONCLUSIVE"
P2_NOTE = ("range truncated to n<=%d" % p2m.n.max()) if not full_range else "full frozen range"
print(p2m[["n", "t_total", "t_sort", "t_sweep", "t_topdown", "ops", "array_bytes"]].to_string(index=False))
print("log-log slopes:", {k: round(v, 3) for k, v in SLOPES.items()}, "| verdict:", P2_VERDICT, "|", P2_NOTE)
set_status("P2_scaling", "EXECUTED", f"{P2_VERDICT} ({P2_NOTE})", skipped_sizes=SKIPPED_SIZES)

# density m/n on sparse random graphs (CSR kernel) and tau-flatness
ck_p2d = Checkpoint("p2_density")
for d in CONFIG["p2_density_degrees"]:
    if ck_p2d.done(d):
        continue
    rng = rng_for("p2d", d); n = CONFIG["p2_density_n"]
    g = graph_er_sparse(n, d, rng); l = rng.standard_normal(n); u_ = l + 0.3 * rng.random(n)
    certify_graph(graph_path(3), np.zeros(3), np.ones(3), 0.1)
    rows = []
    for rep in range(max(3, CONFIG["p2_reps"] // 2)):
        t0 = time.perf_counter(); c = certify_graph(g, l, u_, 0.1, count_ops=True); dt = time.perf_counter() - t0
        rows.append(dict(avg_deg=d, n=n, m=g.m, rep=rep, t_total=dt, ops=c.ops, T=c.T))
    ck_p2d.add(d, rows)
p2d = ck_p2d.frame().groupby(["avg_deg", "n", "m"]).agg(t_total=("t_total", "median"), ops=("ops", "median")).reset_index()
p2d.to_csv(RESULTS / "processed" / "scaling_density.csv", index=False)
SLOPES["time_vs_m_fixed_n"] = loglog_slope(p2d.m, p2d.t_total)
print(p2d.to_string(index=False))

ck_p2t = Checkpoint("p2_tau")
if not ck_p2t.done("all"):
    n = CONFIG["p2_tau_n"]; H = int(math.sqrt(n)); ell, u = scaling_field(rng_for("p2tau"), H, H); rows = []
    for tau in [0.0, 0.01, 0.1, 1.0, 10.0, 100.0]:
        for rep in range(3):
            cert, tm, _ = time_certify_grid(ell, u, tau)
            rows.append(dict(tau=tau, rep=rep, T=cert.T, **tm))
    ck_p2t.add("all", rows)
p2t = ck_p2t.frame().groupby("tau").agg(t_total=("t_total", "median"), T=("T", "first")).reset_index()
p2t.to_csv(RESULTS / "processed" / "scaling_tau.csv", index=False)
print(p2t.to_string(index=False))

n=     10000  median total 0.002s  ops/n 3.654
n=     29929  median total 0.006s  ops/n 3.643
n=     99856  median total 0.024s  ops/n 3.752
n=    299756  median total 0.079s  ops/n 3.778
n=   1000000  median total 0.413s  ops/n 3.821
n=   2999824  median total 1.752s  ops/n 3.842
n=  10001406  median total 7.691s  ops/n 3.849
n=  19998784  median total 17.129s  ops/n 3.842
n=  49999041  median total 48.677s  ops/n 3.855
       n   t_total    t_sort   t_sweep  t_topdown         ops  array_bytes
   10000  0.002021  0.000799  0.000960   0.000052     36537.0    1220000.0
   29929  0.006471  0.002714  0.003099   0.000144    109031.0    3651338.0
   99856  0.023849  0.010422  0.011201   0.000544    374653.0   12182432.0
  299756  0.078598  0.035861  0.034800   0.001976   1132536.0   36570232.0
 1000000  0.413039  0.132484  0.247952   0.010844   3820526.0  122000000.0
 2999824  1.752204  0.486071  1.145508   0.043148  11524716.0  365978528.0
10001406  7.691017  2.039266  5.005409   0.319529 

In [34]:
_warm = scaling_field(np.random.default_rng(2), 30, 30); certify_relabeled_grid(*_warm, 0.1); persistence_grid(_warm[0], 4)   # the timed kernel
ck_p2v = Checkpoint("p2_scaling_v2"); SKIPPED_V2 = []
for n in CONFIG["p2_sizes"]:
    if ck_p2v.done(n):
        continue
    need = (BYTES_PER_VERTEX_GRID + 40) * n * 1.15
    if need > 0.85 * psutil.virtual_memory().available:
        SKIPPED_V2.append(n); print(f"skip n={n}: needs ~{need/1e9:.1f} GB"); continue
    H = int(round(math.sqrt(n))); W = int(round(n / H))
    ell, u = scaling_field(rng_for("p2", n), H, W)
    rows = []
    for rep in range(CONFIG["p2v2_reps"]):
        gc.collect()
        cert, tm = certify_relabeled_grid(ell, u, 0.1, conn=4, return_timing=True)
        ef = np.ascontiguousarray(ell.ravel())
        t0 = time.perf_counter(); o = order_of(ef); t1 = time.perf_counter(); _persistence_grid(H, W, 4, ef, o); t2 = time.perf_counter()
        rows.append(dict(n=H * W, rep=rep, T=cert.T, **{f"v2_{k}": v for k, v in tm.items()},
                         pers_sort=t1 - t0, pers_sweep=t2 - t1, pers_total=t2 - t0))
        del cert
    ck_p2v.add(n, rows)
    print(f"n={H*W:>10d}  relabelled total {np.median([r['v2_t_total'] for r in rows]):.3f}s  persistence {np.median([r['pers_total'] for r in rows]):.3f}s")
    del ell, u; gc.collect()
p2v_runs = ck_p2v.frame(); p2v_runs.to_csv(RESULTS / "raw" / "p2_scaling_v2_runs.csv", index=False)
p2v = p2v_runs.groupby("n").median(numeric_only=True).reset_index().drop(columns=["rep"])
p2v = p2v.merge(p2m[["n", "t_total", "t_sweep", "T"]].rename(columns={"T": "T_p2"}), on="n", how="left")
p2v["overhead_original"] = p2v.t_total / p2v.pers_total; p2v["overhead_relabelled"] = p2v.v2_t_total / p2v.pers_total
p2v.to_csv(RESULTS / "processed" / "scaling_v2.csv", index=False)
SLOPES_V2 = {k: loglog_slope(p2v.n, p2v[k]) for k in ["v2_t_total", "v2_t_sweep", "v2_t_relabel", "pers_total", "pers_sweep"]} if len(p2v) >= 2 else {}
V2_CONSISTENT = bool((p2v["T"] == p2v["T_p2"]).all()) if len(p2v) else False
print(p2v[["n", "t_total", "v2_t_total", "v2_t_sweep", "t_sweep", "pers_total", "overhead_original", "overhead_relabelled"]].round(4).to_string(index=False))
print("slopes (post hoc):", {k: round(v, 3) for k, v in SLOPES_V2.items()}, "| same certificate as P2:", V2_CONSISTENT)
set_status("scaling_v2", "EXECUTED", f"post hoc; skipped {SKIPPED_V2}")

n=     10000  relabelled total 0.002s  persistence 0.002s
n=     29929  relabelled total 0.007s  persistence 0.005s
n=     99856  relabelled total 0.027s  persistence 0.018s
n=    299756  relabelled total 0.091s  persistence 0.064s
n=   1000000  relabelled total 0.457s  persistence 0.255s
n=   2999824  relabelled total 1.734s  persistence 1.010s
n=  10001406  relabelled total 8.144s  persistence 4.986s
n=  19998784  relabelled total 18.775s  persistence 11.282s
n=  49999041  relabelled total 53.732s  persistence 32.501s
       n  t_total  v2_t_total  v2_t_sweep  t_sweep  pers_total  overhead_original  overhead_relabelled
   10000   0.0020      0.0023      0.0008   0.0010      0.0015             1.3244               1.4762
   29929   0.0065      0.0070      0.0024   0.0031      0.0050             1.2888               1.3953
   99856   0.0238      0.0271      0.0094   0.0112      0.0179             1.3317               1.5153
  299756   0.0786      0.0914      0.0330   0.0348      0.0639

In [35]:
def cache_sizes():
    """Cache sizes (bytes) of cpu0 from /sys; empty if unavailable."""
    out = {}; base = Path("/sys/devices/system/cpu/cpu0/cache")
    for d in sorted(base.glob("index*")) if base.exists() else []:
        try:
            lvl = (d / "level").read_text().strip(); typ = (d / "type").read_text().strip(); size = (d / "size").read_text().strip()
            mult = {"K": 1024, "M": 1024 ** 2, "G": 1024 ** 3}
            b = int(size[:-1]) * mult[size[-1]] if size[-1] in mult else int(size)
            out[f"L{lvl}{'' if typ == 'Unified' else typ[0].lower()}"] = b
        except Exception:
            pass
    return out

@njit
def _gather(x, idx):
    s = 0.0
    for i in range(idx.shape[0]):
        s += x[idx[i]]
    return s

def gather_ns_per_access(n, rng, reps=3, min_accesses=2e7):
    """Median ns per access of sum(x[idx]) for a random permutation idx and for sequential idx. Small sizes repeat the pass so that every
    timing covers at least `min_accesses` accesses (timer resolution)."""
    x = rng.random(n); out = {}; inner = max(1, int(min_accesses // n))
    for name, idx in (("random", rng.permutation(n).astype(np.int64)), ("sequential", np.arange(n, dtype=np.int64))):
        _gather(x[:16], idx[:16] % 16); _gather(x, idx); ts = []          # compile, then one untimed pass to warm the cache
        for _ in range(reps):
            t0 = time.perf_counter()
            for _i in range(inner):
                _gather(x, idx)
            ts.append(time.perf_counter() - t0)
        out[name] = float(np.median(ts)) / (n * inner) * 1e9
        del idx
    del x; gc.collect()
    return out

CACHES = cache_sizes(); LLC = max([v for k, v in CACHES.items() if k.startswith(("L2", "L3"))], default=None)
ck_mem = Checkpoint("v4_p2_memory")
for n in CONFIG["p2_sizes"]:
    if ck_mem.done(n):
        continue
    if 16 * n * 1.2 > 0.85 * psutil.virtual_memory().available:
        print(f"skip gather n={n}"); continue
    g_ns = gather_ns_per_access(int(n), rng_for("gather", n), CONFIG["p2_mem_reps"])
    ck_mem.add(n, [dict(n=int(n), gather_random_ns=g_ns["random"], gather_sequential_ns=g_ns["sequential"])])
mem = ck_mem.frame()                          # P2 grids have H*W vertices close to the nominal n: match the nearest size
mem = pd.merge_asof(mem.sort_values("n"), p2m[["n", "t_total", "ops"]].rename(columns={"n": "n_p2", "t_total": "t_cert", "ops": "ops_p2"}).sort_values("n_p2"),
                    left_on="n", right_on="n_p2", direction="nearest")
mem = pd.merge_asof(mem.sort_values("n"), p2v[["n", "pers_total"]].rename(columns={"n": "n_p2v"}).sort_values("n_p2v"),
                    left_on="n", right_on="n_p2v", direction="nearest")
mem["cert_ns_per_vertex"] = mem.t_cert / mem.n_p2 * 1e9; mem["pers_ns_per_vertex"] = mem.pers_total / mem.n_p2v * 1e9
mem["uf_ops_per_vertex"] = mem.ops_p2 / mem.n_p2; mem["working_set_bytes"] = BYTES_PER_VERTEX_GRID * mem.n_p2
mem["in_llc"] = (mem.working_set_bytes <= LLC) if LLC else False
mem = mem[((mem.n_p2 - mem.n).abs() <= 0.05 * mem.n) & ((mem.n_p2v - mem.n).abs() <= 0.05 * mem.n)]     # drop sizes skipped in the scaling runs
mem = mem[["n", "n_p2", "working_set_bytes", "in_llc", "cert_ns_per_vertex", "pers_ns_per_vertex", "uf_ops_per_vertex",
           "gather_random_ns", "gather_sequential_ns"]].reset_index(drop=True)
mem.to_csv(RESULTS / "processed" / "p2_memory_hierarchy.csv", index=False)

def _slope(df, col):
    return loglog_slope(df.n_p2, df[col] * df.n_p2) if len(df) >= 2 else float("nan")
inn, out_ = mem[mem.in_llc], mem[~mem.in_llc]
P2_MEM = dict(caches=CACHES, llc_bytes=LLC, sizes_in_llc=int(len(inn)), sizes_out_llc=int(len(out_)),
              slope_cert_in=_slope(inn, "cert_ns_per_vertex"), slope_cert_out=_slope(out_, "cert_ns_per_vertex"),
              slope_pers_in=_slope(inn, "pers_ns_per_vertex"), slope_pers_out=_slope(out_, "pers_ns_per_vertex"),
              gather_random_ns_min=float(mem.gather_random_ns.min()), gather_random_ns_max=float(mem.gather_random_ns.max()),
              uf_ops_per_vertex_range=[float(mem.uf_ops_per_vertex.min()), float(mem.uf_ops_per_vertex.max())])
lo_s, hi_s = CONFIG["p2_slope_range"]
P2_MEM_VERDICT = ("SUPPORTED (post hoc): in-cache slope in the pre-registered range; the excess appears out of cache, is shared by "
                  "plain persistence, and tracks random-access latency"
                  if (LLC and len(inn) >= 2 and lo_s <= P2_MEM["slope_cert_in"] <= hi_s and len(out_) >= 2
                      and abs(P2_MEM["slope_cert_out"] - P2_MEM["slope_pers_out"]) <= 0.15
                      and P2_MEM["gather_random_ns_max"] >= 2 * P2_MEM["gather_random_ns_min"])
                  else "INCONCLUSIVE (post hoc): " + ("no LLC size reported" if not LLC else
                                                      f"{len(inn)} sizes inside and {len(out_)} outside the LLC; need 2 of each and the slope criteria"))
print(mem.round(3).to_string(index=False)); print("memory hierarchy:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in P2_MEM.items()})
print("P2, post hoc diagnosis:", P2_MEM_VERDICT, "| pre-registered verdict unchanged:", P2_VERDICT)
set_status("p2_memory_hierarchy", "EXECUTED", "post hoc")

       n     n_p2  working_set_bytes  in_llc  cert_ns_per_vertex  pers_ns_per_vertex  uf_ops_per_vertex  gather_random_ns  gather_sequential_ns
   10000    10000            1620000    True             202.108             152.605              3.654             1.319                 1.327
   30000    29929            4848498    True             216.211             167.758              3.643             1.306                 1.331
  100000    99856           16176672    True             238.835             179.351              3.752             1.339                 1.348
  300000   299756           48560472   False             262.206             213.320              3.778             2.133                 1.334
 1000000  1000000          162000000   False             413.039             254.720              3.821             2.670                 1.289
 3000000  2999824          485971488   False             584.102             336.852              3.842             7.811               

## 20 · Lossy compression (Section 7, Table 3, Appendix D)

An error-bounded compressor guarantees that the original field lies in a box around the decompressed field $\tilde f$. The bound therefore certifies degree-0 bars of the original without changing the compressor. The section runs four modes:
- SZ, SZ3 and ZFP with absolute bounds;
- SZ3 on log-magnitudes, which gives a pointwise relative bound and hence a non-uniform box.

Each field is compressed once per mode and bound, and certified at τ equal to 0.5% and 2% of its range. Fill values are replaced by the largest valid value, and the fraction replaced is recorded.

**Uniform-error baseline.** With an absolute bound the box is $\tilde f\pm\varepsilon$. Its image barcode is $\mathrm{Bar}_0(\tilde f)$ with ε cut from both ends of every bar, the smoothing construction of Chubet, Gardner and Sheehy, so the certified count equals the number of bars of $\tilde f$ longer than $\tau+2\varepsilon$. The second cell checks this identity on every absolute-mode run. For the relative mode it compares the certificate with the one from the smallest uniform box containing the non-uniform box. That certificate is valid but weaker, and the comparison measures what non-uniform boxes gain.

In [36]:
def grf_nd(rng, shape, corr_len):
    noise = rng.standard_normal(shape)
    freqs = [np.fft.fftfreq(s) for s in shape[:-1]] + [np.fft.rfftfreq(shape[-1])]
    k2 = sum(g_ ** 2 for g_ in np.meshgrid(*freqs, indexing="ij"))
    ax_ = tuple(range(len(shape)))
    F = np.fft.irfftn(np.fft.rfftn(noise, axes=ax_) * np.exp(-2.0 * (np.pi * corr_len) ** 2 * k2), s=shape, axes=ax_)
    return (F - F.mean()) / F.std()

try:
    _ensure("h5py"); _ensure("hdf5plugin")
    import h5py, hdf5plugin
    HAS_H5PLUGIN = True
    ENV["h5py"] = h5py.__version__; ENV["hdf5plugin"] = getattr(hdf5plugin, "version", "unknown")
except Exception as exc:
    HAS_H5PLUGIN = False
    print("hdf5plugin unavailable -> in-notebook error-bounded quantiser fallback:", exc)

@njit
def lorenzo_quantize(x, eps):
    """Fallback only: SZ-style 2-D Lorenzo prediction + linear quantisation with bound eps."""
    H, W = x.shape
    rec = np.empty_like(x)
    for i in range(H):
        for j in range(W):
            a = rec[i - 1, j] if i > 0 else 0.0
            b = rec[i, j - 1] if j > 0 else 0.0
            c = rec[i - 1, j - 1] if (i > 0 and j > 0) else 0.0
            pred = a + b - c
            val = pred + 2.0 * eps * np.round((x[i, j] - pred) / (2.0 * eps))
            rec[i, j] = val if abs(val - x[i, j]) <= eps else x[i, j]
    return rec

def _h5_roundtrip(arr, filt):
    bio = io.BytesIO()
    with h5py.File(bio, "w") as fh5:
        d = fh5.create_dataset("x", data=arr, chunks=arr.shape, **filt)
        nbytes = d.id.get_storage_size()
    with h5py.File(bio, "r") as fh5:
        out = fh5["x"][...]
    return out, nbytes

def compress_roundtrip(x32, method, bound):
    """Returns reconstruction (float64), certified box lo/hi, achieved error statistic, bits/value, seconds."""
    t0 = time.perf_counter(); xf = x32.astype(np.float64)
    if method in ("SZ3", "SZ", "ZFP") and HAS_H5PLUGIN:
        filt = {"SZ3": hdf5plugin.SZ3(absolute=bound), "SZ": hdf5plugin.SZ(absolute=bound), "ZFP": hdf5plugin.Zfp(accuracy=bound)}[method]
        rec32, nbytes = _h5_roundtrip(x32, filt)
        rec = rec32.astype(np.float64); eps = bound * (1 + 1e-6)
        lo, hi = rec - eps, rec + eps; err = float(np.max(np.abs(rec - xf))); bits = 8.0 * nbytes / x32.size
    elif method == "SZ3-pwrel" and HAS_H5PLUGIN:
        delta = math.log1p(bound) * (1 + 1e-9); mag = np.abs(xf); nz = mag > 0; sgn = np.sign(xf)
        lg = np.where(nz, np.log(np.where(nz, mag, 1.0)), 0.0)
        rl, nbytes = _h5_roundtrip(lg, hdf5plugin.SZ3(absolute=math.log1p(bound)))
        rec = np.where(nz, sgn * np.exp(rl), 0.0)
        lo = np.where(rec >= 0, rec * math.exp(-delta), rec * math.exp(delta)); hi = np.where(rec >= 0, rec * math.exp(delta), rec * math.exp(-delta))
        with np.errstate(divide="ignore", invalid="ignore"):
            err = float(np.max(np.where(nz, np.abs(rec - xf) / mag, 0.0)))
        bits = (8.0 * nbytes + 2.0 * x32.size) / x32.size          # + zero mask and sign bit per value
    elif method == "EBQ-Lorenzo" and x32.ndim == 2:
        rec = lorenzo_quantize(xf, bound); lo, hi = rec - bound, rec + bound; err = float(np.max(np.abs(rec - xf))); bits = float("nan")
    else:
        raise ValueError(f"method {method} unavailable")
    return rec, lo, hi, err, bits, time.perf_counter() - t0

def clean_field(x):
    bad = ~np.isfinite(x) | (np.abs(x) > 1e30)
    if bad.any():
        x = x.copy(); x[bad] = x[~bad].max()
    return x, float(bad.mean())

def compression_rows(name, x32, conn, kind):
    cfg = CONFIG["compression"]; x32 = np.ascontiguousarray(x32, np.float32)
    xf = x32.astype(np.float64); rng_ = float(np.ptp(xf))
    f_flat, d0, i0 = persistence_nd(xf, conn)
    ntrue = {tf: int(np.sum(i0 & ((d0 - f_flat) > tf * rng_))) for tf in cfg["tau_fracs"]}
    methods = cfg["methods"] if HAS_H5PLUGIN else ["EBQ-Lorenzo"]
    rows = []
    for method in methods:
        for eb in CONFIG["comp_rel_bounds"]:
            bound = eb if method == "SZ3-pwrel" else eb * rng_
            try:
                rec, lo, hi, err, bits, secs = compress_roundtrip(x32, method, bound)
            except Exception as exc:
                rows.append(dict(dataset=name, kind=kind, method=method, rel_bound=eb, error=f"{type(exc).__name__}: {exc}"))
                continue
            respected = bool(np.all(xf >= lo) and np.all(xf <= hi))
            eps_u = float(np.max(np.maximum(hi - rec, rec - lo)))        # half-width of the smallest uniform box rec +- eps containing [lo, hi]
            for tf in cfg["tau_fracs"]:
                tau = tf * rng_
                t0 = time.perf_counter(); c = certify_nd(lo.reshape(x32.shape), hi.reshape(x32.shape), tau, conn); t_cert = time.perf_counter() - t0
                naive = n_tau_nd(rec.reshape(x32.shape), tau, conn); l1 = lemma1_check(c, f_flat, d0, i0)
                t_unif = n_tau_nd(rec.reshape(x32.shape), tau + 2.0 * eps_u, conn)       # uniform-error baseline (Section 20)
                rows.append(dict(dataset=name, kind=kind, shape="x".join(map(str, x32.shape)), n=int(x32.size), method=method,
                                 rel_bound=eb, tau_frac=tf, N_true=ntrue[tf], T=c.T, T_uniform=t_unif, eps_uniform=eps_u, N_naive=naive,
                                 overcount=max(0, naive - ntrue[tf]), undercount=max(0, ntrue[tf] - naive),
                                 cert_fraction=c.T / max(ntrue[tf], 1), valid=int(ntrue[tf] >= c.T and l1 == c.T),
                                 bound_respected=respected, error_stat=err, bits_per_value=bits, t_compress=secs, t_certify=t_cert))
    return rows

ck_cmp = Checkpoint("compression_v6")
rngc = rng_for("comp_synth"); cc = CONFIG["compression"]
for k in range(CONFIG["comp_synth_2d"]):
    x = (grf_nd(rngc, CONFIG["comp_shape2d"], 6.0) * 10).astype(np.float32)
    if not ck_cmp.done(f"synthetic-2D-{k}"):
        ck_cmp.add(f"synthetic-2D-{k}", compression_rows(f"synthetic-2D-{k}", x, cc["conn2d"], "synthetic"))
for k in range(CONFIG["comp_synth_3d"]):
    x = (grf_nd(rngc, CONFIG["comp_shape3d"], 4.0) * 10).astype(np.float32)
    if not ck_cmp.done(f"synthetic-3D-{k}"):
        ck_cmp.add(f"synthetic-3D-{k}", compression_rows(f"synthetic-3D-{k}", x, cc["conn3d"], "synthetic"))
set_status("compression_synthetic", "EXECUTED", f"{CONFIG['comp_synth_2d']} 2-D + {CONFIG['comp_synth_3d']} 3-D fields")

SDRB_EXECUTED = []
if CONFIG["run_compression"]:
    for ds in cc["sdrbench"]:
        try:
            dest = RESULTS / "data" / "sdrbench" / ds["name"]; nbytes = int(np.prod(ds["shape"])) * 4
            files = [p for p in sorted(dest.glob("*")) if p.is_file() and p.stat().st_size == nbytes][:ds["n_fields"]]
            if len(files) < ds["n_fields"]:
                files = stream_tar_members(ds["url"], dest, ds["n_fields"], accept_bytes=nbytes, max_bytes=cc["max_download_gb"] * 1e9)
            if not files:
                raise RuntimeError("no member of the expected size in the archive")
            for fpath in files:
                record_manifest(f"sdrbench:{ds['name']}:{Path(fpath).name}", fpath)
                key = f"{ds['name']}:{Path(fpath).name}"
                if ck_cmp.done(key):
                    continue
                x, miss = clean_field(np.fromfile(fpath, dtype="<f4").reshape(ds["shape"]))
                conn = cc["conn3d"] if len(ds["shape"]) == 3 else cc["conn2d"]
                rows = compression_rows(key, x, conn, "sdrbench")
                for r in rows:
                    r["missing_fraction"] = miss
                ck_cmp.add(key, rows)
            SDRB_EXECUTED.append(ds["name"])
        except Exception as exc:
            print(f"SDRBench {ds['name']}: {type(exc).__name__}: {exc}")
    set_status("real_compression", "EXECUTED" if SDRB_EXECUTED else "NOT EXECUTED",
               f"SDRBench datasets executed: {SDRB_EXECUTED or 'none'}")
else:
    set_status("real_compression", "NOT EXECUTED", f"disabled in profile {PROFILE}")

comp = ck_cmp.frame()
comp.to_csv(RESULTS / "raw" / "compression.csv", index=False)
COMP_RESULT = comp
compv = comp[comp.get("T", pd.Series(dtype=float)).notna()] if "T" in comp else comp.iloc[0:0]
comps = (compv.groupby(["kind", "method", "rel_bound", "tau_frac"])
         .agg(runs=("T", "size"), bound_respected=("bound_respected", "mean"), valid=("valid", "mean"),
              cert_fraction=("cert_fraction", "mean"), naive_overcount=("overcount", "mean"), naive_undercount=("undercount", "mean"),
              bits_per_value=("bits_per_value", "mean"), t_compress=("t_compress", "mean"), t_certify=("t_certify", "mean"))
         .reset_index()) if len(compv) else pd.DataFrame()
comps.to_csv(RESULTS / "processed" / "compression_summary.csv", index=False)
resp = compv[compv.bound_respected.astype(bool)] if len(compv) else compv
COMP_STATS = dict(runs=int(len(compv)), bound_violations=int((~compv.bound_respected.astype(bool)).sum()) if len(compv) else 0,
                  valid_given_bound=int(resp.valid.sum()) if len(resp) else 0, runs_with_bound=int(len(resp)),
                  naive_overcount_runs=int((compv.overcount > 0).sum()) if len(compv) else 0)
COMP_VERDICT = ("SUPPORTED" if COMP_STATS["runs_with_bound"] and COMP_STATS["valid_given_bound"] == COMP_STATS["runs_with_bound"]
                else ("NOT SUPPORTED" if COMP_STATS["runs_with_bound"] else "INCONCLUSIVE"))
print(comps.round(4).to_string(index=False) if len(comps) else "no compression rows"); print("compression:", COMP_STATS, COMP_VERDICT)

[STATUS] compression_synthetic: EXECUTED — 4 2-D + 2 3-D fields
[STATUS] real_compression: EXECUTED — SDRBench datasets executed: ['CESM-ATM', 'Hurricane-ISABEL-Pf']
     kind    method  rel_bound  tau_frac  runs  bound_respected  valid  cert_fraction  naive_overcount  naive_undercount  bits_per_value  t_compress  t_certify
 sdrbench        SZ     0.0001     0.005     5              1.0    1.0         0.9694           3.6000            0.0000          2.3324      0.7020     4.1488
 sdrbench        SZ     0.0001     0.020     5              1.0    1.0         0.9863           0.0000            0.4000          2.3324      0.7020     4.0781
 sdrbench        SZ     0.0010     0.005     5              1.0    1.0         0.7489          55.0000            0.0000          0.8633      0.5891     3.9578
 sdrbench        SZ     0.0010     0.020     5              1.0    1.0         0.8685           1.2000            0.2000          0.8633      0.5891     3.9801
 sdrbench        SZ     0.0100    

In [37]:
# Uniform-error baseline. With an absolute bound the certified box is rec ± eps, and the image barcode is Bar_0(rec) with eps cut from
# both ends of every bar (the smoothing construction of Chubet, Gardner and Sheehy). So T equals the number of bars of the decompressed
# field longer than tau + 2 eps, which plain persistence computes. The relative mode gives a non-uniform box; its smallest enclosing
# uniform box rec ± max(hi - rec, rec - lo) is valid but weaker, and by Theorem 5 its count can only be smaller.
compv = compv.copy()
_abs = compv.method.isin(["SZ3", "SZ", "ZFP", "EBQ-Lorenzo"]); _rel = compv.method == "SZ3-pwrel"; _sdr = compv.kind == "sdrbench"
COMP_UNIFORM = dict(
    absolute_runs=int(_abs.sum()), absolute_identity_holds=int((compv[_abs]["T"] == compv[_abs]["T_uniform"]).sum()),
    relative_runs=int(_rel.sum()), relative_T_above_uniform=int((compv[_rel]["T"] > compv[_rel]["T_uniform"]).sum()),
    monotonicity_violations=int((compv["T"] < compv["T_uniform"]).sum()),
    relative_mean_T=float(compv[_rel]["T"].mean()) if _rel.any() else None,
    relative_mean_T_uniform=float(compv[_rel]["T_uniform"].mean()) if _rel.any() else None,
    relative_mean_N_true=float(compv[_rel]["N_true"].mean()) if _rel.any() else None)
comp_rel_tab = (compv[_rel].groupby(["kind", "rel_bound", "tau_frac"])
                .agg(runs=("T", "size"), N_true=("N_true", "mean"), T=("T", "mean"), T_uniform=("T_uniform", "mean"),
                     certified_fraction=("cert_fraction", "mean"),
                     uniform_fraction=("T_uniform", lambda s: float((s / compv.loc[s.index, "N_true"].clip(lower=1)).mean())))
                .reset_index()) if _rel.any() else pd.DataFrame()

# Counts quoted in the paper and in the reviews: compressions versus certificate evaluations, naive over- and undercounts, timings.
_keys = ["dataset", "method", "rel_bound"]
COMP_COUNTS = dict(
    fields=int(compv.dataset.nunique()), compressions=int(len(compv.drop_duplicates(_keys))), certificate_evaluations=int(len(compv)),
    sdrbench_evaluations=int(_sdr.sum()), overcount_runs=int((compv.overcount > 0).sum()),
    overcount_runs_sdrbench=int((compv[_sdr].overcount > 0).sum()), undercount_runs=int((compv.undercount > 0).sum()),
    undercount_runs_sdrbench=int((compv[_sdr].undercount > 0).sum()),
    bound_respected=int(compv.bound_respected.astype(bool).sum()), valid=int(compv.valid.astype(bool).sum()))
_tcert = compv.groupby(["dataset", "n"]).t_certify.median().reset_index()
COMP_TIMING = {r.dataset: dict(n=int(r.n), median_certify_seconds=round(float(r.t_certify), 3)) for r in _tcert.itertuples()}
COMP_MISSING = ({d: float(v) for d, v in compv[_sdr].groupby("dataset").missing_fraction.first().items()}
                if "missing_fraction" in compv and _sdr.any() else {})
# Figure 3(c) and the text: SDRBench at tau = 0.5% of the range
_s5 = compv[_sdr & (compv.tau_frac == 0.005)]
COMP_FIG = dict(
    sdr_mean_true_bars=float(_s5.drop_duplicates("dataset").N_true.mean()) if len(_s5) else None,
    certified_fraction=(_s5.groupby(["method", "rel_bound"]).cert_fraction.mean().round(4).reset_index()
                        .assign(key=lambda d: d.method + "@" + d.rel_bound.map(lambda x: f"{x:g}"))
                        .set_index("key").cert_fraction.to_dict()) if len(_s5) else {},
    naive_excess=(_s5.assign(excess=_s5.N_naive - _s5.N_true).groupby(["method", "rel_bound"]).excess.mean().round(1).reset_index()
                  .assign(key=lambda d: d.method + "@" + d.rel_bound.map(lambda x: f"{x:g}")).set_index("key").excess.to_dict())
                 if len(_s5) else {})
print(comp_rel_tab.round(4).to_string(index=False) if len(comp_rel_tab) else "no relative-mode rows")
print("uniform baseline:", COMP_UNIFORM); print("counts:", COMP_COUNTS); print("median certification seconds:", COMP_TIMING)
print("missing (fill) fractions:", COMP_MISSING); print("Figure 3(c):", COMP_FIG)
COMP_UNIFORM_VERDICT = ("SUPPORTED" if COMP_UNIFORM["absolute_identity_holds"] == COMP_UNIFORM["absolute_runs"]
                        and COMP_UNIFORM["monotonicity_violations"] == 0 else "NOT SUPPORTED")
comp_rel_tab.to_csv(RESULTS / "processed" / "compression_uniform_baseline.csv", index=False)
set_status("compression_uniform_baseline", "EXECUTED", COMP_UNIFORM_VERDICT)

     kind  rel_bound  tau_frac  runs    N_true         T  T_uniform  certified_fraction  uniform_fraction
 sdrbench     0.0001     0.005     5 2055.8000 2036.8000  1997.2000              0.9911            0.9737
 sdrbench     0.0001     0.020     5  459.0000  456.8000   453.6000              0.9979            0.9880
 sdrbench     0.0010     0.005     5 2055.8000 1884.8000  1546.2000              0.9245            0.7577
 sdrbench     0.0010     0.020     5  459.0000  444.2000   411.6000              0.9698            0.8861
 sdrbench     0.0100     0.005     5 2055.8000 1199.0000   405.8000              0.5950            0.2078
 sdrbench     0.0100     0.020     5  459.0000  358.4000   200.2000              0.7384            0.3931
synthetic     0.0001     0.005     6   96.8333   96.8333    96.6667              1.0000            0.9984
synthetic     0.0001     0.020     6   77.8333   77.8333    77.6667              1.0000            0.9977
synthetic     0.0010     0.005     6   96.8333

## 21 · Export and validation

Top-level CSV files (read by the figure script), `results.json`, and a validation pass that repeats every verdict. Sections that could not run (for example SDRBench without Internet access) are listed as warnings, not hidden.

In [38]:
shutil.copy(RESULTS / "processed" / "scaling.csv", RESULTS / "scaling.csv")
shutil.copy(RESULTS / "configs" / "configuration.json", RESULTS / "configuration.json")
for _df, _name in [(opt, "optimality.csv"), (optb, "optimality_regions.csv"), (d1x_cls, "degree1_census.csv"),
                   (d1x_cen, "degree1_census_instances.csv"), (d1x_how, "degree1_census_methods.csv"), (d1x_pl, "degree1_planar.csv"),
                   (transfers_df, "degree1_transfers.csv"), (thmc_s, "theorem_c.csv"), (d1f_ts, "degree1_fast_timing.csv"),
                   (mem, "p2_memory_hierarchy.csv"), (comp, "compression.csv"), (comp_rel_tab, "compression_uniform_baseline.csv"),
                   (t13_df, "theorem13.csv"), (md_df, "middle_degrees.csv"), (g3d_s, "grid3d_degree2.csv"), (g3c_s, "grid3d_degree1_census.csv"),
                   (grid20, "example20_grid.csv")]:
    _df.to_csv(RESULTS / _name, index=False)

WARN = []
def check(cond, msg):
    if not bool(cond):
        WARN.append(msg); print("VALIDATION WARNING:", msg)

check(ALL_UNIT_PASS, "some unit tests failed")
check(GKLS["matches"] == GKLS["instances"], "tau = 0 identity failed")
check(OPT_VERDICT == "SUPPORTED", f"degree-0 attainment check failed: {OPT}")
check(CGS_VERDICT == "SUPPORTED", "an image barcode was not a sub-barcode of a sampled realisation")
check((p2.t_total > 0).all() and np.isfinite(p2.t_total).all(), "non-positive or non-finite runtime")
check(p2.groupby("n")["T"].nunique().max() == 1, "certificate changed across timing repetitions (non-determinism)")
check(np.isfinite(SLOPES["t_total"]), "scaling slope not finite")
check(V2_CONSISTENT, "relabelled sweep produced a different certificate")
check(len(mem) >= 2 and mem.uf_ops_per_vertex.notna().all(), "memory-hierarchy table incomplete")
check(COMP_STATS["bound_violations"] == 0, f"{COMP_STATS['bound_violations']} compression runs violated the stated bound")
check(COMP_STATS["valid_given_bound"] == COMP_STATS["runs_with_bound"], "invalid compression certificate despite a respected bound")
check(COMP_UNIFORM_VERDICT == "SUPPORTED", f"uniform-error identity or monotonicity failed: {COMP_UNIFORM}")
check(DEG1_CHECKS["zoo_ok"] and DEG1_CHECKS["rank_vs_reduction_mismatch"] == 0 and DEG1_CHECKS["image0_vs_sweep_mismatch"] == 0
      and DEG1_CHECKS["negative_multiplicity"] == 0, f"rank-invariant self-checks failed: {DEG1_CHECKS}")
check(D1X_VERDICT_MACHINERY == "SUPPORTED", f"degree-1 exact machinery self-checks failed: {D1X_CHECKS}")
check(CENSUS["witness_failures"] == 0 and CENSUS["undecided"] == 0, "census: a witness failed re-verification or an instance was undecided")
check(CENSUS_METHODS["reconciles"], "census decision methods do not add up to the number of instances")
check(PLANAR_VERDICT == "SUPPORTED", f"Theorem 8 (planar) failed: {PLANAR}")
check(D1X_VERDICT_CERT == "SUPPORTED", "a degree-1 certificate or transfer did not reproduce its hand proof under exhaustive enumeration")
check(EX20_VERDICT == "SUPPORTED", f"Example 20 not confirmed: {EX20}")
check(KC_LEMMA_VERDICT == "SUPPORTED", f"Lemma 16 check failed: {KC_LEMMA}")
check(THMC_VERDICT == "SUPPORTED", f"Theorem 10 check failed: {THMC}")
check(PROPG_VERDICT == "SUPPORTED", f"Proposition 15 check failed: {PROPG}")
check(D1FAST_VERDICT == "SUPPORTED", f"Corollary 14 check failed: {D1FAST}")
check(T13_VERDICT == "SUPPORTED", f"Theorem 13 / Lemma 12 check failed or did not reproduce the reference counts: {T13}")
check(MIDDLE_VERDICT == "SUPPORTED", f"middle-degree constructions not confirmed: {MIDDLE}")
check(G3_VERDICT == "SUPPORTED", f"3-D grid checks failed: {G3_DUAL}, {G3_CENSUS}")
for name, st in STATUS.items():
    if st["status"] != "EXECUTED":
        WARN.append(f"{name}: {st['status']} ({st['reason']})")
save_json(dict(warnings=WARN), RESULTS / "logs" / "validation.json")

RESULTS_JSON = dict(
    profile=PROFILE, config_hash=CONFIG_HASH, environment=ENV, status=STATUS, unit_tests=unit_df.to_dict("records"),
    degree0=dict(tau0_identity=dict(verdict=GKLS_VERDICT, **GKLS), optimality=dict(verdict=OPT_VERDICT, **OPT),
                 subbarcode_validity=CGS_VERDICT),
    degree1=dict(machinery=D1X_CHECKS, kc_lemma=dict(verdict=KC_LEMMA_VERDICT, **KC_LEMMA), planar=dict(verdict=PLANAR_VERDICT, **PLANAR),
                 census=dict(summary=CENSUS, methods=CENSUS_METHODS, table=d1x_cls.to_dict("records")),
                 certificates={k: {kk: vv for kk, vv in C.items() if kk != "sweep"} for k, C in CERTS.items()},
                 transfers=TRANSFERS, certificates_verdict=D1X_VERDICT_CERT, count_note=D1X_COUNT_NOTE,
                 example20=dict(verdict=EX20_VERDICT, **EX20)),
    surfaces=dict(theorem10=dict(verdict=THMC_VERDICT, **THMC, table=thmc_s.to_dict("records")),
                  corollary14=dict(verdict=D1FAST_VERDICT, **D1FAST, timing=d1f_ts.to_dict("records")),
                  theorem13=dict(verdict=T13_VERDICT, **T13, table=t13_df.to_dict("records"))),
    graphs=dict(verdict=PROPG_VERDICT, **PROPG, table=pg_s.to_dict("records")),
    middle_degrees=dict(verdict=MIDDLE_VERDICT, **MIDDLE, rows=MD_ROWS),
    grids3d=dict(verdict=G3_VERDICT, degree2=G3_DUAL, degree1=G3_CENSUS),
    scaling=dict(slopes=SLOPES, verdict=P2_VERDICT, note=P2_NOTE, skipped_sizes=SKIPPED_SIZES,
                 relabelled=dict(slopes=SLOPES_V2, consistent=V2_CONSISTENT, skipped=SKIPPED_V2),
                 memory=dict(verdict=P2_MEM_VERDICT, **P2_MEM)),
    compression=dict(verdict=COMP_VERDICT, stats=COMP_STATS, counts=COMP_COUNTS, uniform_baseline=dict(verdict=COMP_UNIFORM_VERDICT, **COMP_UNIFORM),
                     sdrbench_executed=SDRB_EXECUTED, summary=comps.to_dict("records") if len(comps) else []),
    validation_warnings=WARN)
save_json(RESULTS_JSON, RESULTS / "results.json")
print("validation:", "no warnings" if not WARN else WARN)
print("exported:", sorted(p.name for p in RESULTS.glob("*.*")))

validation: no warnings
exported: ['compression.csv', 'compression_uniform_baseline.csv', 'configuration.json', 'degree1_census.csv', 'degree1_census_instances.csv', 'degree1_census_methods.csv', 'degree1_fast_timing.csv', 'degree1_planar.csv', 'degree1_transfers.csv', 'example20_grid.csv', 'grid3d_degree1_census.csv', 'grid3d_degree2.csv', 'middle_degrees.csv', 'optimality.csv', 'optimality_regions.csv', 'p2_memory_hierarchy.csv', 'results.json', 'scaling.csv', 'theorem13.csv', 'theorem_c.csv']


## 22 · Figures and tables

Figures 1–4 of the paper are drawn from the exported tables by `results/code/make_figures.py`; Figures 1 and 2 are computed from first principles inside the script. Tables 2 and 3 are written as CSV and as LaTeX rows.

In [39]:
# The paper's figures are drawn by a stand-alone script from the exported tables, so they can be regenerated from the results
# bundle alone: python3 results/code/make_figures.py results results/figures
MAKE_FIGURES_SRC = r'''"""Regenerate every figure of the paper as a vector PDF.

Usage:  python3 make_figures.py RESULTS_DIR OUT_DIR

RESULTS_DIR is the results/ folder of the experiment bundle (it must contain
p2_memory_hierarchy.csv, degree1_fast_timing.csv and compression.csv).
Figures 1 and 2 are computed here from first principles (small exact examples).
"""
import sys, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("pdf")
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, FancyArrowPatch

RES = Path(sys.argv[1]); OUT = Path(sys.argv[2]); OUT.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif", "font.serif": ["cmr10"], "mathtext.fontset": "cm",
    "axes.formatter.use_mathtext": True, "axes.unicode_minus": False,
    "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.linewidth": 0.5, "xtick.major.width": 0.5, "ytick.major.width": 0.5,
    "xtick.minor.width": 0.4, "ytick.minor.width": 0.4,
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
})
# categorical slots (validated light-mode order), plus neutral inks
BLUE, ORANGE, AQUA, YELLOW, VIOLET = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#6250d6"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#d9d8d4"
TEXTW = 5.2  # inches, a little under the LIPIcs text width


def barcode0(adj, f):
    """Degree-0 barcode of the lower-star filtration of a graph (union-find, elder rule)."""
    n = len(f); order = sorted(range(n), key=lambda v: (f[v], v))
    parent = list(range(n)); birth = {}; present = [False] * n; bars = []
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for v in order:
        present[v] = True; birth[v] = f[v]
        for w in adj[v]:
            if not present[w]:
                continue
            a, b = find(v), find(w)
            if a == b:
                continue
            young, old = (a, b) if (birth[a], a) > (birth[b], b) else (b, a)
            if birth[young] < f[v]:
                bars.append((birth[young], f[v]))
            parent[young] = old
    roots = {find(v) for v in range(n)}
    bars += [(birth[r], math.inf) for r in roots]
    return sorted(bars)


def f_U(adj, lo, up):
    """Pointwise smallest attaining function: min over w, paths w->v of max(u(w), max of l on the path)."""
    import heapq
    n = len(lo); d = [math.inf] * n; h = []
    for v in range(n):
        d[v] = up[v]; heapq.heappush(h, (d[v], v))
    while h:
        dv, v = heapq.heappop(h)
        if dv > d[v]:
            continue
        for w in adj[v]:
            c = max(dv, lo[w])
            if c < d[w]:
                d[w] = c; heapq.heappush(h, (c, w))
    return d


# ---------------------------------------------------------------- Figure 1: f_U on a path
def figure_path():
    lo = [4, 1, 3, 6, 2, 5, 7, 3, 1, 4, 6, 5, 2, 4]
    up = [6, 2, 7, 8, 5, 6, 9, 7, 3, 8, 7, 9, 6, 6]
    n = len(lo); adj = [[w for w in (v - 1, v + 1) if 0 <= w < n] for v in range(n)]
    fu = f_U(adj, lo, up)
    assert all(l <= x <= u for l, x, u in zip(lo, fu, up))
    bl, bu, bf = barcode0(adj, lo), barcode0(adj, up), barcode0(adj, fu)
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(TEXTW, 1.75), gridspec_kw=dict(width_ratios=[1.55, 1]))
    xs = np.arange(n)
    for v in range(n):
        ax.plot([v, v], [lo[v], up[v]], color=GRID, lw=4.5, solid_capstyle="butt", zorder=1)
    ax.plot(xs, lo, ls=":", lw=0.8, color=INK2, zorder=2, label=r"$\ell$")
    ax.plot(xs, up, ls="--", lw=0.8, color=INK2, zorder=2, label=r"$u$")
    ax.plot(xs, fu, "-o", ms=3.2, lw=1.2, color=BLUE, zorder=3, label=r"$f_U$")
    ax.set_xticks(xs, [str(v) for v in xs]); ax.set_xlabel(r"vertex of the path")
    ax.set_ylabel(r"value"); ax.set_ylim(0, 10); ax.set_xlim(-0.6, n - 0.4)
    ax.legend(frameon=False, ncol=3, loc="lower left", bbox_to_anchor=(0.0, 1.0), handlelength=1.6,
              columnspacing=1.2, borderaxespad=0.1)
    rows = [(r"$\mathrm{Bar}_0(\ell)$", bl, INK2), (r"$\mathrm{Bar}_0(u)$", bu, INK2),
            (r"$\mathrm{Bar}_0(f_U)=\mathrm{Bar}_0(u\geq\ell)$", bf, BLUE)]
    y = 0; ticks = []
    top = 10.6
    for name, B, col in rows:
        start = y
        for (b, d) in sorted(B, key=lambda p: (p[0], -p[1] if math.isfinite(p[1]) else -99)):
            e = top if not math.isfinite(d) else d
            bx.plot([b, e], [y, y], color=col, lw=1.6, solid_capstyle="butt")
            if not math.isfinite(d):
                bx.annotate("", xy=(top + 0.35, y), xytext=(top - 0.1, y),
                            arrowprops=dict(arrowstyle="-|>", lw=0.8, color=col, shrinkA=0, shrinkB=0, mutation_scale=6))
            y += 1
        ticks.append(((start + y - 1) / 2, name)); y += 1.2
    bx.set_yticks([t for t, _ in ticks], [nm for _, nm in ticks]); bx.invert_yaxis()
    bx.set_xlim(0, top + 0.6); bx.set_xlabel(r"value"); bx.tick_params(axis="y", length=0)
    bx.spines["left"].set_visible(False)
    fig.tight_layout(w_pad=1.0)
    fig.savefig(OUT / "fig1_fU_path.pdf")
    plt.close(fig)
    return dict(lo=lo, up=up, fU=fu, bar_l=bl, bar_u=bu, bar_fU=bf)


# ---------------------------------------------------------------- Figure 2: certificate D (Moebius band)
def figure_certD():
    # vertex ids follow the paper: a, b, c on the boundary (fixed at 0); x, y, z interior
    P = {"b": (0, 2), "p": (0, 1), "c": (0, 0), "q": (2, 1), "a": (3, 2), "r": (4, 1),
         "p'": (6, 1), "c'": (6, 2), "b'": (6, 0)}
    T = [("b", "p", "q"), ("p", "c", "q"), ("a", "b", "q"), ("a", "q", "r"), ("c", "q", "r"),
         ("b'", "c", "r"), ("b'", "p'", "r"), ("a", "p'", "r"), ("a", "p'", "c'")]
    box = {"a": "0", "b": "0", "c": "0", "p": "[1,3]", "q": "[1,4]", "r": "[2,4]"}
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(TEXTW, 1.55), gridspec_kw=dict(width_ratios=[1.25, 1]))
    for t in T:
        ax.add_patch(Polygon([P[v] for v in t], closed=True, facecolor="#eef3fb", edgecolor=INK2, lw=0.6))
    # boundary edges drawn heavier
    for e in [("b", "a"), ("a", "c'"), ("c", "b'")]:
        (x0, y0), (x1, y1) = P[e[0]], P[e[1]]
        ax.plot([x0, x1], [y0, y1], color=INK, lw=1.4, solid_capstyle="round")
    # identified sides: left side read top to bottom, right side bottom to top
    for (xa, ya, xb, yb) in [(-0.32, 1.85, -0.32, 0.15), (6.32, 0.15, 6.32, 1.85)]:
        ax.add_patch(FancyArrowPatch((xa, ya), (xb, yb), arrowstyle="-|>", mutation_scale=7, lw=0.7, color=ORANGE))
    off = {"b": (0.0, 0.3), "p": (-0.72, 0.0), "c": (0.0, -0.3), "q": (0.0, -0.33), "a": (0.0, 0.3),
           "r": (0.0, -0.33), "p'": (0.5, 0.0), "c'": (0.0, 0.3), "b'": (0.0, -0.3)}
    for v, (x, y) in P.items():
        ax.plot([x], [y], "o", ms=3.2, color=INK, zorder=5)
        base = v.rstrip("'")
        lab = rf"${base}$" if v.endswith("'") else rf"${base}\colon {box[base]}$"
        dx, dy = off[v]
        ax.text(x + dx, y + dy, lab, ha="center", va="center", fontsize=7,
                bbox=dict(boxstyle="square,pad=0.08", fc="white", ec="none"), zorder=6)
    ax.set_xlim(-1.35, 7.0); ax.set_ylim(-0.55, 2.55); ax.set_aspect("equal"); ax.axis("off")
    rows = [(r"$\mathrm{Bar}_1(u\geq\ell)$", [(0, 2), (3, math.inf)], BLUE),
            (r"$\mathrm{Bar}_1(f_D)$", [(0, 2), (2, math.inf)], INK2),
            (r"$\mathrm{Bar}_1(u)$", [(0, 4), (3, math.inf)], INK2)]
    top = 5.4; y = 0; ticks = []
    for name, B, col in rows:
        start = y
        for (b, d) in B:
            e = top if not math.isfinite(d) else d
            bx.plot([b, e], [y, y], color=col, lw=1.6, solid_capstyle="butt")
            if not math.isfinite(d):
                bx.annotate("", xy=(top + 0.3, y), xytext=(top - 0.1, y),
                            arrowprops=dict(arrowstyle="-|>", lw=0.8, color=col, shrinkA=0, shrinkB=0, mutation_scale=6))
            y += 1
        ticks.append(((start + y - 1) / 2, name)); y += 0.9
    bx.set_yticks([t for t, _ in ticks], [nm for _, nm in ticks]); bx.invert_yaxis()
    bx.set_xlim(0, top + 0.5); bx.set_xticks(range(0, 6)); bx.set_xlabel(r"value")
    bx.tick_params(axis="y", length=0); bx.spines["left"].set_visible(False)
    fig.tight_layout(w_pad=0.8)
    fig.savefig(OUT / "fig2_certificate_D.pdf")
    plt.close(fig)


def check_certD():
    """The drawn triangles are certificate D of the notebook under 0->a, 1->b, 2->p, 3->c, 4->q, 5->r."""
    spec = [(0,1,4),(0,2,3),(0,2,5),(0,4,5),(1,2,4),(1,2,5),(1,3,5),(2,3,4),(3,4,5)]
    name = dict(enumerate("abpcqr"))
    drawn = [("b","p","q"),("p","c","q"),("a","b","q"),("a","q","r"),("c","q","r"),
             ("b","c","r"),("b","p","r"),("a","p","r"),("a","p","c")]
    assert sorted(tuple(sorted(name[i] for i in t)) for t in spec) == sorted(tuple(sorted(t)) for t in drawn)


# ---------------------------------------------------------------- Figure 3: experiments
def figure_experiments():
    mh = pd.read_csv(RES / "p2_memory_hierarchy.csv").sort_values("n")
    d1 = pd.read_csv(RES / "degree1_fast_timing.csv")
    cp = pd.read_csv(RES / "compression.csv")
    fig, (ax, bx, cx) = plt.subplots(1, 3, figsize=(TEXTW + 0.3, 2.05))
    # (a) degree-0 pipeline vs plain persistence, ns per vertex
    ax.plot(mh.n, mh.cert_ns_per_vertex, "-o", ms=3, lw=1.0, color=BLUE, label=r"bound")
    ax.plot(mh.n, mh.pers_ns_per_vertex, "--s", ms=3, lw=1.0, color=ORANGE, label=r"persistence")
    ax.set_xscale("log"); ax.set_ylim(0, max(1100, 1.08 * float(np.nanmax([mh.cert_ns_per_vertex.max(), mh.pers_ns_per_vertex.max()]))))
    ax.set_xlabel(r"vertices $n$"); ax.set_ylabel(r"ns per vertex")
    ax.legend(frameon=False, loc="upper left", handlelength=1.8)
    ax.set_title(r"(a) degree 0", loc="left")
    # (b) Corollary 14: degree-1 image barcode, total seconds
    for kind, col, st, mk, fc, lab in [("terrain", AQUA, "-", "o", AQUA, r"terrain"), ("torus", VIOLET, "--", "s", "white", r"torus")]:
        s = d1[d1.kind == kind].sort_values("n")
        bx.plot(s.n, 1e9 * s.t_total / s.n, ls=st, marker=mk, ms=3.6, mfc=fc, mew=0.9, lw=1.0, color=col, label=lab)
    bx.set_xscale("log"); bx.set_ylim(0, max(2300, 1.08 * float(np.nanmax(1e9 * d1.t_total / d1.n))))
    bx.set_xlabel(r"vertices $n$"); bx.set_ylabel(r"ns per vertex")
    bx.legend(frameon=False, loc="upper left", handlelength=1.8)
    bx.set_title(r"(b) degree 1", loc="left")
    # (c) certified fraction vs bits per value, SDRBench fields, tau = 0.5% of the range
    s = cp[(cp.kind != "synthetic") & (cp.tau_frac == 0.005)]
    g = s.groupby(["method", "rel_bound"]).agg(cf=("cert_fraction", "mean"), bpv=("bits_per_value", "mean")).reset_index()
    styles = [("SZ", BLUE, "-", "o"), ("SZ3", ORANGE, "--", "s"), ("SZ3-pwrel", AQUA, "-.", "^"), ("ZFP", YELLOW, ":", "D")]
    for m, col, st, mk in styles:
        r = g[g.method == m].sort_values("rel_bound")
        cx.plot(r.bpv, r.cf, ls=st, marker=mk, ms=3, lw=1.0, color=col, label=m.replace("-pwrel", r"-rel"))
    cx.set_xscale("log"); cx.set_ylim(0, 1.05); cx.set_xlim(0.012, 10)
    cx.set_xlabel(r"bits per value"); cx.set_ylabel(r"certified fraction")
    cx.legend(frameon=False, loc="upper left", handlelength=1.8, borderaxespad=0.1)
    cx.set_title(r"(c) compression", loc="left")
    for a in (ax, bx, cx):
        a.grid(True, which="major", color=GRID, lw=0.4); a.set_axisbelow(True)
    fig.tight_layout(w_pad=0.9)
    fig.savefig(OUT / "fig3_experiments.pdf")
    plt.close(fig)
    return g


# ---------------------------------------------------------------- Figure 4 (appendix): surface checks
FAMILY_NAMES = {
    "RP^2 (7 vertices)": "projective plane (7 v.)", "torus 3x3": r"torus $3\times3$", "torus 4x4": r"torus $4\times4$",
    "torus 3x5": r"torus $3\times5$", "Klein bottle 3x3": r"Klein bottle $3\times3$", "Klein bottle 4x4": r"Klein bottle $4\times4$",
    "Klein bottle (D+D)": "Klein bottle (glued)", "genus 2 (E+E)": "genus 2 (glued)", "N3 (D+E)": "non-orient. genus 3 (glued)",
    "Moebius band (6 v.)": r"M$\ddot{\rm o}$bius band (6 v.)", "Moebius band (grid)": r"M$\ddot{\rm o}$bius band (grid)",
    "punctured torus (7 v.)": "punctured torus (7 v.)", "punctured torus 4x4": r"punctured torus $4\times4$",
    "punctured Klein bottle": "punctured Klein bottle", "torus with two holes": "torus with two holes",
    "disk": "disk", "annulus": "annulus", "sphere": "sphere"}


def figure_appendix():
    tc = pd.read_csv(RES / "theorem_c.csv")
    raw = pd.read_csv(RES / "raw" / "v5_degree1_fast.csv")
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(TEXTW + 0.3, 3.4), gridspec_kw=dict(width_ratios=[1.3, 1]))
    tc = tc.assign(order=tc.closed.map({True: 0, False: 1})).sort_values(["bhat", "order", "family"], ascending=[True, True, True])
    ys = np.arange(len(tc))[::-1]
    cols = [INK2 if b == 0 else BLUE for b in tc.bhat]
    ax.barh(ys, tc.barcode_attained_by_fD, height=0.62, color=cols)
    ax.set_yticks(ys, [rf"{FAMILY_NAMES[f]}, $\hat b={b}$" for f, b in zip(tc.family, tc.bhat)], fontsize=7)
    ax.set_xlim(0, 1.0); ax.set_xlabel(r"share of boxes where $f_D$ attains $\mathrm{Bar}_1(u\geq\ell)$")
    ax.tick_params(axis="y", length=0)
    ax.set_title(r"(a) checks of Theorem 10", loc="left")
    # (b) Corollary 14 against the exact rank-invariant computation (exact-agreement runs)
    from matplotlib.lines import Line2D
    from matplotlib.ticker import NullFormatter, FixedLocator, FixedFormatter
    ex = raw[raw.t_exact.notna()]
    for kind, mk, fc in [("terrain", "o", None), ("torus", "s", "white")]:
        s = ex[ex.kind == kind].groupby("n")[["t_fast", "t_exact"]].median().reset_index()
        bx.plot(s.n, s.t_exact, ls="--", marker=mk, ms=3.4, lw=1.0, color=ORANGE, mfc=fc or ORANGE, mew=0.9)
        bx.plot(s.n, s.t_fast, ls="-", marker=mk, ms=3.4, lw=1.0, color=BLUE, mfc=fc or BLUE, mew=0.9)
    bx.set_xscale("log"); bx.set_yscale("log")
    ticks = [16, 25, 49, 100]
    bx.xaxis.set_major_locator(FixedLocator(ticks)); bx.xaxis.set_major_formatter(FixedFormatter([str(t) for t in ticks]))
    bx.xaxis.set_minor_formatter(NullFormatter())
    bx.set_xlabel(r"vertices $n$"); bx.set_ylabel(r"seconds (median)")
    handles = [Line2D([], [], color=ORANGE, ls="--", lw=1.0, label="exact rank invariant"),
               Line2D([], [], color=BLUE, ls="-", lw=1.0, label="Corollary 14"),
               Line2D([], [], color=INK2, ls="", marker="o", ms=3.4, label="terrain"),
               Line2D([], [], color=INK2, ls="", marker="s", ms=3.4, mfc="white", label="torus")]
    bx.legend(handles=handles, frameon=False, loc="lower right", bbox_to_anchor=(1.0, 0.16), fontsize=7, handlelength=1.8)
    bx.set_title(r"(b) degree 1, small grids", loc="left")
    for a in (ax, bx):
        a.grid(True, which="major", color=GRID, lw=0.4, axis="x" if a is ax else "both"); a.set_axisbelow(True)
    fig.tight_layout(w_pad=1.0)
    fig.savefig(OUT / "fig4_surface_checks.pdf")
    plt.close(fig)
    return ex


if __name__ == "__main__":
    info = figure_path()
    check_certD()
    figure_certD()
    g = figure_experiments()
    ex = figure_appendix()
    print("exact-agreement runs:", len(ex), "all agree:", bool(ex.agree.all()))
    for k, v in info.items():
        print(k, v)
    print(g.to_string())
'''
(RESULTS / "code").mkdir(exist_ok=True)
(RESULTS / "code" / "make_figures.py").write_text(MAKE_FIGURES_SRC)
_fig = subprocess.run([sys.executable, str(RESULTS / "code" / "make_figures.py"), str(RESULTS), str(RESULTS / "figures")],
                      capture_output=True, text=True, timeout=900)
print(_fig.stdout[-3000:])
if _fig.returncode:
    print(_fig.stderr[-3000:])
FIGURES = {name: (RESULTS / "figures" / name).exists() for name in
           ("fig1_fU_path.pdf", "fig2_certificate_D.pdf", "fig3_experiments.pdf", "fig4_surface_checks.pdf")}
print("figures:", FIGURES)
set_status("figures", "EXECUTED" if _fig.returncode == 0 and all(FIGURES.values()) else "FAILED", f"return code {_fig.returncode}")

# Table 2: the census by complex class, in the paper's order
_order = ["planar surface", "non-planar surface", "KC vertex", "non-manifold edge, no KC", "cycle-or-paths links, not a surface",
          "pinched sphere (KC pinch vertex)"]
_names = {"planar surface": "planar surfaces", "non-planar surface": "non-planar surfaces", "KC vertex": "2-complexes with a KC vertex",
          "non-manifold edge, no KC": "non-manifold edge, no KC vertex", "cycle-or-paths links, not a surface": "links cycles or paths, not a surface",
          "pinched sphere (KC pinch vertex)": "pinched spheres (KC pinch vertex)"}
tab2 = (d1x_cls.groupby("cls").agg(instances=("instances", "sum"), max_vertices=("max_vertices", "max"),
                                   not_attained=("proved_not_attained", "sum"), count_gaps=("count_gaps", "sum"), undecided=("undecided", "sum"))
        .reindex([c for c in _order if c in set(d1x_cls.cls)]).reset_index())
tab2["class"] = tab2.cls.map(_names)
tab2[["class", "instances", "max_vertices", "not_attained", "count_gaps", "undecided"]].to_csv(RESULTS / "tables" / "table2_census.csv", index=False)
_fmt = lambda x: f"{int(x):,}".replace(",", "{,}")
(RESULTS / "tables" / "table2_census.tex").write_text("\n".join(
    [rf"{r['class']} & {_fmt(r.instances)} & {int(r.max_vertices)} & {_fmt(r.not_attained)} & {_fmt(r.count_gaps)}\\" for _, r in tab2.iterrows()]) + "\n")

# Table 3: SDRBench, tau = 0.5% of the range, means over the fields: certified fraction / mean overcount / bits per value
_c = compv[(compv.kind == "sdrbench") & (compv.tau_frac == 0.005)]
if len(_c):
    tab3 = _c.groupby(["rel_bound", "method"]).agg(cf=("cert_fraction", "mean"), over=("overcount", "mean"), bpv=("bits_per_value", "mean"),
                                                   fields=("dataset", "nunique")).reset_index()
    tab3.to_csv(RESULTS / "tables" / "table3_compression.csv", index=False)
    _cols = ["SZ", "SZ3", "SZ3-pwrel", "ZFP"]; _rows = []
    for b in sorted(tab3.rel_bound.unique()):
        cells = []
        for m in _cols:
            r = tab3[(tab3.rel_bound == b) & (tab3.method == m)]
            cells.append(f"{r.cf.iloc[0]:.3f} / {r.over.iloc[0]:.1f} / {r.bpv.iloc[0]:.2f}" if len(r) else "--")
        _rows.append(rf"$10^{{{int(round(math.log10(b)))}}}$ & " + " & ".join(cells) + r"\\")
    (RESULTS / "tables" / "table3_compression.tex").write_text("\n".join(_rows) + "\n")
    print(tab3.round(3).to_string(index=False))
else:
    print("Table 3 needs the SDRBench fields (FULL profile with Internet access)")
d1x_how_design.to_csv(RESULTS / "tables" / "census_by_decision_method.csv")
print(tab2.to_string(index=False))

exact-agreement runs: 42 all agree: True
lo [4, 1, 3, 6, 2, 5, 7, 3, 1, 4, 6, 5, 2, 4]
up [6, 2, 7, 8, 5, 6, 9, 7, 3, 8, 7, 9, 6, 6]
fU [4, 2, 3, 6, 5, 5, 7, 3, 3, 4, 6, 6, 6, 6]
bar_l [(1, 7), (1, inf), (2, 6), (2, 6)]
bar_u [(2, inf), (3, 9), (5, 8), (6, 9), (7, 8)]
bar_fU [(2, inf), (3, 7), (5, 6)]
       method  rel_bound        cf       bpv
0          SZ     0.0001  0.969384  2.332437
1          SZ     0.0010  0.748852  0.863287
2          SZ     0.0100  0.215076  0.223446
3         SZ3     0.0001  0.970083  2.004305
4         SZ3     0.0010  0.751209  0.549827
5         SZ3     0.0100  0.202519  0.078584
6   SZ3-pwrel     0.0001  0.991087  6.474463
7   SZ3-pwrel     0.0010  0.924451  4.175600
8   SZ3-pwrel     0.0100  0.594965  2.604941
9         ZFP     0.0001  0.969760  6.617396
10        ZFP     0.0010  0.732107  3.963450
11        ZFP     0.0100  0.176742  2.307393

figures: {'fig1_fU_path.pdf': True, 'fig2_certificate_D.pdf': True, 'fig3_experiments.pdf': True, 'fig4_surface

## 23 · Numbers for the paper

Every number the paper quotes, recomputed from this run and grouped by the sentence or table it belongs to, and written to `paper_numbers.md` and `paper_numbers.json` in the results folder.

In [40]:
# Every number the paper quotes, recomputed from this run, with the sentence it belongs to. paper_numbers.md is the checklist for
# updating the text; paper_numbers.json holds the same values for scripts.
def _pct(a, b):
    return None if not b else round(100.0 * a / b, 1)

_cls_map = {"planar surface": "planar surfaces", "non-planar surface": "non-planar surfaces", "KC vertex": "2-complexes with a KC vertex",
            "non-manifold edge, no KC": "non-manifold edge, no KC vertex", "cycle-or-paths links, not a surface": "links cycles or paths, not a surface",
            "pinched sphere (KC pinch vertex)": "pinched spheres (KC pinch vertex)"}
_t13 = {(r["check"], r["surface"], int(r["boxes"]), int(r["seed"])): r for r in t13_df.to_dict("records")}
def _t13v(check, fam, N, seed, col):
    v = _t13.get((check, fam, N, seed), {}).get(col)
    return None if v is None or v != v else int(v)
_mem = mem.sort_values("n")
_mid = {r["case"]: r for r in MD_ROWS}

PAPER_NUMBERS = {
    "Paper Section 7 and Appendix D: setup and environment": dict(
        session_minutes=round((time.perf_counter() - NOTEBOOK_T0) / 60, 1), logical_cpus=ENV["logical_cpus"], ram_gb=ENV["ram_gb"],
        cpu=ENV["cpu"], python=ENV["python"], numpy=ENV["numpy"], scipy=ENV["scipy"], pandas=ENV["pandas"], numba=ENV["numba"],
        matplotlib=ENV["matplotlib"], config_hash=CONFIG_HASH, seed=CONFIG["seed"], profile=PROFILE,
        unit_tests_passed=int(sum(t["result"] == "PASS" for t in UNIT)), unit_tests=len(UNIT)),
    "Exact decisions (independent enumeration)": dict(instances=D1X_CHECKS.get("indep_checked"),
                                                      order_mismatches=D1X_CHECKS["indep_order_mismatches"],
                                                      decision_mismatches=D1X_CHECKS["indep_decision_mismatches"]),
    "Degree 0 (Theorem 5, Lemma 2, tau = 0 identities)": dict(
        boxes=OPT["instances"], in_box=OPT["in_box"], barcode_equal=OPT["equal"], max_vertices=int(opt.n.max()) if "n" in opt else None,
        persistence_regions=int((optb.region == "persistence").sum()), persistence_regions_equal=int(optb[optb.region == "persistence"].equal.sum()),
        quadrants=int((optb.region == "quadrant").sum()), quadrants_equal=int(optb[optb.region == "quadrant"].equal.sum()),
        gkls_matches=GKLS.get("matches"), gkls_instances=GKLS.get("instances"),
        gst14=next((t["detail"] for t in UNIT if t["test"].startswith("T08")), None)),
    "Table 2 (census)": [dict(cls=_cls_map.get(k, k), instances=int(g.instances.sum()), max_vertices=int(g.max_vertices.max()),
                              not_attained=int(g.proved_not_attained.sum()), count_gaps=int(g.count_gaps.sum()),
                              undecided=int(g.undecided.sum()))
                         for k, g in d1x_cls.groupby("cls", sort=False)],
    "Degree 1, census text": dict(
        non_planar_failures={d: f"{int(g.proved_not_attained.sum())} of {int(g.instances.sum())}"
                             for d, g in d1x_cls[d1x_cls.design.isin(["projective plane", "Moebius band", "punctured torus", "punctured Klein bottle"])].groupby("design")},
        unpinched_controls_attained=f"{CENSUS['unpinched'] - CENSUS['unpinched_not_attained']} of {CENSUS['unpinched']}",
        undecided=CENSUS["undecided"], **{f"method_{k}": v for k, v in CENSUS_METHODS.items()}),
    "Certificates and transfers (Theorem 11, Examples 17-20)": dict(
        **{k: dict(image=bars_txt(C["I1"]), attained=C["attained"], order_types=C["n_orders"], count_gap_taus=C["pers_gap_taus"],
                   locally_minimal=C["locally_minimal"], kc=C["kc_vertices"]) for k, C in CERTS.items()},
        transfers={t["surface"]: dict(attained=t["attained"], order_types=t["n_orders"]) for t in TRANSFERS}),
    "Example 20": EX20,
    "Theorem 8 (planar)": dict(boxes=PLANAR["instances"], families=int(d1x_pl.family.nunique()), max_vertices=int(d1x_pl.nv.max()),
                               failures=PLANAR["theorem_failures"],
                               fD_on_complex_itself={r.family: round(100 * float(r.plain_dual), 1) for r in d1x_pls.itertuples()}),
    "Theorem 10 and Corollary 14 (surfaces)": dict(
        boxes=THMC["instances"], families=THMC["families"], max_vertices=THMC["max_vertices"], max_bhat=int(thmc.bhat.max()),
        failures=THMC["failures"], fast_mismatches=THMC["fast_mismatches"],
        essential_births_differ=int((~thmc.barcode_equal.astype(bool)).sum()), planar_barcode_failures=THMC["planar_barcode_failures"],
        max_mismatch=int(thmc.essential_mismatch.max())),
    "Theorem 13 and Lemma 12": dict(
        boundary_fD_attains={f"{fam} (seed {s})": f"{_t13v('decide', fam, N, s, 'fD_attains')} of {N}"
                             for fam, N, s in CONFIG["t13_decide"] if fam in ("mobius", "ptorus")},
        boundary_proved_unattainable=int(sum(_t13v("decide", fam, N, s, "fD fails -> NOT attained") or 0
                                             for fam, N, s in CONFIG["t13_decide"] if fam in ("mobius", "ptorus"))),
        closed_fD_attains=int(sum(_t13v("decide", fam, N, s, "fD_attains") or 0 for fam, N, s in CONFIG["t13_decide"] if fam in ("rp2", "torus"))),
        closed_boxes=int(sum(N for fam, N, s in CONFIG["t13_decide"] if fam in ("rp2", "torus"))),
        closed_attained_by_fp=int(sum(_t13v("decide", fam, N, s, "fD fails -> attained | peak witness found") or 0
                                      for fam, N, s in CONFIG["t13_decide"] if fam in ("rp2", "torus"))),
        boundary_max_boxes=int(sum(N for _, N, _ in CONFIG["t13_max"])),
        boundary_near_sup_representatives=int(sum(_t13v("max", fam, N, s, "attaining_max_realisations") or 0 for fam, N, s in CONFIG["t13_max"])),
        closed_max_boxes=int(sum(N for _, N, _ in CONFIG["t13_max_closed"])),
        closed_near_sup_representatives=int(sum(_t13v("max_closed", fam, N, s, "attaining_max_realisations") or 0 for fam, N, s in CONFIG["t13_max_closed"])),
        closed_above_Lstar=int(sum(_t13v("max_closed", fam, N, s, "max_above_Lstar") or 0 for fam, N, s in CONFIG["t13_max_closed"])),
        reference_matches=f"{T13['reference_matches']} of {T13['reference_runs']}", inconsistent_runs=T13["inconsistent"],
        extended_boxes=int(t13_df[t13_df.run == "extended"].boxes.sum()) if "run" in t13_df else 0,
        extended_inconsistent=int((~t13_df[t13_df.run == "extended"].consistent_with_theorem.astype(bool)).sum()) if "run" in t13_df else 0),
    "Proposition 15 and Lemma 16": dict(graphs=PROPG["instances"], graph_failures=PROPG["failures"], kc_functions=KC_LEMMA["functions"],
                                        kc_complexes=KC_LEMMA["complexes"], kc_entrances_doing_both=KC_LEMMA["positive_control_events"],
                                        kc_violations=KC_LEMMA["mv_violations"] + KC_LEMMA["lemma_violations"]),
    "Middle degrees (new theorem)": dict(MIDDLE, rows=[{k: r[k] for k in ("case", "d", "k", "vertices", "facets", "betti", "image_barcode")
                                                         if k in r} | {k: r[k] for k in ("order_types", "attained", "count_gap_taus") if k in r}
                                                        for r in MD_ROWS]),
    "3-D grids (Theorem 8 with d = 3; degree-1 census)": dict(degree2=G3_DUAL, degree1=G3_CENSUS,
                                                             degree1_by_shape=g3c_s.to_dict("records") if len(g3c_s) else []),
    "Scaling (Figure 3a,b)": dict(
        sizes=f"{int(_mem.n.min()):,} to {int(_mem.n.max()):,}", uf_ops_per_vertex=[round(x, 2) for x in P2_MEM["uf_ops_per_vertex_range"]],
        ops_loglog_slope=round(SLOPES.get("ops", float("nan")), 3),
        cert_ns_per_vertex=[round(float(_mem.cert_ns_per_vertex.iloc[0])), round(float(_mem.cert_ns_per_vertex.iloc[-1]))],
        pers_ns_per_vertex=[round(float(_mem.pers_ns_per_vertex.iloc[0])), round(float(_mem.pers_ns_per_vertex.iloc[-1]))],
        random_access_ns=[round(P2_MEM["gather_random_ns_min"], 1), round(P2_MEM["gather_random_ns_max"], 1)],
        llc_bytes=P2_MEM["llc_bytes"], array_bytes_per_vertex=float((p2m.array_bytes / p2m.n).median()),
        working_set_model_bytes_per_vertex=BYTES_PER_VERTEX_GRID,
        degree1_max_vertices=D1FAST["max_n"], degree1_seconds_at_max={k: round(v, 1) for k, v in D1FAST["seconds_at_max_n"].items()},
        degree1_exact_grids=D1FAST["exact_checks"], degree1_exact_agree=D1FAST["exact_agree"]),
    "Compression (paper Section 7, Table 3, Appendix D)": dict(COMP_COUNTS, uniform_baseline=COMP_UNIFORM, median_certify_seconds=COMP_TIMING,
                                                        fill_fractions=COMP_MISSING, figure_3c=COMP_FIG,
                                                        sdrbench_executed=SDRB_EXECUTED),
}
save_json(PAPER_NUMBERS, RESULTS / "paper_numbers.json")
_lines = [f"# Numbers for the paper (profile {PROFILE}, configuration {CONFIG_HASH})", ""]
for sec, val in PAPER_NUMBERS.items():
    _lines.append(f"## {sec}")
    items = val if isinstance(val, list) else [val]
    for it in items:
        if isinstance(it, dict):
            for k, v in it.items():
                _lines.append(f"- {k}: {json.dumps(jsonable(v), default=str)}")
        else:
            _lines.append(f"- {json.dumps(jsonable(it), default=str)}")
        if isinstance(val, list):
            _lines.append("")
    _lines.append("")
(RESULTS / "paper_numbers.md").write_text("\n".join(_lines))
print("\n".join(_lines))

# Numbers for the paper (profile FULL, configuration 8f4b785c7a4a)

## Paper Section 7 and Appendix D: setup and environment
- session_minutes: 71.7
- logical_cpus: 4
- ram_gb: 33.66
- cpu: "x86_64"
- python: "3.13.16"
- numpy: "2.1.3"
- scipy: "1.16.3"
- pandas: "2.3.3"
- numba: "0.61.2"
- matplotlib: "3.10.0"
- config_hash: "8f4b785c7a4a"
- seed: 20260925
- profile: "FULL"
- unit_tests_passed: 33
- unit_tests: 33

## Exact decisions (independent enumeration)
- instances: 60
- order_mismatches: 0
- decision_mismatches: 0

## Degree 0 (Theorem 5, Lemma 2, tau = 0 identities)
- boxes: 100000
- in_box: 100000
- barcode_equal: 100000
- max_vertices: 225
- persistence_regions: 3000
- persistence_regions_equal: 3000
- quadrants: 4000
- quadrants_equal: 4000
- gkls_matches: 500
- gkls_instances: 500
- gst14: "mismatches=0/1000"

## Table 2 (census)
- cls: "planar surfaces"
- instances: 1502
- max_vertices: 9
- not_attained: 0
- count_gaps: 0
- undecided: 0

- cls: "non-planar surfaces"
- ins

## 24 · Reproducibility report, anonymity scan and results bundle

The report records the profile, configuration hash, seeds, software versions, every section's status and a SHA-256 checksum of every result file. The anonymity scan runs on every text file in the results folder before it is zipped (see the top of the notebook).

In [41]:
ELAPSED = time.perf_counter() - NOTEBOOK_T0
files = sorted(p for p in RESULTS.rglob("*") if p.is_file() and "data" not in p.relative_to(RESULTS).parts[:1] and p.suffix not in (".jsonl", ".zip"))
REPRO = dict(profile=PROFILE, config_hash=CONFIG_HASH, master_seed=CONFIG["seed"],
             seed_policy="per-experiment seeds = first 32 bits of sha256(master_seed | experiment keys)",
             environment=ENV, elapsed_seconds=round(ELAPSED, 1), statuses={k: v["status"] for k, v in STATUS.items()},
             checksums={str(p.relative_to(RESULTS)): sha256_file(p) for p in files if p.name not in ("reproducibility_report.json", "run_log.txt")},
             datasets=MANIFEST, validation_warnings=WARN)
save_json(REPRO, RESULTS / "logs" / "reproducibility_report.json")
_required = ["unit_tests", "gkls_objective", "v3_optimality", "degree1_v4", "example20", "theorem_c", "prop_g", "degree1_fast", "theorem13",
             "middle_degrees", "grid3d", "P2_scaling", "scaling_v2", "p2_memory_hierarchy", "compression_synthetic", "compression_uniform_baseline",
             "figures"]
_synthetic_ok = all(STATUS.get(k, {}).get("status") == "EXECUTED" for k in _required)
_real_ok = STATUS.get("real_compression", {}).get("status") == "EXECUTED"
_hard = [w for w in WARN if "NOT EXECUTED" not in w]
REPRO_STATUS = "FAIL" if (not _synthetic_ok or _hard) else ("PASS" if _real_ok else "PARTIAL (no SDRBench data)")
print(f"elapsed {ELAPSED / 60:.1f} min | {len(REPRO['checksums'])} result files checksummed | reproducibility: {REPRO_STATUS}")
if not _synthetic_ok:
    print("missing or failed sections:", [k for k in _required if STATUS.get(k, {}).get("status") != "EXECUTED"])

# ---- anonymity scan -----------------------------------------------------------------------------------------------------------------
# Double-blind review: no output may reveal who ran the notebook. Every text file under RESULTS is scanned for e-mail addresses and
# home-directory paths (which contain user names; they can enter the run log through warnings). Matches are replaced by <redacted> in
# place. Optionally set ANON_FORBIDDEN to a comma-separated list of further strings (names, affiliations, user names) before running;
# they are matched case-insensitively and never printed.
ANON_PATTERNS = [("e-mail address", re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}")),
                 ("home directory", re.compile(r"(?:/home/|/Users/|[A-Za-z]:\\\\?Users\\\\?)[^/\\\r\n\"',;:)]+")),
                 ("mounted drive", re.compile(r"/content/drive/[^\s\"',;]*"))]
_extra = [s.strip() for s in os.environ.get("ANON_FORBIDDEN", "").split(",") if s.strip()]
ANON_PATTERNS += [(f"forbidden string #{i + 1}", re.compile(re.escape(s), re.I)) for i, s in enumerate(_extra)]


def anonymity_scan(root, redact=True):
    hits = collections.Counter(); files_hit = collections.defaultdict(set)
    for p in sorted(Path(root).rglob("*")):
        if not p.is_file() or p.suffix.lower() not in (".txt", ".json", ".jsonl", ".csv", ".md", ".tex", ".py", ".log"):
            continue
        if p.stat().st_size > 200e6:
            continue
        try:
            text = p.read_text(encoding="utf-8")
        except Exception:
            continue
        new = text
        for label, rx in ANON_PATTERNS:
            n = len(rx.findall(new))
            if n:
                hits[label] += n; files_hit[label].add(str(p.relative_to(root)))
                new = rx.sub("<redacted>", new)
        if redact and new != text:
            p.write_text(new, encoding="utf-8")
    return dict(hits), {k: sorted(v) for k, v in files_hit.items()}


sys.stdout.flush(); sys.stderr.flush()
ANON_HITS, ANON_FILES = anonymity_scan(RESULTS)
ANON = dict(patterns=[label for label, _ in ANON_PATTERNS], matches=ANON_HITS, files=ANON_FILES, redacted=bool(ANON_HITS))
save_json(ANON, RESULTS / "logs" / "anonymity_scan.json")
print("anonymity scan:", "no identifying strings found" if not ANON_HITS else f"redacted {ANON_HITS} in {ANON_FILES}")

elapsed 71.7 min | 60 result files checksummed | reproducibility: PASS
anonymity scan: no identifying strings found


In [42]:
SCRUB_SRC = r'''"""Prepare an executed copy of the experiment notebook for double-blind sharing.

Removes notebook and cell metadata (platform provenance, authorship tags, data-source names, execution timestamps) and redacts e-mail
addresses and home-directory paths in every output. Extra strings (names, affiliations, user names) can be redacted with --forbid.

Usage: python3 scrub_notebook.py EXECUTED.ipynb CLEAN.ipynb [--forbid "string one,string two"] [--drop-outputs]
"""
import json, re, sys

PATTERNS = [re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}"),
            re.compile(r"(?:/home/|/Users/|[A-Za-z]:\\\\?Users\\\\?)[^/\\\r\n\"',;:)]+"),
            re.compile(r"/content/drive/[^\s\"',;]*")]


def main(argv):
    src, dst = argv[1], argv[2]; extra = []; drop = "--drop-outputs" in argv
    if "--forbid" in argv:
        extra = [s.strip() for s in argv[argv.index("--forbid") + 1].split(",") if s.strip()]
    pats = PATTERNS + [re.compile(re.escape(s), re.I) for s in extra]
    hits = 0

    def clean(x):
        nonlocal hits
        if isinstance(x, list):
            return [clean(v) for v in x]
        if isinstance(x, str):
            for p in pats:
                x, n = p.subn("<redacted>", x); hits += n
        return x

    with open(src, encoding="utf-8") as fh:
        nb = json.load(fh)
    nb["metadata"] = {"kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
                      "language_info": {"name": "python"}}
    for cell in nb["cells"]:
        cell["metadata"] = {}
        cell["source"] = clean(cell.get("source", ""))
        if cell["cell_type"] != "code":
            continue
        if drop:
            cell["outputs"] = []; cell["execution_count"] = None; continue
        for out in cell.get("outputs", []):
            if "metadata" in out:
                out["metadata"] = {}
            for key in ("text", "traceback", "evalue"):
                if key in out:
                    out[key] = clean(out[key])
            for mime in ("text/plain", "text/html", "text/markdown", "text/latex", "application/json"):
                if mime in out.get("data", {}):
                    out["data"][mime] = clean(out["data"][mime])
    with open(dst, "w", encoding="utf-8") as fh:
        json.dump(nb, fh, indent=1, ensure_ascii=False)
    print(f"wrote {dst}: metadata removed, {hits} identifying strings redacted")


if __name__ == "__main__":
    main(sys.argv)
'''
(RESULTS / "code").mkdir(exist_ok=True)
(RESULTS / "code" / "scrub_notebook.py").write_text(SCRUB_SRC)


def make_bundle():
    sys.stdout.flush(); sys.stderr.flush()
    zpath = RESULTS.parent / f"socg27_results_{PROFILE}_{CONFIG_HASH}.zip"
    readme = (f"Results bundle of the experiment notebook (profile {PROFILE}, configuration {CONFIG_HASH}).\n"
              f"Reproducibility status: {REPRO_STATUS}. Statuses: {json.dumps({k: v['status'] for k, v in STATUS.items()})}\n\n"
              "results/*.csv, results.json      top-level exports read by the figure script\n"
              "results/paper_numbers.md, .json  every number quoted in the paper, recomputed from this run\n"
              "results/tables/                  Tables 2 and 3 of the paper (CSV and LaTeX) and the census by decision method\n"
              "results/figures/                 Figures 1-4 of the paper (PDF), drawn by results/code/make_figures.py\n"
              "results/raw/, results/processed/ per-instance results and summaries\n"
              "results/checkpoints/             resumable checkpoints (JSON lines)\n"
              "results/logs/                    run log, statuses, validation, reproducibility report, anonymity scan\n"
              "results/code/                    figure script and the scrub script for executed notebooks\n")
    n_files = 0
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        for p in sorted(RESULTS.rglob("*")):
            if not p.is_file():
                continue
            rel = p.relative_to(RESULTS)
            if rel.parts[0] == "data" and p.name != "manifest.json":
                continue
            z.write(p, str(Path("results") / rel)); n_files += 1
        z.writestr("README.txt", readme)
    with zipfile.ZipFile(zpath) as z:
        if z.testzip() is not None:
            raise RuntimeError("zip integrity check failed")
    return zpath, n_files


BUNDLE, _nf = make_bundle()
print(f"bundle: {BUNDLE.name} ({_nf} files, {BUNDLE.stat().st_size / 1e6:.1f} MB, integrity OK), next to the results folder")
if Path("/kaggle/working").exists() and BUNDLE.parent.resolve() != Path("/kaggle/working").resolve():
    shutil.copy(BUNDLE, Path("/kaggle/working") / BUNDLE.name)
    print("copied to the Kaggle output folder")

bundle: socg27_results_FULL_8f4b785c7a4a.zip (84 files, 2.6 MB, integrity OK), next to the results folder
